# Project : Rossmann Store Sales Forecasting

## Notebook 03: Feature Engineering and Preprocessing

---

### Objective

The objective of this notebook is to transform the prepared Rossmann datasets into a model-ready format through feature engineering and leakage-aware preprocessing.

The main tasks performed in this notebook include:

1. Loading the prepared training and test datasets.
2. Validating dataset structure and data types.
3. Creating meaningful date-based features.
4. Engineering competition and promotion-related features where appropriate.
5. Handling missing values based on feature meaning.
6. Encoding categorical variables into a machine-learning-ready format.
7. Creating a chronological training and validation split.
8. Fitting preprocessing transformations using training data only.
9. Transforming validation and test datasets consistently.
10. Validating the final model-ready datasets for missing values, feature consistency, and potential data leakage.

---

### Important Methodology

To prevent data leakage:

- The target variable `Sales` will not be used as an input feature.
- The `Customers` variable will not be used as a direct forecasting feature because it is unavailable in the test dataset.
- The training and validation data will be split chronologically.
- Imputation and scaling statistics will be learned from the training portion only.
- The same fitted preprocessing pipeline will then be applied to validation and test data.

---

### Expected Output

By the end of this notebook, the project will produce:

- Engineered training features.
- Engineered validation features.
- Engineered test features.
- A fitted preprocessing pipeline.
- A validated, consistent feature structure ready for machine learning models.

---

**Next Stage:** Machine Learning Model Building and Evaluation

# ============================================================
# SECTION 1: IMPORTS, CONFIGURATION & REPRODUCIBILITY SETUP
# ============================================================

## Purpose of This Section

This section prepares the notebook environment for feature engineering and preprocessing.

The following tasks will be completed:

- Import the required Python libraries.
- Configure display and visualization settings.
- Set a fixed random state for reproducibility.
- Configure logging for important processing steps.
- Prepare the environment for leakage-aware preprocessing.

### Reproducibility

A fixed random state will be used wherever applicable to ensure that the preprocessing workflow produces consistent and reproducible results.

### Logging

A logging system will be used to record important preprocessing steps and support easier debugging and reproducibility.

In [369]:
# ============================================================
# SECTION 1: IMPORTS, CONFIGURATION & REPRODUCIBILITY SETUP
# ============================================================

# -----------------------------
# 1. Import Required Libraries
# -----------------------------

import os
import warnings
import logging

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp

# Scikit-learn preprocessing tools
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Reproducibility
RANDOM_STATE = 42

import random

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# -----------------------------
# 3. Pandas Display Settings
# -----------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

# -----------------------------
# 4. Visualization Configuration
# -----------------------------

plt.rcParams["figure.figsize"] = (10, 6)

# -----------------------------
# 5. Logging Configuration
# -----------------------------

logger = logging.getLogger("Project6_Preprocessing")

# Prevent duplicate log messages if the cell is run again
if not logger.handlers:

    logger.setLevel(logging.INFO)

    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
    )

    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

# -----------------------------
# 6. Environment Verification
# -----------------------------

print("=" * 75)
print("PROJECT 6: FEATURE ENGINEERING & PREPROCESSING")
print("=" * 75)

print("\nConfiguration completed successfully.")

print("\nLibrary Versions:")
print(f"NumPy Version:  {np.__version__}")
print(f"Pandas Version: {pd.__version__}")

print(f"\nRandom State: {RANDOM_STATE}")

logger.info(
    "Imports, configuration and reproducibility setup completed successfully."
)

2026-11-06 16:12:16,970 | INFO | Project6_Preprocessing | Imports, configuration and reproducibility setup completed successfully.


PROJECT 6: FEATURE ENGINEERING & PREPROCESSING

Configuration completed successfully.

Library Versions:
NumPy Version:  2.5.4
Pandas Version: 3.1.0

Random State: 42


# ============================================================
# SECTION 2: DATA LOADING & INITIAL VALIDATION
# ============================================================

## Purpose of This Section

This section loads the prepared Rossmann training and test datasets that will be used for feature engineering and preprocessing.

Before creating new features, the datasets will be validated to ensure that:

- The files are loaded successfully.
- Training and test dataset shapes are verified.
- Column structures are inspected.
- The target variable is correctly identified.
- Date columns are available for temporal feature engineering.
- Missing values are reviewed before preprocessing.

### Important Methodology

The original prepared datasets will not be overwritten during this notebook.

Feature engineering will be performed on working copies to preserve the original data and maintain a clear, reproducible preprocessing workflow.

The training dataset contains the target variable:

`Sales`

The test dataset does not contain `Sales` because it represents the future period for which predictions will eventually be generated.

### Expected Outcome

After this section, we will have validated working datasets ready for feature engineering and leakage-aware preprocessing.

In [370]:
# ============================================================
# SECTION 2: DATA LOADING & INITIAL VALIDATION
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# STEP 1: Define Project Root
# ------------------------------------------------------------

# This notebook is expected to be inside the "notebooks" folder.
PROJECT_ROOT = Path.cwd().parent

# ------------------------------------------------------------
# STEP 2: Define Processed Data Directory
# ------------------------------------------------------------

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DIR / "train_merged.csv"
TEST_PATH = PROCESSED_DIR / "test_merged.csv"

# ------------------------------------------------------------
# STEP 3: Verify Required Files
# ------------------------------------------------------------

print("=" * 85)
print("PROJECT 6: DATA LOADING & INITIAL VALIDATION")
print("=" * 85)

print("\nProject Root:")
print(PROJECT_ROOT.resolve())

print("\nProcessed Data Directory:")
print(PROCESSED_DIR.resolve())

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Processed data directory was not found:\n{PROCESSED_DIR.resolve()}"
    )

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"Training file was not found:\n{TRAIN_PATH.resolve()}"
    )

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Test file was not found:\n{TEST_PATH.resolve()}"
    )

print("\nRequired processed files found successfully.")

# ------------------------------------------------------------
# STEP 4: Load Prepared Datasets
# ------------------------------------------------------------

train_df = pd.read_csv(
    TRAIN_PATH,
    low_memory=False
)

test_df = pd.read_csv(
    TEST_PATH,
    low_memory=False
)

# ------------------------------------------------------------
# STEP 5: Convert Date Columns
# ------------------------------------------------------------

if "Date" not in train_df.columns:
    raise KeyError("Column 'Date' was not found in train_merged.csv.")

if "Date" not in test_df.columns:
    raise KeyError("Column 'Date' was not found in test_merged.csv.")

train_df["Date"] = pd.to_datetime(
    train_df["Date"],
    errors="coerce"
)

test_df["Date"] = pd.to_datetime(
    test_df["Date"],
    errors="coerce"
)

# ------------------------------------------------------------
# STEP 6: Validate Dataset Shapes
# ------------------------------------------------------------

print("\n" + "-" * 85)
print("DATASET SHAPES")
print("-" * 85)

print(f"Training Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:     {test_df.shape}")

# ------------------------------------------------------------
# STEP 7: Validate Target Variable
# ------------------------------------------------------------

print("\n" + "-" * 85)
print("TARGET VARIABLE VALIDATION")
print("-" * 85)

print(f"Sales available in Training: {'Sales' in train_df.columns}")
print(f"Sales available in Test:     {'Sales' in test_df.columns}")

if "Sales" not in train_df.columns:
    raise KeyError(
        "The target variable 'Sales' is missing from the training dataset."
    )

# ------------------------------------------------------------
# STEP 8: Validate Date Ranges
# ------------------------------------------------------------

print("\n" + "-" * 85)
print("DATE RANGE VALIDATION")
print("-" * 85)

print(
    f"Training: {train_df['Date'].min().date()} "
    f"to {train_df['Date'].max().date()}"
)

print(
    f"Test:     {test_df['Date'].min().date()} "
    f"to {test_df['Date'].max().date()}"
)

# ------------------------------------------------------------
# STEP 9: Compare Train and Test Columns
# ------------------------------------------------------------

train_only_columns = sorted(
    set(train_df.columns) - set(test_df.columns)
)

test_only_columns = sorted(
    set(test_df.columns) - set(train_df.columns)
)

print("\n" + "-" * 85)
print("COLUMN COMPARISON")
print("-" * 85)

print("\nColumns only in Training:")
print(train_only_columns)

print("\nColumns only in Test:")
print(test_only_columns)

# ------------------------------------------------------------
# STEP 10: Check Missing Date Values
# ------------------------------------------------------------

print("\n" + "-" * 85)
print("DATE QUALITY CHECK")
print("-" * 85)

print(
    f"Missing Date values in Training: "
    f"{train_df['Date'].isna().sum()}"
)

print(
    f"Missing Date values in Test: "
    f"{test_df['Date'].isna().sum()}"
)

# ------------------------------------------------------------
# STEP 11: Create Working Copies
# ------------------------------------------------------------

train_working = train_df.copy()
test_working = test_df.copy()

print("\nWorking copies created successfully.")

# ------------------------------------------------------------
# STEP 12: Logging and Completion
# ------------------------------------------------------------

logger.info(
    "Section 2 completed successfully. "
    "Train shape=%s; Test shape=%s",
    train_df.shape,
    test_df.shape
)

print("\n" + "=" * 85)
print("DATA LOADING & INITIAL VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 85)

PROJECT 6: DATA LOADING & INITIAL VALIDATION

Project Root:
D:\Project_6_Rossmann_Forecasting

Processed Data Directory:
D:\Project_6_Rossmann_Forecasting\data\processed

Required processed files found successfully.

-------------------------------------------------------------------------------------
DATASET SHAPES
-------------------------------------------------------------------------------------
Training Dataset Shape: (1017209, 19)
Test Dataset Shape:     (41088, 18)

-------------------------------------------------------------------------------------
TARGET VARIABLE VALIDATION
-------------------------------------------------------------------------------------
Sales available in Training: True
Sales available in Test:     False

-------------------------------------------------------------------------------------
DATE RANGE VALIDATION
-------------------------------------------------------------------------------------
Training: 2013-01-01 to 2015-07-31
Test:     2015-08-01 to

2026-11-06 16:12:18,499 | INFO | Project6_Preprocessing | Section 2 completed successfully. Train shape=(1017209, 19); Test shape=(41088, 18)



Working copies created successfully.

DATA LOADING & INITIAL VALIDATION COMPLETED SUCCESSFULLY


# ============================================================
# SECTION 3: INITIAL DATA QUALITY & FEATURE READINESS CHECK
# ============================================================

## Purpose of This Section

Before performing feature engineering, it is important to validate the current quality and structure of the prepared training and test datasets.

This section will examine:

- Duplicate records.
- Missing values in training and test datasets.
- Data types of important variables.
- Availability of features across training and test datasets.
- Columns that require special preprocessing consideration.

### Important Feature Availability Rule

A feature should only be used as a direct forecasting input if it is available at prediction time.

Therefore:

- `Sales` is the target variable and will not be used as an input feature.
- `Customers` is available only in the training dataset and is unavailable in the future test dataset.
- Therefore, `Customers` will not be used as a direct feature for future sales forecasting.

### Objective

The purpose of this validation is to identify any remaining data quality issues before feature engineering and to ensure that subsequent preprocessing steps are applied consistently and without data leakage.

### Expected Outcome

After this section, we will have a clear understanding of:

- Remaining missing values.
- Duplicate records.
- Feature availability differences.
- Data types requiring transformation.
- Features requiring special preprocessing strategies.

In [371]:
# ============================================================
# SECTION 3: INITIAL DATA QUALITY & FEATURE READINESS CHECK
# ============================================================

print("=" * 85)
print("INITIAL DATA QUALITY & FEATURE READINESS CHECK")
print("=" * 85)

# ------------------------------------------------------------
# 1. Dataset Information
# ------------------------------------------------------------

print("\n1. DATASET OVERVIEW")
print("-" * 85)

print(f"Training Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:     {test_df.shape}")

# ============================================================
# ADDITIONAL CHECK: STORE-DATE KEY UNIQUENESS
# ============================================================

print("=" * 85)
print("STORE-DATE COMBINATION VALIDATION")
print("=" * 85)


# Each store should have only one observation per date.

train_key_duplicates = train_df.duplicated(
    subset=["Store", "Date"]
).sum()

test_key_duplicates = test_df.duplicated(
    subset=["Store", "Date"]
).sum()


print(
    f"\nDuplicate Store-Date combinations in Training: "
    f"{train_key_duplicates:,}"
)

print(
    f"Duplicate Store-Date combinations in Test: "
    f"{test_key_duplicates:,}"
)


store_date_unique = (
    train_key_duplicates == 0
    and test_key_duplicates == 0
)


print(
    f"\nStore-Date Combination Unique: "
    f"{store_date_unique}"
)


print("=" * 85)

# ------------------------------------------------------------
# 2. Duplicate Record Check
# ------------------------------------------------------------

print("\n2. DUPLICATE RECORD CHECK")
print("-" * 85)

train_duplicates = train_df.duplicated().sum()
test_duplicates = test_df.duplicated().sum()

print(f"Duplicate records in Training: {train_duplicates:,}")
print(f"Duplicate records in Test:     {test_duplicates:,}")

# ------------------------------------------------------------
# 3. Missing Value Summary
# ------------------------------------------------------------

print("\n3. MISSING VALUE SUMMARY")
print("-" * 85)

train_missing = (
    train_df.isna()
    .sum()
    .sort_values(ascending=False)
)

test_missing = (
    test_df.isna()
    .sum()
    .sort_values(ascending=False)
)

train_missing = train_missing[train_missing > 0]
test_missing = test_missing[test_missing > 0]

print("\nMissing Values in Training Dataset:")

if len(train_missing) == 0:
    print("No missing values found.")
else:
    missing_summary_train = pd.DataFrame({
        "Missing_Count": train_missing,
        "Missing_Percentage": (
            train_missing / len(train_df) * 100
        ).round(2)
    })
    
    print(missing_summary_train)

print("\nMissing Values in Test Dataset:")

if len(test_missing) == 0:
    print("No missing values found.")
else:
    missing_summary_test = pd.DataFrame({
        "Missing_Count": test_missing,
        "Missing_Percentage": (
            test_missing / len(test_df) * 100
        ).round(2)
    })
    
    print(missing_summary_test)

# ------------------------------------------------------------
# 4. Data Type Summary
# ------------------------------------------------------------

print("\n4. DATA TYPE SUMMARY")
print("-" * 85)

print("\nTraining Dataset Data Types:")
print(train_df.dtypes.value_counts())

print("\nTest Dataset Data Types:")
print(test_df.dtypes.value_counts())

# ------------------------------------------------------------
# 5. Feature Availability Check
# ------------------------------------------------------------

print("\n5. FEATURE AVAILABILITY CHECK")
print("-" * 85)

train_only_features = sorted(
    set(train_df.columns) - set(test_df.columns)
)

test_only_features = sorted(
    set(test_df.columns) - set(train_df.columns)
)

common_features = sorted(
    set(train_df.columns).intersection(set(test_df.columns))
)

print(f"\nCommon Features: {len(common_features)}")
print(f"Training-only Features: {train_only_features}")
print(f"Test-only Features: {test_only_features}")

# ------------------------------------------------------------
# 6. Important Forecasting Feature Checks
# ------------------------------------------------------------

print("\n6. IMPORTANT FORECASTING FEATURE CHECKS")
print("-" * 85)

print(f"Target 'Sales' in Training: {'Sales' in train_df.columns}")
print(f"Target 'Sales' in Test:     {'Sales' in test_df.columns}")

print(f"\n'Customers' in Training: {'Customers' in train_df.columns}")
print(f"'Customers' in Test:     {'Customers' in test_df.columns}")

# ------------------------------------------------------------
# 7. Date Column Validation
# ------------------------------------------------------------

print("\n7. DATE COLUMN VALIDATION")
print("-" * 85)

print(f"Training Date dtype: {train_df['Date'].dtype}")
print(f"Test Date dtype:     {test_df['Date'].dtype}")

print(
    f"\nMissing Dates in Training: "
    f"{train_df['Date'].isna().sum():,}"
)

print(
    f"Missing Dates in Test: "
    f"{test_df['Date'].isna().sum():,}"
)

# ------------------------------------------------------------
# 8. Completion
# ------------------------------------------------------------

logger.info(
    "Section 3 data quality and feature readiness check completed successfully."
)

print("\n" + "=" * 85)
print("INITIAL DATA QUALITY & FEATURE READINESS CHECK COMPLETED")
print("=" * 85)

# ============================================================
# ADDITIONAL CHECK: SALES TARGET VALIDATION
# ============================================================

print("=" * 85)
print("SALES TARGET DATA VALIDATION")
print("=" * 85)


sales_target = pd.to_numeric(
    train_df["Sales"],
    errors="coerce"
)

sales_missing = sales_target.isna().sum()

sales_infinite = np.isinf(
    sales_target.to_numpy(dtype=float)
).sum()

sales_negative = (
    sales_target < 0
).sum()


print(f"\nMissing Sales Values: {sales_missing:,}")
print(f"Infinite Sales Values: {sales_infinite:,}")
print(f"Negative Sales Values: {sales_negative:,}")


sales_target_valid = (
    sales_missing == 0
    and sales_infinite == 0
    and sales_negative == 0
)


print(
    f"\nSales Target Valid: {sales_target_valid}"
)

print("=" * 85)

INITIAL DATA QUALITY & FEATURE READINESS CHECK

1. DATASET OVERVIEW
-------------------------------------------------------------------------------------
Training Dataset Shape: (1017209, 19)
Test Dataset Shape:     (41088, 18)
STORE-DATE COMBINATION VALIDATION

Duplicate Store-Date combinations in Training: 0
Duplicate Store-Date combinations in Test: 0

Store-Date Combination Unique: True

2. DUPLICATE RECORD CHECK
-------------------------------------------------------------------------------------


2026-11-06 16:12:19,466 | INFO | Project6_Preprocessing | Section 3 data quality and feature readiness check completed successfully.


Duplicate records in Training: 0
Duplicate records in Test:     0

3. MISSING VALUE SUMMARY
-------------------------------------------------------------------------------------

Missing Values in Training Dataset:
                           Missing_Count  Missing_Percentage
Promo2SinceYear                   508031               49.94
Promo2SinceWeek                   508031               49.94
PromoInterval                     508031               49.94
CompetitionOpenSinceYear          323348               31.79
CompetitionOpenSinceMonth         323348               31.79
CompetitionDistance                 2642                0.26

Missing Values in Test Dataset:
                           Missing_Count  Missing_Percentage
Promo2SinceWeek                    17232               41.94
PromoInterval                      17232               41.94
Promo2SinceYear                    17232               41.94
CompetitionOpenSinceMonth          15216               37.03
CompetitionOpenSince

# SECTION 4: DATE-BASED AND HOLIDAY FEATURE ENGINEERING

## Purpose

Date information contains important patterns that can influence
daily store sales, including weekdays, weekends, seasonal effects
and public or school holidays.

This section transforms the original Date column into useful
calendar and holiday-related features.

## 1. Calendar Feature Creation

The following features are created:

- Year
- Month
- Day
- Date_DayOfWeek
- WeekOfYear
- Quarter
- IsWeekend
- Month_Position

## 2. Day of Week Validation

The original Rossmann DayOfWeek variable is preserved.

A separate Date_DayOfWeek feature is generated using the
pandas weekday convention:

- Monday = 0
- Sunday = 6

The original Rossmann convention is:

- Monday = 1
- Sunday = 7

The two representations are checked for consistency.

## 3. Month Position

Each observation is assigned to one of three periods:

- Beginning: Days 1–10
- Mid: Days 11–20
- End: Days 21–31

## 4. Holiday Proximity Features

The following two features are created:

- DaysSinceHoliday
- DaysUntilHoliday

A holiday event is identified when either StateHoliday is not zero or SchoolHoliday equals one.

Both features are calculated separately for each store.

A holiday occurring on the observation date receives a distance value of zero.

### Missing-Date Handling

The original dataset contains 180 stores with a continuous missing calendar period from 01 July 2014 to 31 December 2014.

To prevent holiday-distance calculations from incorrectly crossing missing calendar periods:

- Each store's observations are sorted chronologically.
- A new time-series segment is created whenever a calendar-date gap greater than one day is detected.
- Previous and next holiday dates are calculated separately within each continuous segment.
- Missing holiday distances at segment boundaries are preserved as NaN rather than estimated using unavailable holiday records.

Missing calendar dates are not interpreted as zero-sales days or ordinary non-holiday days.

## 5. Leakage Prevention

Feature creation uses calendar and holiday information only.

Sales and Customers are not used.

The original Date column is retained for temporal splitting but excluded from final model input features.

Known holiday information from the available training and test calendar records may be used for date-based feature construction. No future Sales or Customers information is used.

## Expected Outcome

Both datasets will contain consistent calendar and holiday-related features.

Holiday-distance calculations will not cross missing calendar periods, and unavailable boundary values will remain missing for downstream preprocessing.



In [372]:
# ============================================================
# SECTION 4: DATE-BASED AND HOLIDAY PROXIMITY FEATURE ENGINEERING
# ============================================================

print("=" * 90)
print("SECTION 4: DATE-BASED FEATURE ENGINEERING")
print("=" * 90)


# ============================================================
# STEP 1: VALIDATE REQUIRED SOURCE COLUMNS
# ============================================================

required_source_columns = [
    "Store",
    "Date",
    "DayOfWeek",
    "StateHoliday",
    "SchoolHoliday"
]

for dataset_name, frame in [
    ("Training", train_working),
    ("Test", test_working)
]:

    missing_columns = [
        col
        for col in required_source_columns
        if col not in frame.columns
    ]

    if missing_columns:

        raise KeyError(
            f"{dataset_name} dataset is missing required columns: "
            f"{missing_columns}"
        )

    if frame["Date"].isna().any():

        raise ValueError(
            f"{dataset_name} dataset contains missing dates."
        )


# ============================================================
# STEP 2: CREATE CALENDAR FEATURES
# ============================================================

def add_calendar_features(frame):
    """
    Creates calendar features without using Sales or Customers.
    Original row order and index are preserved.
    """

    result = frame.copy()

    dates = pd.to_datetime(
        result["Date"],
        errors="coerce"
    )

    if dates.isna().any():

        raise ValueError(
            "Invalid Date values found."
        )

    # Extract year, month and day.
    result["Year"] = dates.dt.year.astype("int16")

    result["Month"] = dates.dt.month.astype("int8")

    result["Day"] = dates.dt.day.astype("int8")

    # Monday=0, Sunday=6.
    result["Date_DayOfWeek"] = (
        dates.dt.dayofweek.astype("int8")
    )

    # Extract ISO calendar week.
    result["WeekOfYear"] = (
        dates.dt.isocalendar().week.astype("int16")
    )

    result["Quarter"] = (
        dates.dt.quarter.astype("int8")
    )

    # Saturday and Sunday are weekends.
    result["IsWeekend"] = (
        dates.dt.dayofweek.isin([5, 6]).astype("int8")
    )

    # Categorize the position of each day in its month.
    day = dates.dt.day

    result["Month_Position"] = np.select(

        [
            day.between(1, 10),

            day.between(11, 20),

            day.between(21, 31)

        ],

        [
            "Beginning",
            "Mid",
            "End"
        ],

        default=None
    )

    return result


# Apply calendar feature creation to both datasets.

train_working = add_calendar_features(
    train_working
)

test_working = add_calendar_features(
    test_working
)


# ============================================================
# STEP 3: VALIDATE ORIGINAL DAYOFWEEK
# ============================================================

def check_original_dayofweek(frame, dataset_name):
    """
    Rossmann convention:
    Monday=1, Tuesday=2, ..., Sunday=7.

    Pandas convention:
    Monday=0, ..., Sunday=6.
    """

    original_dow = pd.to_numeric(

        frame["DayOfWeek"],

        errors="coerce"

    )

    expected_dow = (
        pd.to_datetime(frame["Date"]).dt.dayofweek + 1
    )

    valid = (

        original_dow.notna().all()

        and original_dow.between(1, 7).all()

        and original_dow.astype("int64").equals(
            expected_dow.astype("int64")
        )
    )

    print(
        f"\n{dataset_name} Original DayOfWeek Matches Date: "
        f"{valid}"
    )

    return valid


train_dayofweek_consistent = (
    check_original_dayofweek(
        train_working,
        "Training"
    )
)

test_dayofweek_consistent = (
    check_original_dayofweek(
        test_working,
        "Test"
    )
)

dayofweek_consistent = (
    train_dayofweek_consistent
    and test_dayofweek_consistent
)


if not dayofweek_consistent:

    raise ValueError(
        "Original DayOfWeek does not match Date. "
        "Please check the source dataset."
    )


# ============================================================
# STEP 4: PREPARE COMBINED HOLIDAY CALENDAR
# ============================================================

calendar_columns = [
    "Store",
    "Date",
    "StateHoliday",
    "SchoolHoliday"
]


train_calendar = train_working[
    calendar_columns
].copy()

test_calendar = test_working[
    calendar_columns
].copy()


# Preserve dataset identity and original row position.

train_calendar["_dataset"] = "train"

test_calendar["_dataset"] = "test"


train_calendar["_row_position"] = np.arange(
    len(train_calendar)
)

test_calendar["_row_position"] = np.arange(
    len(test_calendar)
)


# Combine available calendar records.

combined_calendar = pd.concat(
    [
        train_calendar,
        test_calendar
    ],
    ignore_index=True
)


combined_calendar["Date"] = pd.to_datetime(
    combined_calendar["Date"],
    errors="coerce"
)


if combined_calendar["Date"].isna().any():

    raise ValueError(
        "Invalid dates found in combined holiday calendar."
    )


if combined_calendar["Store"].isna().any():

    raise ValueError(
        "Missing Store identifiers found."
    )


if combined_calendar["StateHoliday"].isna().any():

    raise ValueError(
        "Missing StateHoliday values found."
    )


# Each Store-Date combination must be unique.

duplicate_store_date_count = (
    combined_calendar.duplicated(
        subset=["Store", "Date"]
    ).sum()
)


if duplicate_store_date_count > 0:

    raise ValueError(
        f"Duplicate Store-Date records found: "
        f"{duplicate_store_date_count}"
    )


# ============================================================
# STEP 5: IDENTIFY STATE AND SCHOOL HOLIDAYS
# ============================================================

state_holiday_values = (

    combined_calendar["StateHoliday"]

    .astype("string")

    .str.strip()

    .str.lower()

)


normal_day_values = {
    "0",
    "0.0",
    "none",
    ""
}


is_state_holiday = ~state_holiday_values.isin(
    normal_day_values
)


school_holiday_values = pd.to_numeric(

    combined_calendar["SchoolHoliday"],

    errors="coerce"

)


if school_holiday_values.isna().any():

    raise ValueError(
        "Missing or invalid SchoolHoliday values found."
    )


if not school_holiday_values.isin([0, 1]).all():

    raise ValueError(
        "SchoolHoliday must contain only 0 and 1."
    )


combined_calendar["_is_holiday"] = (

    is_state_holiday

    | school_holiday_values.eq(1)

)



# ============================================================
# STEP 6: CALCULATE HOLIDAY DISTANCES WITH GAP-AWARE SEGMENTS
# ============================================================

# Sort observations chronologically for each store.

combined_calendar = combined_calendar.sort_values(
    ["Store", "Date"],
    kind="mergesort"
).copy()


# ------------------------------------------------------------
# STEP 6.1: IDENTIFY MISSING CALENDAR-DATE GAPS
# ------------------------------------------------------------

# Calculate the number of days between consecutive
# available observations for each store.

combined_calendar["_date_gap_days"] = (
    combined_calendar.groupby(
        "Store",
        sort=False
    )["Date"].diff().dt.days
)


# A gap greater than one day indicates that one or more
# calendar dates are missing from the available records.

combined_calendar["_has_date_gap"] = (
    combined_calendar["_date_gap_days"].gt(1)
)


# ------------------------------------------------------------
# STEP 6.2: CREATE CONTINUOUS TIME-SERIES SEGMENTS
# ------------------------------------------------------------

# Each new missing-date gap starts a new segment.
# The first available observation of a store belongs
# to segment zero.

combined_calendar["_segment_id"] = (
    combined_calendar.groupby(
        "Store",
        sort=False
    )["_has_date_gap"]
    .cumsum()
    .astype("int16")
)


# ------------------------------------------------------------
# STEP 6.3: CREATE A MISSING-DATE AUDIT
# ------------------------------------------------------------

gap_mask = combined_calendar["_has_date_gap"]


gap_audit = combined_calendar.loc[
    gap_mask,
    ["Store", "Date", "_date_gap_days"]
].copy()


# Recover the last available date before each gap.

previous_available_dates = (
    combined_calendar.groupby(
        "Store",
        sort=False
    )["Date"].shift(1)
)


gap_audit["PreviousAvailableDate"] = (
    previous_available_dates.loc[gap_mask].to_numpy()
)


# Number of missing calendar days between two observations.

gap_audit["MissingCalendarDays"] = (
    gap_audit["_date_gap_days"].astype("int64") - 1
)


# ------------------------------------------------------------
# STEP 6.4: PRINT GAP AUDIT RESULTS
# ------------------------------------------------------------

affected_store_count = int(
    gap_audit["Store"].nunique()
)

total_missing_calendar_days = int(
    gap_audit["MissingCalendarDays"].sum()
)


print("\n" + "-" * 90)
print("STORE-WISE MISSING CALENDAR-DATE AUDIT")
print("-" * 90)

print(
    f"Stores with at least one date gap: "
    f"{affected_store_count:,}"
)

print(
    f"Total missing calendar-date records: "
    f"{total_missing_calendar_days:,}"
)

if not gap_audit.empty:

    print("\nMissing gap duration distribution:")

    print(
        gap_audit["MissingCalendarDays"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    print("\nFirst 10 detected gaps:")

    print(
        gap_audit.head(10).to_string(index=False)
    )


# ------------------------------------------------------------
# STEP 6.5: IDENTIFY HOLIDAY EVENT DATES
# ------------------------------------------------------------

combined_calendar["_holiday_date"] = (
    combined_calendar["Date"].where(
        combined_calendar["_is_holiday"]
    )
)


# ------------------------------------------------------------
# STEP 6.6: FIND PREVIOUS HOLIDAY WITHIN EACH SEGMENT
# ------------------------------------------------------------

# Important:
# Group by both Store and Segment ID.
# This prevents holiday information from crossing
# a missing calendar-date gap.

combined_calendar["_previous_holiday"] = (
    combined_calendar.groupby(
        ["Store", "_segment_id"],
        sort=False
    )["_holiday_date"].ffill()
)


# ------------------------------------------------------------
# STEP 6.7: FIND NEXT HOLIDAY WITHIN EACH SEGMENT
# ------------------------------------------------------------

combined_calendar["_next_holiday"] = (
    combined_calendar.groupby(
        ["Store", "_segment_id"],
        sort=False
    )["_holiday_date"].bfill()
)


# ------------------------------------------------------------
# STEP 6.8: CALCULATE HOLIDAY DISTANCE FEATURES
# ------------------------------------------------------------

combined_calendar["DaysSinceHoliday"] = (
    combined_calendar["Date"]
    - combined_calendar["_previous_holiday"]
).dt.days


combined_calendar["DaysUntilHoliday"] = (
    combined_calendar["_next_holiday"]
    - combined_calendar["Date"]
).dt.days


# ------------------------------------------------------------
# STEP 6.9: VALIDATE HOLIDAY DISTANCE LOGIC
# ------------------------------------------------------------

# Holiday distances must never be negative.

for feature in [
    "DaysSinceHoliday",
    "DaysUntilHoliday"
]:

    nonnegative = (
        combined_calendar[feature]
        .dropna()
        .ge(0)
        .all()
    )

    if not nonnegative:

        raise ValueError(
            f"Negative values detected in {feature}."
        )


# Both distances must be zero on an observed holiday date.

holiday_rows = combined_calendar["_is_holiday"]


same_day_holiday_valid = (
    combined_calendar.loc[
        holiday_rows,
        ["DaysSinceHoliday", "DaysUntilHoliday"]
    ].eq(0).all().all()
)


if not same_day_holiday_valid:

    raise ValueError(
        "Holiday dates must have zero distance "
        "for both holiday proximity features."
    )


# ------------------------------------------------------------
# STEP 6.10: VERIFY SEGMENT BOUNDARIES
# ------------------------------------------------------------

# Confirm that no holiday-distance calculation has
# used a holiday from another segment.

segment_group = combined_calendar.groupby(
    ["Store", "_segment_id"],
    sort=False
)

previous_check = segment_group[
    "_holiday_date"
].transform("ffill")

next_check = segment_group[
    "_holiday_date"
].transform("bfill")


segment_logic_valid = (
    combined_calendar["_previous_holiday"]
    .equals(previous_check)
    and
    combined_calendar["_next_holiday"]
    .equals(next_check)
)


if not segment_logic_valid:

    raise ValueError(
        "Holiday distance calculations crossed "
        "a segment boundary."
    )


print("\n" + "-" * 90)
print("GAP-AWARE HOLIDAY FEATURE VALIDATION")
print("-" * 90)

print(
    "Continuous segments created: ",
    combined_calendar.groupby(
        "Store"
    )["_segment_id"].nunique().sum()
)

print(
    "Holiday same-day zero validation: ",
    same_day_holiday_valid
)

print(
    "Segment boundary validation: ",
    segment_logic_valid
)

print(
    "Gap-aware holiday calculation completed."
)

# ============================================================
# STEP 7: RESTORE ORIGINAL TRAIN AND TEST ROW ORDER
# ============================================================

train_calendar_result = (

    combined_calendar.loc[

        combined_calendar["_dataset"].eq("train")

    ]

    .sort_values("_row_position")

)


test_calendar_result = (

    combined_calendar.loc[

        combined_calendar["_dataset"].eq("test")

    ]

    .sort_values("_row_position")

)


# Assign holiday features to original DataFrames.

holiday_features = [
    "DaysSinceHoliday",
    "DaysUntilHoliday"
]


for feature in holiday_features:

    train_working[feature] = (
        train_calendar_result[feature].to_numpy()
    )

    test_working[feature] = (
        test_calendar_result[feature].to_numpy()
    )


# ============================================================
# STEP 8: VALIDATE ALL CREATED DATE FEATURES
# ============================================================

calendar_features = [

    "Year",
    "Month",
    "Day",
    "Date_DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "IsWeekend",
    "Month_Position"

]


date_features = (
    calendar_features + holiday_features
)


missing_date_features_train = [

    feature

    for feature in date_features

    if feature not in train_working.columns

]


missing_date_features_test = [

    feature

    for feature in date_features

    if feature not in test_working.columns

]


if missing_date_features_train or missing_date_features_test:

    raise KeyError(

        f"Missing date features in training: "
        f"{missing_date_features_train}; "

        f"Missing date features in test: "
        f"{missing_date_features_test}"

    )


# ============================================================
# STEP 9: VALIDATE HOLIDAY DISTANCE FEATURES
# ============================================================

def validate_holiday_features(frame, dataset_name):

    state_values = (

        frame["StateHoliday"]

        .astype("string")

        .str.strip()

        .str.lower()

    )


    state_event = ~state_values.isin(
        normal_day_values
    )


    school_event = pd.to_numeric(
        frame["SchoolHoliday"],
        errors="coerce"
    ).eq(1)


    holiday_event = (
        state_event | school_event
    )


    results = {}


    for feature in holiday_features:

        values = pd.to_numeric(
            frame[feature],
            errors="coerce"
        )


        nonnegative = (
            values.dropna().ge(0).all()
        )


        finite = np.isfinite(
            values.dropna().to_numpy(dtype=float)
        ).all()


        same_day_zero = (
            values.loc[holiday_event].eq(0).all()
        )


        results[feature] = all([

            nonnegative,

            finite,

            same_day_zero

        ])


        print(f"\n{dataset_name}: {feature}")

        print(
            f"Missing Values: "
            f"{values.isna().sum():,}"
        )

        print(
            f"Non-negative: {nonnegative}"
        )

        print(
            f"Finite: {finite}"
        )

        print(
            f"Holiday Day Equals Zero: {same_day_zero}"
        )


    return all(results.values())


train_holiday_valid = validate_holiday_features(
    train_working,
    "Training Dataset"
)


test_holiday_valid = validate_holiday_features(
    test_working,
    "Test Dataset"
)


calendar_feature_valid = all(

    train_working[feature].notna().all()

    and test_working[feature].notna().all()

    for feature in calendar_features

)


holiday_distance_features_valid = (

    train_holiday_valid

    and test_holiday_valid

)


# ============================================================
# STEP 10: FINAL SECTION 4 VALIDATION
# ============================================================

section4_successful = all([

    calendar_feature_valid,

    dayofweek_consistent,

    holiday_distance_features_valid

])


print("\n" + "-" * 90)
print("DATE FEATURE SUMMARY")
print("-" * 90)


print(
    f"\nCalendar Features: {calendar_features}"
)

print(
    f"Holiday Features: {holiday_features}"
)

print(
    f"\nTraining Rows Preserved: "
    f"{len(train_working) == len(train_df)}"
)

print(
    f"Test Rows Preserved: "
    f"{len(test_working) == len(test_df)}"
)

print(
    f"Calendar Features Valid: "
    f"{calendar_feature_valid}"
)

print(
    f"DayOfWeek Consistency Valid: "
    f"{dayofweek_consistent}"
)

print(
    f"Holiday Distance Features Valid: "
    f"{holiday_distance_features_valid}"
)


print("\n" + "=" * 90)

print(
    f"SECTION 4 COMPLETED SUCCESSFULLY: "
    f"{section4_successful}"
)

print("=" * 90)


if not section4_successful:

    raise ValueError(
        "Section 4 validation failed. "
        "Review the feature checks above."
    )

else:

    logger.info(
        "Section 4 date and holiday feature engineering "
        "completed successfully."
    )

SECTION 4: DATE-BASED FEATURE ENGINEERING

Training Original DayOfWeek Matches Date: True

Test Original DayOfWeek Matches Date: True

------------------------------------------------------------------------------------------
STORE-WISE MISSING CALENDAR-DATE AUDIT
------------------------------------------------------------------------------------------
Stores with at least one date gap: 180
Total missing calendar-date records: 33,120

Missing gap duration distribution:
MissingCalendarDays
184    180

First 10 detected gaps:
 Store       Date  _date_gap_days PreviousAvailableDate  MissingCalendarDays
    13 2015-01-01           185.0            2014-06-30                  184
    20 2015-01-01           185.0            2014-06-30                  184
    22 2015-01-01           185.0            2014-06-30                  184
    32 2015-01-01           185.0            2014-06-30                  184
    36 2015-01-01           185.0            2014-06-30                  184
    41 

2026-11-06 16:12:22,104 | INFO | Project6_Preprocessing | Section 4 date and holiday feature engineering completed successfully.



Training Dataset: DaysSinceHoliday
Missing Values: 0
Non-negative: True
Finite: True
Holiday Day Equals Zero: True

Training Dataset: DaysUntilHoliday
Missing Values: 1,800
Non-negative: True
Finite: True
Holiday Day Equals Zero: True

Test Dataset: DaysSinceHoliday
Missing Values: 0
Non-negative: True
Finite: True
Holiday Day Equals Zero: True

Test Dataset: DaysUntilHoliday
Missing Values: 16,782
Non-negative: True
Finite: True
Holiday Day Equals Zero: True

------------------------------------------------------------------------------------------
DATE FEATURE SUMMARY
------------------------------------------------------------------------------------------

Calendar Features: ['Year', 'Month', 'Day', 'Date_DayOfWeek', 'WeekOfYear', 'Quarter', 'IsWeekend', 'Month_Position']
Holiday Features: ['DaysSinceHoliday', 'DaysUntilHoliday']

Training Rows Preserved: True
Test Rows Preserved: True
Calendar Features Valid: True
DayOfWeek Consistency Valid: True
Holiday Distance Features Valid:

# SECTION 5: COMPETITION FEATURE ENGINEERING

## Purpose

Competition-related information can influence store sales.
This section creates meaningful features from competitor distance
and competition opening information.

## Features Created

1. CompetitionDistanceMissing
2. CompetitionOpenDate
3. CompetitionOpenInfoMissing
4. CompetitionOpenDurationMonths

## Feature Engineering Logic

- Original CompetitionDistance is retained.
- Missing competition distance is identified using a binary indicator.
- Competition opening month and year are combined into an approximate date.
- Competition duration is calculated in months.
- If the competitor has not opened by the observation date,
  duration is assigned zero.
- If opening information is genuinely unavailable, duration remains
  missing for later training-only imputation.

## Leakage Prevention

No statistical imputation is performed in this section.
Missing numerical values will be imputed using the training portion
after the temporal train-validation split.

## Expected Outcome

Both training and test datasets will contain consistent competition
features without full-dataset statistical imputation.

In [373]:
# ============================================================
# SECTION 5: COMPETITION FEATURE ENGINEERING
# ============================================================

print("=" * 85)
print("COMPETITION FEATURE ENGINEERING")
print("=" * 85)


def create_competition_features(df):
    """
    Creates competition-related features.

    No statistical imputation is performed here.
    Future competition opening dates are represented by
    zero months of competition activity.
    """

    df = df.copy()

    # --------------------------------------------------------
    # STEP 1: Remove Previously Engineered Columns
    # --------------------------------------------------------

    old_columns = [
        "CompetitionDistance_missing",
        "CompetitionDistanceMissing",
        "CompetitionDistanceFilled",
        "CompetitionOpenInfoMissing",
        "CompetitionOpenDate",
        "CompetitionOpenDurationMonths"
    ]

    df = df.drop(
        columns=old_columns,
        errors="ignore"
    )

    # --------------------------------------------------------
    # STEP 2: Ensure Date Is Datetime
    # --------------------------------------------------------

    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # STEP 3: Competition Distance Missing Indicator
    # --------------------------------------------------------

    df["CompetitionDistanceMissing"] = (
        df["CompetitionDistance"]
        .isna()
        .astype("int8")
    )

    # --------------------------------------------------------
    # STEP 4: Create Competition Opening Date
    # --------------------------------------------------------

    opening_year = pd.to_numeric(
        df["CompetitionOpenSinceYear"],
        errors="coerce"
    )

    opening_month = pd.to_numeric(
        df["CompetitionOpenSinceMonth"],
        errors="coerce"
    )

    df["CompetitionOpenDate"] = pd.to_datetime(
        {
            "year": opening_year,
            "month": opening_month,
            "day": 1
        },
        errors="coerce"
    )

    # Missing or invalid opening date indicator
    df["CompetitionOpenInfoMissing"] = (
        df["CompetitionOpenDate"]
        .isna()
        .astype("int8")
    )

    # --------------------------------------------------------
    # STEP 5: Calculate Competition Duration
    # --------------------------------------------------------

    raw_duration_months = (
        (
            df["Date"].dt.year
            - df["CompetitionOpenDate"].dt.year
        ) * 12
        +
        (
            df["Date"].dt.month
            - df["CompetitionOpenDate"].dt.month
        )
    )

    # --------------------------------------------------------
    # STEP 6: Handle Future Competition Opening Dates
    # --------------------------------------------------------

    # A future competition opening date means that competition
    # has not started at the observation date.
    # Therefore, duration is zero, not missing.

    df["CompetitionOpenDurationMonths"] = (
        raw_duration_months.clip(lower=0)
    )

    return df


# ============================================================
# STEP 7: APPLY FEATURE ENGINEERING
# ============================================================

train_working = create_competition_features(
    train_working
)

test_working = create_competition_features(
    test_working
)


# ============================================================
# STEP 8: VALIDATE CREATED FEATURES
# ============================================================

competition_features = [
    "CompetitionDistance",
    "CompetitionDistanceMissing",
    "CompetitionOpenInfoMissing",
    "CompetitionOpenDate",
    "CompetitionOpenDurationMonths"
]


print("\n" + "-" * 85)
print("FEATURE AVAILABILITY CHECK")
print("-" * 85)

for feature in competition_features:

    print(
        f"{feature:<35} "
        f"Train: {feature in train_working.columns} | "
        f"Test: {feature in test_working.columns}"
    )


# ============================================================
# STEP 9: MISSING VALUE SUMMARY
# ============================================================

print("\n" + "-" * 85)
print("COMPETITION FEATURE MISSING VALUE SUMMARY")
print("-" * 85)

print("\nTraining Dataset:")

print(
    train_working[competition_features]
    .isna()
    .sum()
)

print("\nTest Dataset:")

print(
    test_working[competition_features]
    .isna()
    .sum()
)


# ============================================================
# STEP 10: FUTURE OPENING DATE VALIDATION
# ============================================================

print("\n" + "-" * 85)
print("FUTURE COMPETITION OPENING VALIDATION")
print("-" * 85)


future_opening_train = (
    train_working["CompetitionOpenDate"]
    > train_working["Date"]
).sum()


future_opening_test = (
    test_working["CompetitionOpenDate"]
    > test_working["Date"]
).sum()


print(
    f"\nFuture Opening Records in Training: "
    f"{future_opening_train:,}"
)

print(
    f"Future Opening Records in Test: "
    f"{future_opening_test:,}"
)


# All future competition durations must be zero.

future_duration_train = train_working.loc[
    train_working["CompetitionOpenDate"]
    > train_working["Date"],
    "CompetitionOpenDurationMonths"
]

future_duration_test = test_working.loc[
    test_working["CompetitionOpenDate"]
    > test_working["Date"],
    "CompetitionOpenDurationMonths"
]


future_duration_valid = (
    future_duration_train.eq(0).all()
    and future_duration_test.eq(0).all()
)


print(
    f"\nFuture Competition Duration Correct: "
    f"{future_duration_valid}"
)


# No negative durations should remain.

no_negative_durations = (
    train_working[
        "CompetitionOpenDurationMonths"
    ].dropna().ge(0).all()
    and
    test_working[
        "CompetitionOpenDurationMonths"
    ].dropna().ge(0).all()
)


print(
    f"Negative Durations Remaining: "
    f"{not no_negative_durations}"
)


# ============================================================
# STEP 11: DATA PREVIEW
# ============================================================

preview_columns = [
    "Date",
    "CompetitionDistance",
    "CompetitionDistanceMissing",
    "CompetitionOpenDate",
    "CompetitionOpenInfoMissing",
    "CompetitionOpenDurationMonths"
]


print("\n" + "-" * 85)
print("TRAINING DATA PREVIEW")
print("-" * 85)

display(
    train_working[preview_columns].head()
)


# ============================================================
# STEP 12: FINAL COMPLETION
# ============================================================

competition_feature_validation = (
    future_duration_valid
    and no_negative_durations
    and "CompetitionDistanceFilled"
    not in train_working.columns
)


print("\n" + "=" * 85)
print("COMPETITION FEATURE ENGINEERING STATUS")
print("=" * 85)

print(
    f"\nCompetition Features Valid: "
    f"{competition_feature_validation}"
)

print("=" * 85)


if competition_feature_validation:

    logger.info(
        "Competition feature engineering completed successfully."
    )

else:

    logger.warning(
        "Competition feature validation requires review."
    )

COMPETITION FEATURE ENGINEERING

-------------------------------------------------------------------------------------
FEATURE AVAILABILITY CHECK
-------------------------------------------------------------------------------------
CompetitionDistance                 Train: True | Test: True
CompetitionDistanceMissing          Train: True | Test: True
CompetitionOpenInfoMissing          Train: True | Test: True
CompetitionOpenDate                 Train: True | Test: True
CompetitionOpenDurationMonths       Train: True | Test: True

-------------------------------------------------------------------------------------
COMPETITION FEATURE MISSING VALUE SUMMARY
-------------------------------------------------------------------------------------

Training Dataset:
CompetitionDistance                2642
CompetitionDistanceMissing            0
CompetitionOpenInfoMissing            0
CompetitionOpenDate              323348
CompetitionOpenDurationMonths    323348
dtype: int64

Test Dataset:
C

,Date,CompetitionDistance,CompetitionDistanceMissing,CompetitionOpenDate,CompetitionOpenInfoMissing,CompetitionOpenDurationMonths
0,2015-07-31,1270.0,0,2008-09-01,0,82.0
1,2015-07-31,570.0,0,2007-11-01,0,92.0
2,2015-07-31,14130.0,0,2006-12-01,0,103.0
3,2015-07-31,620.0,0,2009-09-01,0,70.0
4,2015-07-31,29910.0,0,2015-04-01,0,3.0


2026-11-06 16:12:22,517 | INFO | Project6_Preprocessing | Competition feature engineering completed successfully.



COMPETITION FEATURE ENGINEERING STATUS

Competition Features Valid: True


# SECTION 6: PROMOTION FEATURE ENGINEERING

## Purpose

This section creates meaningful features from regular promotional
activities and the extended Promo2 program.

## Original Variables

- Promo
- Promo2
- Promo2SinceWeek
- Promo2SinceYear
- PromoInterval

## Engineered Features

1. Promo2InfoMissing
2. PromoIntervalMissing
3. Promo2StartDate
4. Promo2DurationMonths
5. Promo2NotStarted
6. Promo2IntervalMonth

## Feature Engineering Logic

- Promo2 = 0 indicates that the store does not participate in Promo2.
- Inactive Promo2 stores receive a duration of zero.
- Future promotion start dates receive a duration of zero.
- Genuine missing promotion information is preserved.
- Promo2IntervalMonth identifies whether the observation month
  belongs to the store's configured recurring promotion interval,
  after the Promo2 program has started.

## Missing Value Strategy

Structural missingness for inactive Promo2 stores is handled
separately from unavailable information for active Promo2 stores.

No statistical imputation is performed in this section.

Remaining numerical missing values will be handled after the
temporal train-validation split using the training-only
preprocessing pipeline.

## Expected Outcome

Training and test datasets will contain consistent promotion
features without using future target information or full-dataset
statistical imputation.

In [374]:
# ============================================================
# SECTION 6: PROMOTION FEATURE ENGINEERING
# ============================================================

print("=" * 85)
print("PROMOTION FEATURE ENGINEERING")
print("=" * 85)


# ============================================================
# STEP 1: DEFINE PROMOTION MONTH MAPPING
# ============================================================

# Standard month abbreviations used in PromoInterval.

MONTH_MAP = {
    "jan": 1,
    "feb": 2,
    "mar": 3,
    "apr": 4,
    "may": 5,
    "jun": 6,
    "jul": 7,
    "aug": 8,
    "sep": 9,
    "sept": 9,
    "oct": 10,
    "nov": 11,
    "dec": 12
}


# ============================================================
# STEP 2: CREATE PROMOTION FEATURE FUNCTION
# ============================================================

def create_promo_features(df):
    """
    Creates promotion-related features without performing
    statistical imputation.

    Structural missingness is distinguished from genuinely
    unavailable promotion information.

    Future promotion start dates are handled as zero duration.
    """

    df = df.copy()

    # --------------------------------------------------------
    # 2.1 Remove Previously Engineered Columns
    # --------------------------------------------------------

    old_columns = [
        "Promo2InfoMissing",
        "PromoIntervalMissing",
        "Promo2StartDate",
        "Promo2DurationMonths",
        "Promo2NotStarted",
        "Promo2IntervalMonth"
    ]

    df = df.drop(
        columns=old_columns,
        errors="ignore"
    )

    # --------------------------------------------------------
    # 2.2 Validate Observation Date and Promo2
    # --------------------------------------------------------

    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )

    promo2 = pd.to_numeric(
        df["Promo2"],
        errors="coerce"
    )

    if promo2.isna().any() or not promo2.isin([0, 1]).all():

        raise ValueError(
            "Promo2 must contain only valid 0 and 1 values."
        )

    promo2_active = promo2.eq(1)
    promo2_inactive = promo2.eq(0)

    # --------------------------------------------------------
    # 2.3 Extract and Validate Promo2 Start Information
    # --------------------------------------------------------

    promo_week = pd.to_numeric(
        df["Promo2SinceWeek"],
        errors="coerce"
    )

    promo_year = pd.to_numeric(
        df["Promo2SinceYear"],
        errors="coerce"
    )

    valid_start_information = (
        promo2_active
        & promo_week.notna()
        & promo_year.notna()
        & promo_week.between(1, 53)
        & promo_week.mod(1).eq(0)
        & promo_year.mod(1).eq(0)
    )

    # --------------------------------------------------------
    # 2.4 Create Promo2 Start Date
    # --------------------------------------------------------

    promo2_start_date = pd.Series(
        pd.NaT,
        index=df.index,
        dtype="datetime64[ns]"
    )

    iso_week_strings = (
        promo_year.loc[valid_start_information]
        .astype("int64")
        .astype(str)
        +
        "-W"
        +
        promo_week.loc[valid_start_information]
        .astype("int64")
        .astype(str)
        .str.zfill(2)
        +
        "-1"
    )

    parsed_start_dates = pd.to_datetime(
        iso_week_strings,
        format="%G-W%V-%u",
        errors="coerce"
    )

    promo2_start_date.loc[
        valid_start_information
    ] = parsed_start_dates.to_numpy()

    df["Promo2StartDate"] = promo2_start_date

    # --------------------------------------------------------
    # 2.5 Promo2 Information Missing Indicator
    # --------------------------------------------------------

    # Only active Promo2 stores are checked for missing
    # promotion start information.
    #
    # Inactive stores are not marked as missing.

    df["Promo2InfoMissing"] = (
        promo2_active
        & df["Promo2StartDate"].isna()
    ).astype("int8")

    # --------------------------------------------------------
    # 2.6 Normalize PromoInterval
    # --------------------------------------------------------

    normalized_interval = (
        df["PromoInterval"]
        .astype("string")
        .str.lower()
        .str.replace(r"\s+", "", regex=True)
    )

    # --------------------------------------------------------
    # 2.7 Parse Promotion Interval Months
    # --------------------------------------------------------

    interval_to_months = {}

    for interval_value in normalized_interval.dropna().unique():

        month_tokens = interval_value.split(",")

        parsed_months = [
            MONTH_MAP[token]
            for token in month_tokens
            if token in MONTH_MAP
        ]

        # Accept an interval only if every token is valid.

        if (
            len(parsed_months) == len(month_tokens)
            and len(parsed_months) > 0
        ):

            interval_to_months[
                interval_value
            ] = set(parsed_months)

        else:

            interval_to_months[
                interval_value
            ] = set()

    interval_valid_lookup = {
        key: bool(value)
        for key, value in interval_to_months.items()
    }

    interval_available = (
        normalized_interval
        .map(interval_valid_lookup)
        .eq(True)
    )

    # --------------------------------------------------------
    # 2.8 PromoInterval Missing Indicator
    # --------------------------------------------------------

    # Inactive Promo2 stores are not marked as having
    # missing promotion intervals.

    df["PromoIntervalMissing"] = (
        promo2_active
        & ~interval_available
    ).astype("int8")

    # --------------------------------------------------------
    # 2.9 Calculate Promo2 Duration in Months
    # --------------------------------------------------------

    promo2_duration = pd.Series(
        np.nan,
        index=df.index,
        dtype="float64"
    )

    # Inactive stores have zero Promo2 duration.

    promo2_duration.loc[
        promo2_inactive
    ] = 0.0

    # Calculate duration only when valid start information
    # and observation date are available.

    known_start = (
        promo2_active
        & df["Promo2StartDate"].notna()
        & df["Date"].notna()
    )

    duration_months = (
        (
            df.loc[known_start, "Date"].dt.year
            -
            df.loc[known_start, "Promo2StartDate"].dt.year
        ) * 12
        +
        (
            df.loc[known_start, "Date"].dt.month
            -
            df.loc[known_start, "Promo2StartDate"].dt.month
        )
    )

    # Future start dates receive zero duration.

    promo2_duration.loc[known_start] = (
        duration_months.clip(lower=0).astype(float)
    )

    df["Promo2DurationMonths"] = promo2_duration

    # --------------------------------------------------------
    # 2.10 Create Future Promotion Indicator
    # --------------------------------------------------------

    future_promo_start = (
        promo2_active
        & df["Promo2StartDate"].notna()
        & df["Date"].notna()
        & (
            df["Promo2StartDate"]
            > df["Date"]
        )
    )

    df["Promo2NotStarted"] = (
        future_promo_start
    ).astype("int8")

    # --------------------------------------------------------
    # 2.11 Create Promo2 Interval Month Feature
    # --------------------------------------------------------

    # This feature identifies whether the observation month
    # matches a configured Promo2 interval month.
    #
    # It is evaluated only after the program has started.

    promo2_interval_month = pd.Series(
        np.nan,
        index=df.index,
        dtype="float64"
    )

    # Inactive stores receive zero.

    promo2_interval_month.loc[
        promo2_inactive
    ] = 0.0

    # Active stores whose program has not started yet
    # also receive zero.

    promo2_interval_month.loc[
        future_promo_start
    ] = 0.0

    known_interval_information = (
        promo2_active
        & df["Promo2StartDate"].notna()
        & df["Date"].notna()
        & interval_available
    )

    promotion_already_started = (
        known_interval_information
        & ~future_promo_start
    )

    for interval_value, months in interval_to_months.items():

        if not months:
            continue

        matching_rows = (
            promotion_already_started
            & normalized_interval.eq(interval_value)
        )

        promo2_interval_month.loc[
            matching_rows
        ] = (
            df.loc[
                matching_rows,
                "Date"
            ].dt.month.isin(months).astype(float)
        )

    df["Promo2IntervalMonth"] = (
        promo2_interval_month
    )

    return df


# ============================================================
# STEP 3: APPLY FEATURE ENGINEERING
# ============================================================

train_working = create_promo_features(
    train_working
)

test_working = create_promo_features(
    test_working
)


# ============================================================
# STEP 4: VALIDATE CREATED FEATURES
# ============================================================

promo_features = [
    "Promo2InfoMissing",
    "PromoIntervalMissing",
    "Promo2StartDate",
    "Promo2DurationMonths",
    "Promo2NotStarted",
    "Promo2IntervalMonth"
]


print("\n" + "-" * 85)
print("FEATURE AVAILABILITY CHECK")
print("-" * 85)

for feature in promo_features:

    print(
        f"{feature:<30}"
        f"Train: {feature in train_working.columns} | "
        f"Test: {feature in test_working.columns}"
    )


# ============================================================
# STEP 5: PROMO2 STRUCTURAL MISSINGNESS CHECK
# ============================================================

print("\n" + "-" * 85)
print("PROMO2 STRUCTURAL MISSINGNESS CHECK")
print("-" * 85)

print("\nTraining Dataset:")

print(
    train_working
    .groupby("Promo2")[
        ["Promo2SinceWeek", "Promo2SinceYear"]
    ]
    .count()
)

print("\nTest Dataset:")

print(
    test_working
    .groupby("Promo2")[
        ["Promo2SinceWeek", "Promo2SinceYear"]
    ]
    .count()
)


# ============================================================
# STEP 6: DURATION VALIDATION
# ============================================================

print("\n" + "-" * 85)
print("PROMO2 DURATION VALIDATION")
print("-" * 85)


def validate_promo_duration(df, dataset_name):

    inactive_mask = df["Promo2"].eq(0)

    future_mask = (
        df["Promo2"].eq(1)
        & df["Promo2StartDate"].notna()
        & (df["Promo2StartDate"] > df["Date"])
    )

    inactive_zero = (
        df.loc[
            inactive_mask,
            "Promo2DurationMonths"
        ].eq(0).all()
    )

    future_zero = (
        df.loc[
            future_mask,
            "Promo2DurationMonths"
        ].eq(0).all()
    )

    nonnegative = (
        df["Promo2DurationMonths"]
        .dropna()
        .ge(0)
        .all()
    )

    print(f"\n{dataset_name}")

    print(
        f"Inactive Promo2 Duration Zero: {inactive_zero}"
    )

    print(
        f"Future Promo2 Duration Zero: {future_zero}"
    )

    print(
        f"No Negative Duration: {nonnegative}"
    )

    return (
        inactive_zero
        and future_zero
        and nonnegative
    )


train_duration_valid = validate_promo_duration(
    train_working,
    "Training Dataset"
)

test_duration_valid = validate_promo_duration(
    test_working,
    "Test Dataset"
)


# ============================================================
# STEP 7: MISSING VALUE SUMMARY
# ============================================================

print("\n" + "-" * 85)
print("PROMOTION FEATURE MISSING VALUE SUMMARY")
print("-" * 85)

print("\nTraining Dataset:")

print(
    train_working[promo_features]
    .isna()
    .sum()
)

print("\nTest Dataset:")

print(
    test_working[promo_features]
    .isna()
    .sum()
)


# ============================================================
# STEP 8: PROMO2 DISTRIBUTION
# ============================================================

print("\n" + "-" * 85)
print("PROMO2 DURATION DISTRIBUTION")
print("-" * 85)

print(
    train_working
    .groupby("Promo2")["Promo2DurationMonths"]
    .agg(
        ["count", "min", "max", "mean"]
    )
)


# ============================================================
# STEP 9: FEATURE DISTRIBUTION CHECK
# ============================================================

print("\n" + "-" * 85)
print("ENGINEERED PROMOTION FEATURE DISTRIBUTION")
print("-" * 85)

for feature in [
    "Promo2InfoMissing",
    "PromoIntervalMissing",
    "Promo2NotStarted",
    "Promo2IntervalMonth"
]:

    print(f"\n{feature}:")

    print(
        train_working[feature]
        .value_counts(dropna=False)
    )


# ============================================================
# STEP 10: PREVIEW
# ============================================================

preview_columns = [
    "Date",
    "Promo",
    "Promo2",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",
    "Promo2InfoMissing",
    "PromoIntervalMissing",
    "Promo2StartDate",
    "Promo2DurationMonths",
    "Promo2NotStarted",
    "Promo2IntervalMonth"
]


print("\n" + "-" * 85)
print("TRAINING DATA PREVIEW")
print("-" * 85)

display(
    train_working[preview_columns].head(10)
)


# ============================================================
# STEP 11: FINAL FEATURE VALIDATION
# ============================================================

required_features_exist = all(
    feature in train_working.columns
    and feature in test_working.columns
    for feature in promo_features
)

promotion_feature_successful = (
    required_features_exist
    and train_duration_valid
    and test_duration_valid
)


print("\n" + "=" * 85)
print("FINAL PROMOTION FEATURE ENGINEERING STATUS")
print("=" * 85)

print(
    f"\nRequired Features Available: "
    f"{required_features_exist}"
)

print(
    f"Promotion Features Valid: "
    f"{promotion_feature_successful}"
)

print("=" * 85)


# ============================================================
# STEP 12: LOG COMPLETION
# ============================================================

if promotion_feature_successful:

    logger.info(
        "Promotion feature engineering completed successfully."
    )

else:

    logger.warning(
        "Promotion feature validation requires review."
    )

PROMOTION FEATURE ENGINEERING

-------------------------------------------------------------------------------------
FEATURE AVAILABILITY CHECK
-------------------------------------------------------------------------------------
Promo2InfoMissing             Train: True | Test: True
PromoIntervalMissing          Train: True | Test: True
Promo2StartDate               Train: True | Test: True
Promo2DurationMonths          Train: True | Test: True
Promo2NotStarted              Train: True | Test: True
Promo2IntervalMonth           Train: True | Test: True

-------------------------------------------------------------------------------------
PROMO2 STRUCTURAL MISSINGNESS CHECK
-------------------------------------------------------------------------------------

Training Dataset:
        Promo2SinceWeek  Promo2SinceYear
Promo2                                  
0                     0                0
1                509178           509178

Test Dataset:
        Promo2SinceWeek  Promo2Si

,Date,Promo,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Promo2InfoMissing,PromoIntervalMissing,Promo2StartDate,Promo2DurationMonths,Promo2NotStarted,Promo2IntervalMonth
0,2015-07-31,1,0,NaN,NaN,NaN,0,0,NaT,0.0,0,0.0
1,2015-07-31,1,1,13.0,2010.0,"Jan,Apr,Jul,Oct",0,0,2010-03-29,64.0,0,1.0
2,2015-07-31,1,1,14.0,2011.0,"Jan,Apr,Jul,Oct",0,0,2011-04-04,51.0,0,1.0
3,2015-07-31,1,0,NaN,NaN,NaN,0,0,NaT,0.0,0,0.0
4,2015-07-31,1,0,NaN,NaN,NaN,0,0,NaT,0.0,0,0.0
5,2015-07-31,1,0,NaN,NaN,NaN,0,0,NaT,0.0,0,0.0
6,2015-07-31,1,0,NaN,NaN,NaN,0,0,NaT,0.0,0,0.0
7,2015-07-31,1,0,NaN,NaN,NaN,0,0,NaT,0.0,0,0.0
8,2015-07-31,1,0,NaN,NaN,NaN,0,0,NaT,0.0,0,0.0
9,2015-07-31,1,0,NaN,NaN,NaN,0,0,NaT,0.0,0,0.0


2026-11-06 16:12:24,429 | INFO | Project6_Preprocessing | Promotion feature engineering completed successfully.



FINAL PROMOTION FEATURE ENGINEERING STATUS

Required Features Available: True
Promotion Features Valid: True


# ============================================================
# # SECTION 7: STORE-LEVEL FEATURE VALIDATION & PREPARATION
# ============================================================

## Purpose of This Section

Store characteristics can influence sales performance and customer behavior. The Rossmann dataset contains several store-level variables that describe the type and assortment characteristics of each store.

This section will prepare and validate store-level features for use in the forecasting workflow.

### Store-Level Variables

The following variables provide store-specific information:

- `Store`
- `StoreType`
- `Assortment`

### Feature Engineering Strategy

The store identifier will be preserved for grouping and store-specific analysis. However, it will not automatically be treated as a continuous numerical variable because store numbers represent identifiers rather than meaningful quantities.

The categorical variables `StoreType` and `Assortment` will be validated and prepared for categorical preprocessing in a later section.

### Important Methodology

Store-level characteristics are static or store-specific information and are available before making future sales predictions.

No store-level feature created in this section uses the target variable `Sales`. Therefore, these features do not introduce target leakage.

Categorical encoding will be performed later using a preprocessing strategy that is applied consistently to both the training and test datasets.

### Expected Outcome

After this section:

- Store-level variables will be validated.
- Store identifiers will be retained for grouping and analysis.
- Categorical store characteristics will be ready for later encoding.
- No target-derived store statistics will be created at this stage, helping to prevent target leakage.

In [375]:
# ============================================================
# SECTION 7: STORE-LEVEL FEATURE ENGINEERING
# ============================================================

print("=" * 85)
print("STORE-LEVEL FEATURE VALIDATION & PREPARATION")
print("=" * 85)

# ------------------------------------------------------------
# STEP 1: Define Store-Level Features
# ------------------------------------------------------------

store_features = [
    "Store",
    "StoreType",
    "Assortment"
]

# ------------------------------------------------------------
# STEP 2: Check Feature Availability
# ------------------------------------------------------------

print("\n" + "-" * 85)
print("FEATURE AVAILABILITY CHECK")
print("-" * 85)

for feature in store_features:
    print(
        f"{feature:<15} "
        f"Train: {feature in train_working.columns} | "
        f"Test: {feature in test_working.columns}"
    )


# ------------------------------------------------------------
# STEP 3: Missing Value Check
# ------------------------------------------------------------

print("\n" + "-" * 85)
print("MISSING VALUE CHECK")
print("-" * 85)

print("\nTraining Dataset:")
print(train_working[store_features].isna().sum())

print("\nTest Dataset:")
print(test_working[store_features].isna().sum())


# ------------------------------------------------------------
# STEP 4: Unique Value Summary
# ------------------------------------------------------------

print("\n" + "-" * 85)
print("UNIQUE VALUE SUMMARY")
print("-" * 85)

for feature in store_features:

    print(f"\n{feature}:")

    print(
        f"Training unique values: "
        f"{train_working[feature].nunique()}"
    )

    print(
        f"Test unique values: "
        f"{test_working[feature].nunique()}"
    )


# ------------------------------------------------------------
# STEP 5: Validate Store Coverage
# ------------------------------------------------------------

print("\n" + "-" * 85)
print("STORE COVERAGE VALIDATION")
print("-" * 85)

train_stores = set(train_working["Store"].unique())
test_stores = set(test_working["Store"].unique())

common_stores = train_stores.intersection(test_stores)
test_only_stores = test_stores - train_stores

print(f"\nUnique Stores in Training: {len(train_stores)}")
print(f"Unique Stores in Test:     {len(test_stores)}")
print(f"Common Stores:             {len(common_stores)}")
print(f"Test-only Stores:          {len(test_only_stores)}")

if len(test_only_stores) == 0:
    print("\nAll test stores are present in the training dataset.")
else:
    print("\nWarning: Some test stores are not present in training.")


# ------------------------------------------------------------
# STEP 6: StoreType Distribution
# ------------------------------------------------------------

print("\n" + "-" * 85)
print("STORE TYPE DISTRIBUTION")
print("-" * 85)

print("\nTraining Dataset:")
print(
    train_working["StoreType"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nTest Dataset:")
print(
    test_working["StoreType"]
    .value_counts(dropna=False)
    .sort_index()
)


# ------------------------------------------------------------
# STEP 7: Assortment Distribution
# ------------------------------------------------------------

print("\n" + "-" * 85)
print("ASSORTMENT DISTRIBUTION")
print("-" * 85)

print("\nTraining Dataset:")
print(
    train_working["Assortment"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nTest Dataset:")
print(
    test_working["Assortment"]
    .value_counts(dropna=False)
    .sort_index()
)


# ------------------------------------------------------------
# STEP 8: Completion
# ------------------------------------------------------------

logger.info(
    "Store-level feature validation and preparation completed successfully."
)

print("\n" + "=" * 85)
print("STORE-LEVEL FEATURE VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 85)

STORE-LEVEL FEATURE VALIDATION & PREPARATION

-------------------------------------------------------------------------------------
FEATURE AVAILABILITY CHECK
-------------------------------------------------------------------------------------
Store           Train: True | Test: True
StoreType       Train: True | Test: True
Assortment      Train: True | Test: True

-------------------------------------------------------------------------------------
MISSING VALUE CHECK
-------------------------------------------------------------------------------------

Training Dataset:
Store         0
StoreType     0
Assortment    0
dtype: int64

Test Dataset:
Store         0
StoreType     0
Assortment    0
dtype: int64

-------------------------------------------------------------------------------------
UNIQUE VALUE SUMMARY
-------------------------------------------------------------------------------------

Store:
Training unique values: 1115
Test unique values: 856

StoreType:
Training unique 

2026-11-06 16:12:24,657 | INFO | Project6_Preprocessing | Store-level feature validation and preparation completed successfully.


Assortment
a    537445
b      8294
c    471470
Name: count, dtype: int64

Test Dataset:
Assortment
a    20304
b      432
c    20352
Name: count, dtype: int64

STORE-LEVEL FEATURE VALIDATION COMPLETED SUCCESSFULLY


# SECTION 8: CATEGORICAL FEATURE VALIDATION AND PREPARATION

## Purpose

This section identifies and reviews categorical variables
available in the prepared Rossmann training and test datasets.

Categorical variables represent store characteristics,
holiday information, promotion schedules and calendar
categories.

## Categorical Features Reviewed

The following variables are reviewed:

- Store
- StoreType
- Assortment
- StateHoliday
- PromoInterval
- Month_Position

## Validation Activities

The following checks are performed:

1. Feature availability in training and test datasets.
2. Data types of categorical variables.
3. Missing value counts.
4. Number of unique categories.
5. Categories available only in training or test.
6. Category frequency distributions.
7. Store coverage across training and test datasets.

## Important Considerations

- Store is treated as a categorical identifier rather than
  a continuous numerical quantity.
- Store has high cardinality, so only a compact frequency
  summary will be displayed.
- PromoInterval may contain structural missing values
  because not every store participates in Promo2.
- Differences in category distributions between training
  and test datasets are reported but are not automatically
  considered errors.
- Month_Position is derived from the calendar features
  created in Section 4.

## Encoding Strategy

No categorical encoding is performed in this section.

Categorical encoding will be performed after temporal
train-validation splitting.

The OneHotEncoder will be fitted only on X_train and will
use handle_unknown="ignore" to handle unseen categories.

PromoInterval is reviewed here but excluded from final
model inputs because its relevant information is represented
through engineered promotion features and missingness indicators.

## Expected Outcome

A validated categorical feature summary containing
feature availability, data types, missing values,
unique categories and training-test category comparisons.

In [376]:
# ============================================================
# SECTION 8: CATEGORICAL FEATURE VALIDATION AND PREPARATION
# ============================================================

print("=" * 90)
print("SECTION 8: CATEGORICAL FEATURE VALIDATION")
print("=" * 90)


# ============================================================
# STEP 1: VALIDATE INPUT DATASETS
# ============================================================

if not isinstance(train_working, pd.DataFrame):

    raise TypeError(
        "train_working must be a pandas DataFrame. "
        "Please execute the previous sections first."
    )


if not isinstance(test_working, pd.DataFrame):

    raise TypeError(
        "test_working must be a pandas DataFrame. "
        "Please execute the previous sections first."
    )


# ============================================================
# STEP 2: DEFINE CATEGORICAL FEATURES FOR REVIEW
# ============================================================

# Separate variable for review only.
# Do not overwrite the final categorical_features variable
# that will be created later in Section 12.

categorical_features_review = [

    "Store",
    "StoreType",
    "Assortment",
    "StateHoliday",
    "PromoInterval",
    "Month_Position"

]


# ============================================================
# STEP 3: CHECK FEATURE AVAILABILITY
# ============================================================

print("\n" + "-" * 90)
print("CATEGORICAL FEATURE AVAILABILITY")
print("-" * 90)


missing_train_columns = [
    feature
    for feature in categorical_features_review
    if feature not in train_working.columns
]


missing_test_columns = [
    feature
    for feature in categorical_features_review
    if feature not in test_working.columns
]


for feature in categorical_features_review:

    train_available = (
        feature in train_working.columns
    )

    test_available = (
        feature in test_working.columns
    )

    print(
        f"{feature:<20} "
        f"Train: {train_available} | "
        f"Test: {test_available}"
    )


# Stop before accessing a missing column.

if missing_train_columns or missing_test_columns:

    raise KeyError(
        "Categorical feature availability check failed. "
        f"Missing in training: {missing_train_columns}; "
        f"Missing in test: {missing_test_columns}"
    )


# ============================================================
# STEP 4: CREATE CATEGORICAL FEATURE SUMMARY
# ============================================================

categorical_summary = []

unseen_test_categories = {}


print("\n" + "-" * 90)
print("CATEGORICAL FEATURE SUMMARY")
print("-" * 90)


for feature in categorical_features_review:

    train_series = train_working[feature]

    test_series = test_working[feature]


    # --------------------------------------------------------
    # A. Data Type
    # --------------------------------------------------------

    train_dtype = str(train_series.dtype)

    test_dtype = str(test_series.dtype)


    # --------------------------------------------------------
    # B. Missing Values
    # --------------------------------------------------------

    train_missing = int(
        train_series.isna().sum()
    )

    test_missing = int(
        test_series.isna().sum()
    )


    # --------------------------------------------------------
    # C. Unique Categories
    # --------------------------------------------------------

    train_unique = int(
        train_series.nunique(dropna=True)
    )

    test_unique = int(
        test_series.nunique(dropna=True)
    )


    # --------------------------------------------------------
    # D. Compare Categories
    # --------------------------------------------------------

    train_categories = set(
        train_series.dropna().astype(str).tolist()
    )

    test_categories = set(
        test_series.dropna().astype(str).tolist()
    )


    train_only = sorted(
        train_categories - test_categories,
        key=str
    )

    test_only = sorted(
        test_categories - train_categories,
        key=str
    )


    unseen_test_categories[feature] = test_only


    # --------------------------------------------------------
    # E. Store Summary Information
    # --------------------------------------------------------

    categorical_summary.append({

        "Feature": feature,

        "Training_Dtype": train_dtype,

        "Test_Dtype": test_dtype,

        "Training_Missing": train_missing,

        "Test_Missing": test_missing,

        "Training_Unique": train_unique,

        "Test_Unique": test_unique,

        "Train_Only_Categories": len(train_only),

        "Test_Only_Categories": len(test_only)

    })


    # --------------------------------------------------------
    # F. Display Feature Summary
    # --------------------------------------------------------

    print("\n" + "=" * 85)

    print(f"FEATURE: {feature}")

    print("=" * 85)


    print(
        f"\nTraining Data Type: {train_dtype}"
    )

    print(
        f"Test Data Type: {test_dtype}"
    )

    print(
        f"\nTraining Missing Values: {train_missing:,}"
    )

    print(
        f"Test Missing Values: {test_missing:,}"
    )

    print(
        f"\nTraining Unique Categories: {train_unique:,}"
    )

    print(
        f"Test Unique Categories: {test_unique:,}"
    )


    # Store is a high-cardinality identifier.
    # Display only a compact summary.

    if feature == "Store":

        print(
            "\nStore is treated as a categorical identifier."
        )

        print(
            f"Categories only in Training: "
            f"{len(train_only):,}"
        )

        print(
            f"Categories only in Test: "
            f"{len(test_only):,}"
        )

        print("\nTop 10 Training Stores:")

        print(
            train_series.value_counts(
                dropna=False
            ).head(10)
        )

        print("\nTop 10 Test Stores:")

        print(
            test_series.value_counts(
                dropna=False
            ).head(10)
        )


    else:

        print("\nTraining Categories:")

        print(
            sorted(train_categories, key=str)
        )

        print("\nTest Categories:")

        print(
            sorted(test_categories, key=str)
        )

        print("\nCategories Only in Training:")

        print(
            train_only if train_only else "None"
        )

        print("\nCategories Only in Test:")

        print(
            test_only if test_only else "None"
        )


# ============================================================
# STEP 5: CREATE FINAL SUMMARY TABLE
# ============================================================

categorical_summary_df = pd.DataFrame(
    categorical_summary
)


print("\n" + "-" * 90)
print("COMPLETE CATEGORICAL SUMMARY TABLE")
print("-" * 90)


display(
    categorical_summary_df
)


# ============================================================
# STEP 6: REVIEW CATEGORY FREQUENCY DISTRIBUTIONS
# ============================================================

print("\n" + "=" * 90)
print("CATEGORY FREQUENCY DISTRIBUTIONS")
print("=" * 90)


for feature in categorical_features_review:

    print("\n" + "-" * 90)

    print(f"FEATURE: {feature}")

    print("-" * 90)


    if feature == "Store":

        # Store contains many unique identifiers.
        # Display only the most frequent 10 stores.

        print("\nTraining Dataset - Top 10 Stores:")

        print(
            train_working[feature]
            .value_counts(dropna=False)
            .head(10)
        )

        print("\nTest Dataset - Top 10 Stores:")

        print(
            test_working[feature]
            .value_counts(dropna=False)
            .head(10)
        )


    else:

        print("\nTraining Dataset:")

        print(
            train_working[feature]
            .value_counts(dropna=False)
        )

        print("\nTest Dataset:")

        print(
            test_working[feature]
            .value_counts(dropna=False)
        )


# ============================================================
# STEP 7: VALIDATE STORE COVERAGE
# ============================================================

print("\n" + "-" * 90)
print("STORE COVERAGE VALIDATION")
print("-" * 90)


train_stores = set(
    train_working["Store"].dropna().unique()
)

test_stores = set(
    test_working["Store"].dropna().unique()
)


common_stores = (
    train_stores.intersection(test_stores)
)

test_only_stores = (
    test_stores - train_stores
)


store_coverage_valid = (
    len(test_only_stores) == 0
)


print(
    f"\nUnique Training Stores: "
    f"{len(train_stores):,}"
)

print(
    f"Unique Test Stores: "
    f"{len(test_stores):,}"
)

print(
    f"Common Stores: "
    f"{len(common_stores):,}"
)

print(
    f"Test-only Stores: "
    f"{len(test_only_stores):,}"
)

print(
    f"\nStore Coverage Valid: "
    f"{store_coverage_valid}"
)


# ============================================================
# STEP 8: REVIEW UNSEEN TEST CATEGORIES
# ============================================================

print("\n" + "-" * 90)
print("UNSEEN TEST CATEGORY REVIEW")
print("-" * 90)


for feature, unseen_categories in unseen_test_categories.items():

    print(
        f"\n{feature}: "
        f"{len(unseen_categories)} unseen test categories"
    )

    if unseen_categories:

        print(unseen_categories)

    else:

        print("None")


print(
    "\nUnseen test categories will be handled by "
    "OneHotEncoder(handle_unknown='ignore') "
    "during preprocessing."
)


# ============================================================
# STEP 9: REVIEW PROMO INTERVAL MISSING VALUES
# ============================================================

print("\n" + "-" * 90)
print("PROMO INTERVAL MISSINGNESS REVIEW")
print("-" * 90)


train_promo_interval_missing = int(
    train_working["PromoInterval"].isna().sum()
)

test_promo_interval_missing = int(
    test_working["PromoInterval"].isna().sum()
)


print(
    f"\nTraining PromoInterval Missing: "
    f"{train_promo_interval_missing:,}"
)

print(
    f"Test PromoInterval Missing: "
    f"{test_promo_interval_missing:,}"
)


print(
    "\nPromoInterval missingness is reviewed as part "
    "of the Promo2 promotion structure."
)

print(
    "Statistical categorical imputation will be performed "
    "later inside the preprocessing pipeline."
)


# ============================================================
# STEP 10: VALIDATE REQUIRED STORE AND CALENDAR CATEGORIES
# ============================================================

# These columns are expected to be complete in the current
# prepared dataset. PromoInterval is excluded from this check
# because its missingness is structural.

required_complete_categories = [

    "Store",
    "StoreType",
    "Assortment",
    "StateHoliday",
    "Month_Position"

]


required_category_values_valid = (

    train_working[
        required_complete_categories
    ].notna().all().all()

    and test_working[
        required_complete_categories
    ].notna().all().all()

)


print("\n" + "-" * 90)
print("REQUIRED CATEGORY VALUE VALIDATION")
print("-" * 90)


print(
    f"\nRequired Categories Have No Missing Values: "
    f"{required_category_values_valid}"
)


# ============================================================
# STEP 11: FINAL SECTION 8 VALIDATION
# ============================================================

categorical_features_available = (

    len(missing_train_columns) == 0

    and len(missing_test_columns) == 0

)


categorical_summary_complete = (

    len(categorical_summary_df)

    == len(categorical_features_review)

)


section_8_successful = all([

    categorical_features_available,

    categorical_summary_complete,

    store_coverage_valid,

    required_category_values_valid

])


print("\n" + "=" * 90)
print("SECTION 8 FINAL VALIDATION")
print("=" * 90)


print(
    f"\nAll Categorical Features Available: "
    f"{categorical_features_available}"
)

print(
    f"Categorical Summary Complete: "
    f"{categorical_summary_complete}"
)

print(
    f"Store Coverage Valid: "
    f"{store_coverage_valid}"
)

print(
    f"Required Category Values Valid: "
    f"{required_category_values_valid}"
)


print(
    f"\nSECTION 8 COMPLETED SUCCESSFULLY: "
    f"{section_8_successful}"
)


if section_8_successful:

    logger.info(
        "Section 8 categorical feature validation "
        "completed successfully."
    )

    print(
        "\nSECTION 8 COMPLETED SUCCESSFULLY."
    )

else:

    raise ValueError(
        "Section 8 validation failed. "
        "Please review the failed checks."
    )


print("=" * 90)

SECTION 8: CATEGORICAL FEATURE VALIDATION

------------------------------------------------------------------------------------------
CATEGORICAL FEATURE AVAILABILITY
------------------------------------------------------------------------------------------
Store                Train: True | Test: True
StoreType            Train: True | Test: True
Assortment           Train: True | Test: True
StateHoliday         Train: True | Test: True
PromoInterval        Train: True | Test: True
Month_Position       Train: True | Test: True

------------------------------------------------------------------------------------------
CATEGORICAL FEATURE SUMMARY
------------------------------------------------------------------------------------------

FEATURE: Store

Training Data Type: int64
Test Data Type: int64

Training Missing Values: 0
Test Missing Values: 0

Training Unique Categories: 1,115
Test Unique Categories: 856

Store is treated as a categorical identifier.
Categories only in Training: 

,Feature,Training_Dtype,Test_Dtype,Training_Missing,Test_Missing,Training_Unique,Test_Unique,Train_Only_Categories,Test_Only_Categories
0,Store,int64,int64,0,0,1115,856,259,0
1,StoreType,str,str,0,0,4,4,0,0
2,Assortment,str,str,0,0,3,3,0,0
3,StateHoliday,str,str,0,0,4,2,2,0
4,PromoInterval,str,str,508031,17232,3,3,0,0
5,Month_Position,str,str,0,0,3,3,0,0


2026-11-06 16:12:25,760 | INFO | Project6_Preprocessing | Section 8 categorical feature validation completed successfully.



CATEGORY FREQUENCY DISTRIBUTIONS

------------------------------------------------------------------------------------------
FEATURE: Store
------------------------------------------------------------------------------------------

Training Dataset - Top 10 Stores:
Store
1     942
2     942
3     942
4     942
5     942
6     942
7     942
8     942
9     942
10    942
Name: count, dtype: int64

Test Dataset - Top 10 Stores:
Store
1     48
3     48
7     48
8     48
9     48
10    48
11    48
12    48
13    48
14    48
Name: count, dtype: int64

------------------------------------------------------------------------------------------
FEATURE: StoreType
------------------------------------------------------------------------------------------

Training Dataset:
StoreType
a    551627
d    312912
c    136840
b     15830
Name: count, dtype: int64

Test Dataset:
StoreType
a    22128
d    14112
c     4272
b      576
Name: count, dtype: int64

-----------------------------------------------

# SECTION 9: MISSING VALUE HANDLING AND STRUCTURAL VALIDATION

## Purpose

This section validates the original and engineered features
before final feature selection.

The objective is to identify structural missingness,
validate missingness indicators and ensure that all
engineered features are logically consistent.

## 1. Calendar and Holiday Features

The calendar features created in Section 4 are reviewed.

The following features are validated:

- Year
- Month
- Day
- Date_DayOfWeek
- WeekOfYear
- Quarter
- IsWeekend
- Month_Position
- DaysSinceHoliday
- DaysUntilHoliday

Holiday proximity values may contain missing values
at the boundaries of the available calendar period.

## 2. Open Feature

The original Open variable is preserved.

The OpenMissing indicator identifies records where
the Open value is unavailable.

The original Open values are not imputed in this section.

## 3. Competition Features

Competition distance and opening information indicators
are validated against their original source values.

Future competition opening dates must have zero duration.

Negative competition durations are not permitted.

## 4. Promotion Features

Promotion indicators and duration features are validated.

Inactive Promo2 stores must have zero promotion duration.

Future promotion start dates must have zero duration.

Promo2NotStarted and Promo2InfoMissing must agree
with the actual promotion start information.

PromoIntervalMissing is checked as a structural indicator.
Missing or invalid intervals are not treated as equivalent
to inactive promotions.

## 5. Missing Value Summary

Missing values are summarized separately for training
and test datasets.

Remaining missing values are retained for the
preprocessing stage.

## 6. Leakage Prevention

No statistical imputation is performed here.

Numerical and categorical imputers, the StandardScaler
and OneHotEncoder will be fitted only on X_train
after chronological splitting.

## Expected Outcome

- Calendar and holiday features are validated.
- Open missingness is correctly represented.
- Competition indicators and durations are validated.
- Promotion indicators and durations are validated.
- Missing values are summarized.
- Both datasets are prepared for final feature selection.

In [377]:
# ============================================================
# SECTION 9: MISSING VALUE HANDLING AND STRUCTURAL VALIDATION
# ============================================================

print("=" * 90)
print("SECTION 9: MISSING VALUE HANDLING AND STRUCTURAL VALIDATION")
print("=" * 90)


# ============================================================
# STEP 1: VALIDATE INPUT DATASETS
# ============================================================

if not isinstance(train_working, pd.DataFrame):

    raise TypeError(
        "train_working must be a pandas DataFrame."
    )

if not isinstance(test_working, pd.DataFrame):

    raise TypeError(
        "test_working must be a pandas DataFrame."
    )


# Preserve original index and row counts for validation.

train_original_index = train_df.index.copy()
test_original_index = test_df.index.copy()

train_original_rows = len(train_df)
test_original_rows = len(test_df)


# ============================================================
# STEP 2: CHECK REQUIRED SOURCE AND ENGINEERED FEATURES
# ============================================================

required_features = [

    # Original features
    "Store",
    "Date",
    "Open",
    "StateHoliday",
    "SchoolHoliday",

    # Section 4: Calendar and holiday features
    "Year",
    "Month",
    "Day",
    "Date_DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "IsWeekend",
    "Month_Position",
    "DaysSinceHoliday",
    "DaysUntilHoliday",

    # Section 5: Competition features
    "CompetitionDistance",
    "CompetitionDistanceMissing",
    "CompetitionOpenDate",
    "CompetitionOpenInfoMissing",
    "CompetitionOpenDurationMonths",

    # Section 6: Promotion features
    "Promo2",
    "Promo2StartDate",
    "Promo2InfoMissing",
    "PromoIntervalMissing",
    "Promo2DurationMonths",
    "Promo2NotStarted",
    "Promo2IntervalMonth",
    "PromoInterval"

]


print("\n" + "-" * 90)
print("REQUIRED FEATURE AVAILABILITY")
print("-" * 90)


missing_train_features = [
    col
    for col in required_features
    if col not in train_working.columns
]

missing_test_features = [
    col
    for col in required_features
    if col not in test_working.columns
]


print(
    f"\nMissing in Training: {missing_train_features}"
)

print(
    f"Missing in Test: {missing_test_features}"
)


if missing_train_features or missing_test_features:

    raise KeyError(
        "Required features are missing. "
        f"Training: {missing_train_features}; "
        f"Test: {missing_test_features}. "
        "Please check the earlier feature-engineering sections."
    )


# ============================================================
# STEP 3: VALIDATE ROW AND INDEX PRESERVATION
# ============================================================

print("\n" + "-" * 90)
print("ROW AND INDEX PRESERVATION")
print("-" * 90)


train_rows_preserved = (
    len(train_working) == train_original_rows
)

test_rows_preserved = (
    len(test_working) == test_original_rows
)

train_index_preserved = (
    train_working.index.equals(train_original_index)
)

test_index_preserved = (
    test_working.index.equals(test_original_index)
)


print(
    f"\nTraining Rows Preserved: {train_rows_preserved}"
)

print(
    f"Test Rows Preserved: {test_rows_preserved}"
)

print(
    f"Training Index Preserved: {train_index_preserved}"
)

print(
    f"Test Index Preserved: {test_index_preserved}"
)


if not all([
    train_rows_preserved,
    test_rows_preserved,
    train_index_preserved,
    test_index_preserved
]):

    raise ValueError(
        "Training or test row/index alignment changed "
        "during feature engineering."
    )


# ============================================================
# STEP 4: CREATE AND VALIDATE OPEN MISSINGNESS INDICATOR
# ============================================================

print("\n" + "-" * 90)
print("OPEN MISSINGNESS VALIDATION")
print("-" * 90)


# Recreate deterministically to ensure the indicator
# exactly represents the original Open missingness.

train_working["OpenMissing"] = (
    train_working["Open"]
    .isna()
    .astype("int8")
)

test_working["OpenMissing"] = (
    test_working["Open"]
    .isna()
    .astype("int8")
)


def validate_open_feature(frame, dataset_name):

    original_open = frame["Open"]

    numeric_open = pd.to_numeric(
        original_open,
        errors="coerce"
    )


    # Non-missing Open values must be numeric.

    invalid_open_values = (

        original_open.notna()

        & numeric_open.isna()

    )


    open_values_valid = (

        not invalid_open_values.any()

        and numeric_open.dropna().isin([0, 1]).all()

    )


    expected_indicator = (
        original_open.isna().astype("int8")
    )


    actual_indicator = (
        frame["OpenMissing"]
    )


    indicator_valid = np.array_equal(

        actual_indicator.to_numpy(dtype="int8"),

        expected_indicator.to_numpy(dtype="int8")

    )


    print(f"\n{dataset_name}")

    print(
        f"Missing Open Values: "
        f"{original_open.isna().sum():,}"
    )

    print(
        f"Open Values Valid: {open_values_valid}"
    )

    print(
        f"OpenMissing Indicator Correct: {indicator_valid}"
    )


    return (
        open_values_valid
        and indicator_valid
    )


train_open_missing_correct = validate_open_feature(
    train_working,
    "Training Dataset"
)

test_open_missing_correct = validate_open_feature(
    test_working,
    "Test Dataset"
)


open_missingness_valid = (

    train_open_missing_correct

    and test_open_missing_correct

)


# ============================================================
# STEP 5: VALIDATE CALENDAR FEATURES
# ============================================================

print("\n" + "-" * 90)
print("CALENDAR FEATURE VALIDATION")
print("-" * 90)


calendar_features = [

    "Year",
    "Month",
    "Day",
    "Date_DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "IsWeekend",
    "Month_Position"

]


def validate_calendar_features(frame, dataset_name):

    dates = pd.to_datetime(
        frame["Date"],
        errors="coerce"
    )


    if dates.isna().any():

        print(
            f"{dataset_name}: Invalid dates found."
        )

        return False


    # Recalculate calendar values directly from Date.

    expected_values = {

        "Year": dates.dt.year.to_numpy(),

        "Month": dates.dt.month.to_numpy(),

        "Day": dates.dt.day.to_numpy(),

        "Date_DayOfWeek": dates.dt.dayofweek.to_numpy(),

        "WeekOfYear": (
            dates.dt.isocalendar().week
            .to_numpy(dtype="int64")
        ),

        "Quarter": dates.dt.quarter.to_numpy(),

        "IsWeekend": (
            dates.dt.dayofweek.isin([5, 6])
            .astype("int8")
            .to_numpy()
        )

    }


    checks = {}


    # Validate numeric calendar features.

    for feature, expected in expected_values.items():

        actual = pd.to_numeric(
            frame[feature],
            errors="coerce"
        )


        valid = (

            actual.notna().all()

            and np.array_equal(

                actual.to_numpy(),

                expected

            )

        )


        checks[feature] = bool(valid)


    # Validate month position.

    day_values = dates.dt.day


    expected_month_position = np.where(

        day_values <= 10,

        "Beginning",

        np.where(
            day_values <= 20,
            "Mid",
            "End"
        )

    )


    month_position_valid = np.array_equal(

        frame["Month_Position"].astype(str).to_numpy(),

        expected_month_position.astype(str)

    )


    checks["Month_Position"] = bool(
        month_position_valid
    )


    print(f"\n{dataset_name}")


    for feature, result in checks.items():

        print(
            f"{feature:<25}: {result}"
        )


    return all(checks.values())


train_calendar_valid = validate_calendar_features(
    train_working,
    "Training Dataset"
)

test_calendar_valid = validate_calendar_features(
    test_working,
    "Test Dataset"
)


calendar_features_valid = (

    train_calendar_valid

    and test_calendar_valid

)


# ============================================================
# STEP 6: VALIDATE HOLIDAY PROXIMITY FEATURES
# ============================================================

print("\n" + "-" * 90)
print("HOLIDAY PROXIMITY VALIDATION")
print("-" * 90)


def validate_holiday_features(frame, dataset_name):

    state_values = (

        frame["StateHoliday"]

        .astype("string")

        .str.strip()

        .str.lower()

    )


    is_state_holiday = ~state_values.isin([
        "0",
        "0.0",
        "none",
        ""
    ])


    school_values = pd.to_numeric(

        frame["SchoolHoliday"],

        errors="coerce"

    )


    if school_values.isna().any():

        return False


    holiday_event = (

        is_state_holiday

        | school_values.eq(1)

    )


    checks = []


    for feature in [
        "DaysSinceHoliday",
        "DaysUntilHoliday"
    ]:

        original_values = frame[feature]

        values = pd.to_numeric(
            original_values,
            errors="coerce"
        )


        # Missing boundary values are permitted.
        # Unexpected text values are not permitted.

        invalid_text = (

            original_values.notna()

            & values.isna()

        )


        nonnegative = (

            not invalid_text.any()

            and values.dropna().ge(0).all()

        )


        finite = np.isfinite(

            values.dropna().to_numpy(dtype=float)

        ).all()


        same_day_zero = (

            values.loc[holiday_event]

            .eq(0)

            .all()

        )


        feature_valid = all([

            nonnegative,

            finite,

            same_day_zero

        ])


        checks.append(feature_valid)


        print(f"\n{dataset_name}: {feature}")

        print(
            f"Missing Values: {values.isna().sum():,}"
        )

        print(
            f"Non-negative: {nonnegative}"
        )

        print(
            f"Finite Values: {finite}"
        )

        print(
            f"Same-day Holiday Equals Zero: {same_day_zero}"
        )


    return all(checks)


train_holiday_valid = validate_holiday_features(
    train_working,
    "Training Dataset"
)

test_holiday_valid = validate_holiday_features(
    test_working,
    "Test Dataset"
)


holiday_features_valid = (

    train_holiday_valid

    and test_holiday_valid

)


# ============================================================
# STEP 7: VALIDATE COMPETITION FEATURES
# ============================================================

print("\n" + "-" * 90)
print("COMPETITION STRUCTURAL VALIDATION")
print("-" * 90)


def validate_competition_structure(frame, dataset_name):

    dates = pd.to_datetime(
        frame["Date"],
        errors="coerce"
    )

    opening_date = pd.to_datetime(

        frame["CompetitionOpenDate"],

        errors="coerce"

    )


    distance = pd.to_numeric(

        frame["CompetitionDistance"],

        errors="coerce"

    )


    duration = pd.to_numeric(

        frame["CompetitionOpenDurationMonths"],

        errors="coerce"

    )


    # --------------------------------------------------------
    # A. Check source values.
    # --------------------------------------------------------

    invalid_distance = (

        frame["CompetitionDistance"].notna()

        & distance.isna()

    )


    invalid_dates = dates.isna().any()


    # --------------------------------------------------------
    # B. Validate distance missingness indicator.
    # --------------------------------------------------------

    expected_distance_missing = (
        distance.isna().astype("int8")
    )


    distance_indicator_valid = np.array_equal(

        frame["CompetitionDistanceMissing"]
        .to_numpy(dtype="int8"),

        expected_distance_missing.to_numpy(dtype="int8")

    )


    # --------------------------------------------------------
    # C. Validate opening information indicator.
    # --------------------------------------------------------

    expected_opening_missing = (
        opening_date.isna().astype("int8")
    )


    opening_indicator_valid = np.array_equal(

        frame["CompetitionOpenInfoMissing"]
        .to_numpy(dtype="int8"),

        expected_opening_missing.to_numpy(dtype="int8")

    )


    # --------------------------------------------------------
    # D. Validate future competition opening.
    # --------------------------------------------------------

    future_opening = (

        opening_date.notna()

        & (opening_date > dates)

    )


    future_duration_valid = (

        duration.loc[future_opening]

        .eq(0)

        .all()

    )


    # --------------------------------------------------------
    # E. Validate duration.
    # --------------------------------------------------------

    no_negative_duration = (

        duration.dropna()

        .ge(0)

        .all()

    )


    checks = {

        "Source Values Valid":

            not invalid_distance.any() and not invalid_dates,

        "Distance Indicator Correct":

            distance_indicator_valid,

        "Opening Indicator Correct":

            opening_indicator_valid,

        "Future Opening Duration Zero":

            future_duration_valid,

        "No Negative Duration":

            no_negative_duration

    }


    print(f"\n{dataset_name}")


    for name, result in checks.items():

        print(
            f"{name}: {result}"
        )


    return all(checks.values())


train_competition_valid = validate_competition_structure(
    train_working,
    "Training Dataset"
)

test_competition_valid = validate_competition_structure(
    test_working,
    "Test Dataset"
)


competition_structure_valid = (

    train_competition_valid

    and test_competition_valid

)


# ============================================================
# STEP 8: VALIDATE PROMOTION FEATURES
# ============================================================

print("\n" + "-" * 90)
print("PROMOTION STRUCTURAL VALIDATION")
print("-" * 90)


def validate_promotion_structure(frame, dataset_name):

    promo2 = pd.to_numeric(
        frame["Promo2"],
        errors="coerce"
    )


    dates = pd.to_datetime(
        frame["Date"],
        errors="coerce"
    )


    promo_start = pd.to_datetime(

        frame["Promo2StartDate"],

        errors="coerce"

    )


    duration = pd.to_numeric(

        frame["Promo2DurationMonths"],

        errors="coerce"

    )


    # --------------------------------------------------------
    # A. Validate Promo2 source values.
    # --------------------------------------------------------

    promo2_source_valid = (

        promo2.notna().all()

        and promo2.isin([0, 1]).all()

        and dates.notna().all()

    )


    inactive = promo2.eq(0)

    active = promo2.eq(1)


    # --------------------------------------------------------
    # B. Inactive Promo2 stores must have zero duration.
    # --------------------------------------------------------

    inactive_duration_zero = (

        duration.loc[inactive]

        .eq(0)

        .all()

    )


    # --------------------------------------------------------
    # C. Future promotion start dates must have zero duration.
    # --------------------------------------------------------

    future_promo = (

        active

        & promo_start.notna()

        & (promo_start > dates)

    )


    future_duration_zero = (

        duration.loc[future_promo]

        .eq(0)

        .all()

    )


    # --------------------------------------------------------
    # D. No negative duration.
    # --------------------------------------------------------

    no_negative_duration = (

        duration.dropna()

        .ge(0)

        .all()

    )


    # --------------------------------------------------------
    # E. Validate Promo2NotStarted.
    # --------------------------------------------------------

    expected_not_started = (
        future_promo.astype("int8")
    )


    actual_not_started = pd.to_numeric(

        frame["Promo2NotStarted"],

        errors="coerce"

    )


    not_started_valid = (

        actual_not_started.notna().all()

        and np.array_equal(

            actual_not_started.to_numpy(dtype="int8"),

            expected_not_started.to_numpy(dtype="int8")

        )

    )


    # --------------------------------------------------------
    # F. Validate Promo2InfoMissing.
    # --------------------------------------------------------

    expected_start_missing = (

        active

        & promo_start.isna()

    ).astype("int8")


    actual_start_missing = pd.to_numeric(

        frame["Promo2InfoMissing"],

        errors="coerce"

    )


    start_missing_valid = (

        actual_start_missing.notna().all()

        and np.array_equal(

            actual_start_missing.to_numpy(dtype="int8"),

            expected_start_missing.to_numpy(dtype="int8")

        )

    )


    # --------------------------------------------------------
    # G. Validate PromoIntervalMissing.
    # --------------------------------------------------------

    actual_interval_missing = pd.to_numeric(

        frame["PromoIntervalMissing"],

        errors="coerce"

    )


    # Section 6 sets this indicator to zero for inactive stores.
    # For active stores, one can represent a missing OR invalid
    # promotion interval. Therefore, do not compare it only
    # with raw PromoInterval.isna().

    interval_indicator_binary = (

        actual_interval_missing.notna().all()

        and actual_interval_missing.isin([0, 1]).all()

    )


    inactive_interval_zero = (

        actual_interval_missing.loc[inactive]

        .eq(0)

        .all()

    )


    # If an active store has a raw missing interval,
    # the indicator must be 1.

    raw_missing_active_interval = (

        active

        & frame["PromoInterval"].isna()

    )


    raw_missing_interval_correct = (

        actual_interval_missing.loc[
            raw_missing_active_interval
        ]

        .eq(1)

        .all()

    )


    interval_missing_valid = all([

        interval_indicator_binary,

        inactive_interval_zero,

        raw_missing_interval_correct

    ])


    # --------------------------------------------------------
    # H. Validate Promo2IntervalMonth.
    # --------------------------------------------------------

    interval_month = pd.to_numeric(

        frame["Promo2IntervalMonth"],

        errors="coerce"

    )


    # Missing is allowed for active stores with unavailable
    # promotion interval information.

    interval_month_values_valid = (

        interval_month.dropna()

        .isin([0, 1])

        .all()

    )


    # Inactive stores must have zero.

    inactive_interval_month_zero = (

        interval_month.loc[inactive]

        .eq(0)

        .all()

    )


    # Future promotion starts must have zero.

    future_interval_month_zero = (

        interval_month.loc[future_promo]

        .eq(0)

        .all()

    )


    interval_month_valid = all([

        interval_month_values_valid,

        inactive_interval_month_zero,

        future_interval_month_zero

    ])


    # --------------------------------------------------------
    # I. Display validation results.
    # --------------------------------------------------------

    checks = {

        "Promo2 Source Values Valid":

            promo2_source_valid,

        "Inactive Promo2 Duration Zero":

            inactive_duration_zero,

        "Future Promo2 Duration Zero":

            future_duration_zero,

        "No Negative Promo2 Duration":

            no_negative_duration,

        "Promo2NotStarted Indicator Correct":

            not_started_valid,

        "Promo2InfoMissing Indicator Correct":

            start_missing_valid,

        "PromoIntervalMissing Indicator Valid":

            interval_missing_valid,

        "Promo2IntervalMonth Valid":

            interval_month_valid

    }


    print(f"\n{dataset_name}")


    for name, result in checks.items():

        print(
            f"{name}: {result}"
        )


    return all(checks.values())


train_promo_valid = validate_promotion_structure(
    train_working,
    "Training Dataset"
)

test_promo_valid = validate_promotion_structure(
    test_working,
    "Test Dataset"
)


promotion_structure_valid = (

    train_promo_valid

    and test_promo_valid

)


# ============================================================
# STEP 9: MISSING VALUE SUMMARY
# ============================================================

print("\n" + "-" * 90)
print("MISSING VALUE SUMMARY BEFORE IMPUTATION")
print("-" * 90)


missing_review_columns = [

    # Original features
    "Open",
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",

    # Engineered helper and duration features
    "CompetitionOpenDate",
    "CompetitionOpenDurationMonths",
    "Promo2StartDate",
    "Promo2DurationMonths",

    # Holiday features
    "DaysSinceHoliday",
    "DaysUntilHoliday"

]


train_review_columns = [

    col

    for col in missing_review_columns

    if col in train_working.columns

]


test_review_columns = [

    col

    for col in missing_review_columns

    if col in test_working.columns

]


train_missing_summary = (

    train_working[train_review_columns]

    .isna()

    .sum()

    .sort_values(ascending=False)

)


test_missing_summary = (

    test_working[test_review_columns]

    .isna()

    .sum()

    .sort_values(ascending=False)

)


print("\nTraining Dataset:")

print(
    train_missing_summary
)


print("\nTest Dataset:")

print(
    test_missing_summary
)


# ============================================================
# STEP 10: MISSINGNESS INDICATOR SUMMARY
# ============================================================

print("\n" + "-" * 90)
print("MISSINGNESS INDICATOR DISTRIBUTION")
print("-" * 90)


indicator_features = [

    "OpenMissing",
    "CompetitionDistanceMissing",
    "CompetitionOpenInfoMissing",
    "Promo2InfoMissing",
    "PromoIntervalMissing"

]


for dataset_name, frame in [

    ("Training Dataset", train_working),

    ("Test Dataset", test_working)

]:

    print(f"\n{dataset_name}")


    for feature in indicator_features:

        print(f"\n{feature}")

        print(
            frame[feature].value_counts(dropna=False)
        )


# ============================================================
# STEP 11: CONFIRM THAT STATISTICAL IMPUTATION IS DEFERRED
# ============================================================

print("\n" + "-" * 90)
print("STATISTICAL IMPUTATION STATUS")
print("-" * 90)


print("""
No statistical imputation has been performed in Section 9.

Original and engineered missing values have been preserved.

Numerical missing values will be handled using median
imputation inside the preprocessing pipeline.

Categorical missing values will be handled using
most-frequent imputation.

The imputers, StandardScaler and OneHotEncoder will be
fitted using X_train only after chronological splitting.

The fitted transformations will then be applied consistently
to validation and test datasets.
""")


# ============================================================
# STEP 12: FINAL SECTION 9 VALIDATION
# ============================================================

section_9_checks = {

    "Training and test rows preserved":

        train_rows_preserved and test_rows_preserved,

    "Training and test indexes preserved":

        train_index_preserved and test_index_preserved,

    "Open missingness valid":

        open_missingness_valid,

    "Calendar features valid":

        calendar_features_valid,

    "Holiday proximity features valid":

        holiday_features_valid,

    "Competition structure valid":

        competition_structure_valid,

    "Promotion structure valid":

        promotion_structure_valid

}


print("\n" + "=" * 90)
print("SECTION 9 FINAL VALIDATION")
print("=" * 90)


for check_name, result in section_9_checks.items():

    print(
        f"{check_name:<45}: {result}"
    )


section9_successful = all(
    section_9_checks.values()
)


# Keep both naming styles available for downstream sections.
section_9_successful = section9_successful


print("\n" + "=" * 90)

print(
    f"SECTION 9 COMPLETED SUCCESSFULLY: "
    f"{section9_successful}"
)

print("=" * 90)


# ============================================================
# STEP 13: LOG FINAL STATUS
# ============================================================

if section9_successful:

    logger.info(
        "Section 9 missing value handling and structural "
        "validation completed successfully."
    )

    print(
        "\nSection 9 is ready for final feature selection."
    )

else:

    failed_checks = [

        name

        for name, result in section_9_checks.items()

        if not result

    ]


    logger.error(
        "Section 9 validation failed: %s",
        failed_checks
    )


    raise ValueError(
        "Section 9 validation failed. "
        f"Review failed checks: {failed_checks}"
    )


print("\nSECTION 9 COMPLETED SUCCESSFULLY.")

print("=" * 90)

SECTION 9: MISSING VALUE HANDLING AND STRUCTURAL VALIDATION

------------------------------------------------------------------------------------------
REQUIRED FEATURE AVAILABILITY
------------------------------------------------------------------------------------------

Missing in Training: []
Missing in Test: []

------------------------------------------------------------------------------------------
ROW AND INDEX PRESERVATION
------------------------------------------------------------------------------------------

Training Rows Preserved: True
Test Rows Preserved: True
Training Index Preserved: True
Test Index Preserved: True

------------------------------------------------------------------------------------------
OPEN MISSINGNESS VALIDATION
------------------------------------------------------------------------------------------

Training Dataset
Missing Open Values: 0
Open Values Valid: True
OpenMissing Indicator Correct: True

Test Dataset
Missing Open Values: 11
Open Va

2026-11-06 16:12:26,590 | INFO | Project6_Preprocessing | Section 9 missing value handling and structural validation completed successfully.



Training Dataset:
Promo2SinceWeek                  508031
PromoInterval                    508031
Promo2StartDate                  508031
Promo2SinceYear                  508031
CompetitionOpenSinceYear         323348
CompetitionOpenDurationMonths    323348
CompetitionOpenDate              323348
CompetitionOpenSinceMonth        323348
CompetitionDistance                2642
DaysUntilHoliday                   1800
Open                                  0
Promo2DurationMonths                  0
DaysSinceHoliday                      0
dtype: int64

Test Dataset:
Promo2SinceWeek                  17232
PromoInterval                    17232
Promo2StartDate                  17232
Promo2SinceYear                  17232
DaysUntilHoliday                 16782
CompetitionOpenDurationMonths    15216
CompetitionOpenDate              15216
CompetitionOpenSinceMonth        15216
CompetitionOpenSinceYear         15216
CompetitionDistance                 96
Open                                11
Prom

# SECTION 10: FINAL FEATURE SELECTION AND FEATURE ORGANIZATION

## Purpose

This section prepares the final candidate feature datasets for the
Rossmann Store Sales forecasting workflow.

It separates the Sales target from the predictor variables, preserves
the original test identifiers, and ensures that training and test
datasets contain the same model input features.

## Feature Selection Strategy

The following rules are applied:

1. Sales is retained separately as the training target.
2. Customers is excluded because it is unavailable in the test dataset
   and cannot be used for future predictions.
3. Id is preserved separately for the final prediction file.
4. Date is excluded after useful calendar information has been
   extracted into engineered features.
5. Competition and Promo2 helper dates and their raw start-period
   columns are excluded after their information is represented by
   engineered duration and missingness features.
6. Calendar, holiday, competition, promotion, store and daily operating
   features are retained as model candidates.
7. Store is treated as a categorical identifier rather than a
   continuous numerical measurement.
8. Missing values are preserved at this stage. Statistical imputation
   and scaling will be performed later through a leakage-aware
   preprocessing pipeline fitted only on the training split.

## Important Methodology

Training and test features must have identical column names and order.

The validation period will be selected chronologically in Section 11.
No statistical imputation, scaling or encoding is performed in this
section.

## Expected Outcome

- X_candidate: Training predictor variables.
- X_test_candidate: Matching test predictor variables.
- y: Training Sales target.
- test_ids: Original test identifiers.
- section_10_successful: Final feature-selection validation status.

These outputs will be used to create the chronological
training-validation split in Section 11.

In [378]:
# ============================================================
# SECTION 10: FINAL FEATURE SELECTION AND FEATURE ORGANIZATION
# ============================================================

print("=" * 90)
print("SECTION 10: FINAL FEATURE SELECTION AND FEATURE ORGANIZATION")
print("=" * 90)


# ============================================================
# STEP 1: CONFIRM SECTION 9 AND INPUT DATASETS ARE READY
# ============================================================

if "section9_successful" not in globals():

    raise RuntimeError(
        "Section 9 completion flag was not found. "
        "Run Sections 1-9 in order first."
    )


if not bool(section9_successful):

    raise RuntimeError(
        "Section 9 did not pass validation. "
        "Correct Section 9 before running Section 10."
    )


if not isinstance(train_working, pd.DataFrame):

    raise TypeError(
        "train_working must be a pandas DataFrame."
    )


if not isinstance(test_working, pd.DataFrame):

    raise TypeError(
        "test_working must be a pandas DataFrame."
    )


# ============================================================
# STEP 2: CHECK REQUIRED SOURCE COLUMNS
# ============================================================

required_train_source_columns = [
    "Store",
    "Date",
    "Sales"
]

required_test_source_columns = [
    "Store",
    "Date",
    "Id"
]


missing_train_source_columns = [
    column
    for column in required_train_source_columns
    if column not in train_working.columns
]


missing_test_source_columns = [
    column
    for column in required_test_source_columns
    if column not in test_working.columns
]


if missing_train_source_columns or missing_test_source_columns:

    raise KeyError(
        "Required source columns are missing. "
        f"Training: {missing_train_source_columns}; "
        f"Test: {missing_test_source_columns}."
    )


# ============================================================
# STEP 3: VALIDATE AND PRESERVE THE SALES TARGET
# ============================================================

# Customers is intentionally NOT used as a predictor because
# it is unavailable in the test dataset.

target_column = "Sales"


y = pd.to_numeric(
    train_working[target_column],
    errors="coerce"
).copy()


y.name = target_column


target_valid = (

    y.notna().all()

    and np.isfinite(
        y.to_numpy(dtype=float)
    ).all()

    and y.ge(0).all()

    and y.index.equals(train_working.index)

)


print("\n" + "-" * 90)
print("SALES TARGET VALIDATION")
print("-" * 90)


print(f"Target Name: {y.name}")

print(f"Target Rows: {len(y):,}")

print(f"Missing Target Values: {y.isna().sum():,}")

print(f"Minimum Sales: {y.min():,.2f}")

print(f"Maximum Sales: {y.max():,.2f}")

print(f"Target Valid: {target_valid}")


if not target_valid:

    raise ValueError(
        "Sales target contains missing, non-numeric, infinite, "
        "negative, or misaligned values."
    )


# ============================================================
# STEP 4: PRESERVE ORIGINAL TEST IDENTIFIERS
# ============================================================

# Id is saved separately for the final prediction file.
# It must not be included in model input features.

test_ids = test_working["Id"].copy()


test_ids_valid = (

    test_ids.notna().all()

    and test_ids.is_unique

    and test_ids.index.equals(test_working.index)

)


print("\n" + "-" * 90)
print("ORIGINAL TEST IDENTIFIER VALIDATION")
print("-" * 90)


print(f"Test ID Count: {len(test_ids):,}")

print(f"Missing Test IDs: {test_ids.isna().sum():,}")

print(f"Unique Test IDs: {test_ids.is_unique}")

print(
    f"Test ID Index Aligned: "
    f"{test_ids.index.equals(test_working.index)}"
)


if not test_ids_valid:

    raise ValueError(
        "Test Id values must be complete, unique, "
        "and aligned with test rows."
    )


# ============================================================
# STEP 5: DEFINE REQUIRED MODEL FEATURES
# ============================================================

# These features must be available after Sections 4-9.

required_model_features = [

    # Store and daily operating information
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "StateHoliday",
    "SchoolHoliday",
    "StoreType",
    "Assortment",

    # Calendar features from Section 4
    "Year",
    "Month",
    "Day",
    "Date_DayOfWeek",
    "WeekOfYear",
    "Quarter",
    "IsWeekend",
    "Month_Position",

    # Holiday proximity features from Section 4
    "DaysSinceHoliday",
    "DaysUntilHoliday",

    # Competition features from Section 5
    "CompetitionDistance",
    "CompetitionDistanceMissing",
    "CompetitionOpenInfoMissing",
    "CompetitionOpenDurationMonths",

    # Promotion features from Section 6
    "Promo2",
    "Promo2InfoMissing",
    "PromoIntervalMissing",
    "Promo2DurationMonths",
    "Promo2NotStarted",
    "Promo2IntervalMonth",

    # Missingness feature from Section 9
    "OpenMissing"
]


missing_model_features_train = [
    column
    for column in required_model_features
    if column not in train_working.columns
]


missing_model_features_test = [
    column
    for column in required_model_features
    if column not in test_working.columns
]


print("\n" + "-" * 90)
print("REQUIRED MODEL FEATURE AVAILABILITY")
print("-" * 90)


print(
    f"Required Features: {len(required_model_features)}"
)

print(
    f"Missing in Training: "
    f"{missing_model_features_train or 'None'}"
)

print(
    f"Missing in Test: "
    f"{missing_model_features_test or 'None'}"
)


if missing_model_features_train or missing_model_features_test:

    raise KeyError(
        "Required model features are missing. "
        f"Training: {missing_model_features_train}; "
        f"Test: {missing_model_features_test}. "
        "Check Sections 4-9."
    )


# ============================================================
# STEP 6: DEFINE COLUMNS TO EXCLUDE FROM MODEL INPUT
# ============================================================

# Sales:
# Training target, not a predictor.
#
# Customers:
# Not available in the test dataset and cannot be used
# for future forecasting.
#
# Id:
# Original test identifier, stored separately.
#
# Date:
# Useful date information has already been converted
# into calendar features in Section 4.
#
# CompetitionOpenDate and Promo2StartDate:
# Helper datetime columns used to calculate duration features.
#
# Raw competition and Promo2 start-period columns:
# Their useful information is represented by engineered
# duration and missingness features.
#
# PromoInterval:
# Its date-specific promotion information is represented
# by Promo2IntervalMonth.

columns_to_exclude = [

    "Sales",
    "Customers",
    "Id",
    "Date",

    "CompetitionOpenDate",
    "Promo2StartDate",

    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",

    "Promo2SinceWeek",
    "Promo2SinceYear",

    "PromoInterval"

]


print("\n" + "-" * 90)
print("COLUMNS EXCLUDED FROM MODEL INPUT")
print("-" * 90)


for column in columns_to_exclude:

    if (
        column in train_working.columns
        or column in test_working.columns
    ):

        print(f"- {column}")


# ============================================================
# STEP 7: BUILD TRAINING AND TEST CANDIDATE FEATURES
# ============================================================

train_candidate_columns = [

    column

    for column in train_working.columns

    if column not in columns_to_exclude

]


test_candidate_columns = [

    column

    for column in test_working.columns

    if column not in columns_to_exclude

]


# Check for unexpected feature differences.

missing_in_test = [

    column

    for column in train_candidate_columns

    if column not in test_candidate_columns

]


extra_in_test = [

    column

    for column in test_candidate_columns

    if column not in train_candidate_columns

]


print("\n" + "-" * 90)
print("TRAINING AND TEST FEATURE ALIGNMENT")
print("-" * 90)


print(
    f"Features Missing in Test: "
    f"{missing_in_test or 'None'}"
)


print(
    f"Extra Features in Test: "
    f"{extra_in_test or 'None'}"
)


if missing_in_test or extra_in_test:

    raise ValueError(
        "Training and test candidate features do not match. "
        f"Missing in test: {missing_in_test}; "
        f"Extra in test: {extra_in_test}."
    )


# Use training feature order as the reference.

X_candidate = train_working.loc[
    :,
    train_candidate_columns
].copy()


X_test_candidate = test_working.loc[
    :,
    train_candidate_columns
].copy()


candidate_features_aligned = (

    X_candidate.columns.equals(
        X_test_candidate.columns
    )

    and X_candidate.columns.is_unique

    and X_test_candidate.columns.is_unique

)


if not candidate_features_aligned:

    raise ValueError(
        "Training and test feature columns are not aligned."
    )


# ============================================================
# STEP 8: VALIDATE CATEGORICAL AND NUMERICAL FEATURE TYPES
# ============================================================

# Store is a numeric identifier, not a continuous measurement.
# It will be treated as categorical during preprocessing.

categorical_features_review = [

    "Store",
    "StoreType",
    "Assortment",
    "StateHoliday",
    "Month_Position"

]


# If the prepared dataset includes a numeric State code,
# treat it as a category rather than a continuous quantity.

if "State" in X_candidate.columns:

    categorical_features_review.append("State")


missing_categorical_features = [

    column

    for column in categorical_features_review

    if column not in X_candidate.columns

]


if missing_categorical_features:

    raise KeyError(
        "Expected categorical features are missing: "
        f"{missing_categorical_features}."
    )


# All remaining features are expected to be numerical.

numerical_features_review = [

    column

    for column in X_candidate.columns

    if column not in categorical_features_review

]


# Convert numerical columns safely.
# Invalid non-empty text values are not silently accepted.

invalid_numeric_columns = []


for column in numerical_features_review:

    train_original = X_candidate[column]

    test_original = X_test_candidate[column]


    train_numeric = pd.to_numeric(
        train_original,
        errors="coerce"
    )

    test_numeric = pd.to_numeric(
        test_original,
        errors="coerce"
    )


    train_invalid_text = (
        train_original.notna()
        & train_numeric.isna()
    )

    test_invalid_text = (
        test_original.notna()
        & test_numeric.isna()
    )


    if (
        train_invalid_text.any()
        or test_invalid_text.any()
    ):

        invalid_numeric_columns.append(column)

        continue


    X_candidate[column] = train_numeric

    X_test_candidate[column] = test_numeric


if invalid_numeric_columns:

    raise TypeError(
        "Non-numeric values were found in numerical features: "
        f"{invalid_numeric_columns}. "
        "Review the source data before preprocessing."
    )


# Check infinite values in numerical features.

infinite_numeric_columns = []


for column in numerical_features_review:

    train_values = (
        X_candidate[column]
        .dropna()
        .to_numpy(dtype=float)
    )

    test_values = (
        X_test_candidate[column]
        .dropna()
        .to_numpy(dtype=float)
    )


    if (
        not np.isfinite(train_values).all()
        or not np.isfinite(test_values).all()
    ):

        infinite_numeric_columns.append(column)


if infinite_numeric_columns:

    raise ValueError(
        "Infinite values were found in numerical features: "
        f"{infinite_numeric_columns}."
    )


print("\n" + "-" * 90)
print("FEATURE TYPE REVIEW")
print("-" * 90)


print(
    f"Categorical Features: "
    f"{categorical_features_review}"
)

print(
    f"\nNumerical Feature Count: "
    f"{len(numerical_features_review)}"
)

print(
    f"Numerical Feature Types Valid: "
    f"{len(invalid_numeric_columns) == 0}"
)


# ============================================================
# STEP 9: VALIDATE MISSINGNESS AND ROW/INDEX PRESERVATION
# ============================================================

# A feature that is completely missing in training cannot
# provide information for fitting an imputer.

train_all_missing_columns = [

    column

    for column in X_candidate.columns

    if X_candidate[column].isna().all()

]


test_all_missing_columns = [

    column

    for column in X_test_candidate.columns

    if X_test_candidate[column].isna().all()

]


if train_all_missing_columns:

    raise ValueError(
        "These features are completely missing in training: "
        f"{train_all_missing_columns}."
    )


if test_all_missing_columns:

    print(
        "\nWarning: Some test features are entirely missing:"
    )

    print(test_all_missing_columns)

    print(
        "Their imputers will still be learned from training data."
    )


# Verify that no rows or indexes were changed.

row_index_preserved = (

    len(X_candidate) == len(train_working)

    and len(X_test_candidate) == len(test_working)

    and X_candidate.index.equals(train_working.index)

    and X_test_candidate.index.equals(test_working.index)

    and y.index.equals(X_candidate.index)

    and test_ids.index.equals(X_test_candidate.index)

)


if not row_index_preserved:

    raise ValueError(
        "Rows or indexes are not aligned across features, "
        "target and test IDs."
    )


# ============================================================
# STEP 10: VERIFY ENGINEERED FEATURE RETENTION
# ============================================================

retention_groups = {

    "Calendar features": [

        "Year",
        "Month",
        "Day",
        "Date_DayOfWeek",
        "WeekOfYear",
        "Quarter",
        "IsWeekend",
        "Month_Position"

    ],

    "Holiday features": [

        "DaysSinceHoliday",
        "DaysUntilHoliday"

    ],

    "Competition features": [

        "CompetitionDistance",
        "CompetitionDistanceMissing",
        "CompetitionOpenInfoMissing",
        "CompetitionOpenDurationMonths"

    ],

    "Promotion features": [

        "Promo2",
        "Promo2InfoMissing",
        "PromoIntervalMissing",
        "Promo2DurationMonths",
        "Promo2NotStarted",
        "Promo2IntervalMonth"

    ],

    "Open missingness feature": [

        "OpenMissing"

    ]

}


retention_results = {}


for group_name, group_features in retention_groups.items():

    retained = [

        feature

        for feature in group_features

        if (
            feature in X_candidate.columns
            and feature in X_test_candidate.columns
        )

    ]


    retention_results[group_name] = (

        len(retained) == len(group_features)

    )


    print(
        f"{group_name}: "
        f"{len(retained)}/{len(group_features)} retained"
    )


all_engineered_features_retained = all(
    retention_results.values()
)


if not all_engineered_features_retained:

    raise ValueError(
        "One or more engineered feature groups "
        f"were not retained: {retention_results}"
    )


# ============================================================
# STEP 11: CHECK TARGET, IDENTIFIER AND DATE LEAKAGE
# ============================================================

forbidden_model_features = [

    "Sales",
    "Customers",
    "Id",
    "Date"

]


leakage_features_found = [

    column

    for column in forbidden_model_features

    if (
        column in X_candidate.columns
        or column in X_test_candidate.columns
    )

]


no_target_identifier_or_date_leakage = (
    len(leakage_features_found) == 0
)


if not no_target_identifier_or_date_leakage:

    raise ValueError(
        "Leakage or identifier columns remain in model inputs: "
        f"{leakage_features_found}."
    )


# ============================================================
# STEP 12: DISPLAY FINAL CANDIDATE DATASET SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("FINAL CANDIDATE DATASET SUMMARY")
print("=" * 90)


print(
    f"\nTraining Feature Matrix: "
    f"{X_candidate.shape}"
)

print(
    f"Test Feature Matrix: "
    f"{X_test_candidate.shape}"
)

print(
    f"Sales Target: "
    f"{y.shape}"
)

print(
    f"Original Test IDs: "
    f"{test_ids.shape}"
)

print(
    f"Number of Candidate Features: "
    f"{X_candidate.shape[1]:,}"
)


print("\nCandidate Features:")


for number, feature in enumerate(
    X_candidate.columns,
    start=1
):

    print(
        f"{number:02d}. {feature}"
    )


# ============================================================
# STEP 13: REVIEW MISSING VALUES BEFORE IMPUTATION
# ============================================================

print("\n" + "-" * 90)
print("MISSING VALUE SUMMARY BEFORE IMPUTATION")
print("-" * 90)


train_missing_summary = X_candidate.isna().sum()

test_missing_summary = X_test_candidate.isna().sum()


print("\nTraining Dataset:")

print(
    train_missing_summary[
        train_missing_summary > 0
    ].to_string()
    or "None"
)


print("\nTest Dataset:")

print(
    test_missing_summary[
        test_missing_summary > 0
    ].to_string()
    or "None"
)


print(
    "\nNo statistical imputation is performed in Section 10."
)

print(
    "Imputation and scaling will be fitted using X_train "
    "only in Section 12."
)


# ============================================================
# STEP 14: FINAL SECTION 10 VALIDATION
# ============================================================

feature_selection_checks = {

    "Section 9 validation passed":
        bool(section9_successful),

    "Sales target valid":
        bool(target_valid),

    "Original test IDs valid":
        bool(test_ids_valid),

    "Required model features available":
        True,

    "Train/test candidate features aligned":
        bool(candidate_features_aligned),

    "Categorical features available":
        len(missing_categorical_features) == 0,

    "Numerical feature types valid":
        len(invalid_numeric_columns) == 0,

    "No infinite numerical values":
        len(infinite_numeric_columns) == 0,

    "No all-missing training features":
        len(train_all_missing_columns) == 0,

    "Rows and indexes preserved":
        bool(row_index_preserved),

    "Engineered feature groups retained":
        bool(all_engineered_features_retained),

    "No target/ID/date leakage":
        bool(no_target_identifier_or_date_leakage),

    "At least one candidate feature":
        X_candidate.shape[1] > 0

}


section_10_successful = all(
    feature_selection_checks.values()
)


# Alias for final notebook summary.

feature_selection_successful = section_10_successful


print("\n" + "=" * 90)
print("SECTION 10 FINAL VALIDATION")
print("=" * 90)


for check_name, check_result in feature_selection_checks.items():

    print(
        f"{check_name:<48}: {check_result}"
    )


print("\n" + "=" * 90)

print(
    f"FEATURE SELECTION COMPLETED SUCCESSFULLY: "
    f"{section_10_successful}"
)

print("=" * 90)


if not section_10_successful:

    failed_checks = [

        name

        for name, result in feature_selection_checks.items()

        if not result

    ]


    logger.error(
        "Section 10 validation failed: %s",
        failed_checks
    )


    raise ValueError(
        "Section 10 validation failed. "
        f"Review these checks: {failed_checks}"
    )


logger.info(
    "Section 10 feature selection and organization "
    "completed successfully."
)


print("\nSECTION 10 COMPLETED SUCCESSFULLY.")

print("=" * 90)

SECTION 10: FINAL FEATURE SELECTION AND FEATURE ORGANIZATION

------------------------------------------------------------------------------------------
SALES TARGET VALIDATION
------------------------------------------------------------------------------------------
Target Name: Sales
Target Rows: 1,017,209
Missing Target Values: 0
Minimum Sales: 0.00
Maximum Sales: 41,551.00
Target Valid: True

------------------------------------------------------------------------------------------
ORIGINAL TEST IDENTIFIER VALIDATION
------------------------------------------------------------------------------------------
Test ID Count: 41,088
Missing Test IDs: 0
Unique Test IDs: True
Test ID Index Aligned: True

------------------------------------------------------------------------------------------
REQUIRED MODEL FEATURE AVAILABILITY
------------------------------------------------------------------------------------------
Required Features: 29
Missing in Training: None
Missing in Test: None



2026-11-06 16:12:27,293 | INFO | Project6_Preprocessing | Section 10 feature selection and organization completed successfully.


CompetitionDistance                2642
DaysUntilHoliday                   1800
CompetitionOpenDurationMonths    323348

Test Dataset:
Open                                11
CompetitionDistance                 96
DaysUntilHoliday                 16782
CompetitionOpenDurationMonths    15216

No statistical imputation is performed in Section 10.
Imputation and scaling will be fitted using X_train only in Section 12.

SECTION 10 FINAL VALIDATION
Section 9 validation passed                     : True
Sales target valid                              : True
Original test IDs valid                         : True
Required model features available               : True
Train/test candidate features aligned           : True
Categorical features available                  : True
Numerical feature types valid                   : True
No infinite numerical values                    : True
No all-missing training features                : True
Rows and indexes preserved                      : True
Eng

# SECTION 11: TEMPORAL TRAIN-VALIDATION SPLIT

## Purpose

This section creates a chronological training and validation split
for the Rossmann Store Sales forecasting problem.

The final 42 calendar days of the available historical training
period are reserved for validation to simulate a six-week
forecasting horizon.

## Temporal Validation Strategy

- Training data contains observations before the validation period.
- Validation data contains the final 42 calendar days of the
  historical training period.
- Test data represents the future period and remains separate.
- Random train-test splitting is not used because it can introduce
  temporal leakage in time-dependent forecasting.

## Feature Preparation

The original Rossmann DayOfWeek feature and the engineered
Date_DayOfWeek feature represent the same weekday information.
Their consistency will be verified before removing the redundant
engineered feature.

Constant features will be identified using the training split only.
The same columns will then be removed from validation and test data.

## Important Methodology

The validation period is not used to fit preprocessing parameters.

The Sales target and original test IDs are preserved separately.

The chronological split maintains the original row indexes so
that features, targets and test identifiers remain aligned.

## Expected Outcome

- X_train and y_train: historical training data.
- X_valid and y_valid: final 42-day validation data.
- X_test_final: future test features.
- test_ids: original test identifiers.
- temporal_split_successful: final section validation status.

These datasets will be used for leakage-aware preprocessing
in Section 12.

In [379]:
# ============================================================
# SECTION 11: TEMPORAL TRAIN-VALIDATION SPLIT
# ============================================================

print("=" * 90)
print("SECTION 11: TEMPORAL TRAIN-VALIDATION SPLIT")
print("=" * 90)


# ------------------------------------------------------------
# STEP 1: CONFIRM SECTION 10 COMPLETED SUCCESSFULLY
# ------------------------------------------------------------

if not globals().get("section_10_successful", False):
    raise RuntimeError(
        "Section 10 has not completed successfully. "
        "Run Sections 1-10 in order before Section 11."
    )

required_objects = {
    "train_working": train_working,
    "test_working": test_working,
    "X_candidate": X_candidate,
    "X_test_candidate": X_test_candidate,
    "y": y,
    "test_ids": test_ids
}

for name, value in required_objects.items():
    if value is None:
        raise ValueError(f"{name} is None.")

for name, value in [
    ("train_working", train_working),
    ("test_working", test_working),
    ("X_candidate", X_candidate),
    ("X_test_candidate", X_test_candidate)
]:
    if not isinstance(value, pd.DataFrame):
        raise TypeError(f"{name} must be a pandas DataFrame.")

if not isinstance(y, pd.Series):
    raise TypeError("y must be a pandas Series.")

if not isinstance(test_ids, pd.Series):
    raise TypeError("test_ids must be a pandas Series.")


# ------------------------------------------------------------
# STEP 2: VERIFY ROW, INDEX AND FEATURE ALIGNMENT
# ------------------------------------------------------------

if not train_working.index.is_unique or not test_working.index.is_unique:
    raise ValueError("Working datasets contain duplicate index labels.")

if not X_candidate.index.equals(train_working.index):
    raise ValueError("X_candidate is not aligned with train_working.")

if not X_test_candidate.index.equals(test_working.index):
    raise ValueError("X_test_candidate is not aligned with test_working.")

if not y.index.equals(X_candidate.index):
    raise ValueError("Sales target is not aligned with X_candidate.")

if not test_ids.index.equals(X_test_candidate.index):
    raise ValueError("Original test IDs are not aligned with X_test_candidate.")

if X_candidate.columns.has_duplicates or X_test_candidate.columns.has_duplicates:
    raise ValueError("Duplicate feature names were found.")

if not X_candidate.columns.equals(X_test_candidate.columns):
    raise ValueError("Training and test candidate features do not match.")


# ------------------------------------------------------------
# STEP 3: PREPARE AND VALIDATE DATE REFERENCES
# ------------------------------------------------------------

date_reference = pd.to_datetime(
    train_working.loc[X_candidate.index, "Date"],
    errors="coerce"
).dt.normalize()

test_date_reference = pd.to_datetime(
    test_working.loc[X_test_candidate.index, "Date"],
    errors="coerce"
).dt.normalize()

if date_reference.isna().any():
    raise ValueError("Invalid or missing training dates were found.")

if test_date_reference.isna().any():
    raise ValueError("Invalid or missing test dates were found.")

if not date_reference.index.equals(X_candidate.index):
    raise ValueError("Training dates are not aligned with training features.")

if not test_date_reference.index.equals(X_test_candidate.index):
    raise ValueError("Test dates are not aligned with test features.")

min_date = date_reference.min()
max_date = date_reference.max()
test_start = test_date_reference.min()
test_end = test_date_reference.max()

if test_start <= max_date:
    raise ValueError("Test dates must start after the training date range.")

print("\n" + "-" * 90)
print("DATE RANGE VALIDATION")
print("-" * 90)
print(f"Training: {min_date.date()} to {max_date.date()}")
print(f"Test:     {test_start.date()} to {test_end.date()}")
print(f"Unique Training Dates: {date_reference.nunique():,}")


# ------------------------------------------------------------
# STEP 4: VERIFY THE REDUNDANT DAY-OF-WEEK FEATURE
# ------------------------------------------------------------

# Date_DayOfWeek uses Monday=0 to Sunday=6.
# Original Rossmann DayOfWeek uses Monday=1 to Sunday=7.
# Therefore, Date_DayOfWeek duplicates DayOfWeek minus one.

weekday_columns = ["Date_DayOfWeek", "DayOfWeek"]

missing_weekday_columns = [
    col for col in weekday_columns
    if col not in X_candidate.columns or col not in X_test_candidate.columns
]

if missing_weekday_columns:
    raise KeyError(
        f"Weekday columns missing from candidate features: "
        f"{missing_weekday_columns}"
    )

train_date_dow = pd.to_numeric(
    X_candidate["Date_DayOfWeek"], errors="coerce"
)

train_rossmann_dow = pd.to_numeric(
    X_candidate["DayOfWeek"], errors="coerce"
)

test_date_dow = pd.to_numeric(
    X_test_candidate["Date_DayOfWeek"], errors="coerce"
)

test_rossmann_dow = pd.to_numeric(
    X_test_candidate["DayOfWeek"], errors="coerce"
)

train_weekday_consistent = (
    train_date_dow.notna().all()
    and train_rossmann_dow.notna().all()
    and np.array_equal(
        train_date_dow.to_numpy(dtype=int),
        date_reference.dt.dayofweek.to_numpy(dtype=int)
    )
    and np.array_equal(
        train_rossmann_dow.to_numpy(dtype=int),
        (date_reference.dt.dayofweek + 1).to_numpy(dtype=int)
    )
)

test_weekday_consistent = (
    test_date_dow.notna().all()
    and test_rossmann_dow.notna().all()
    and np.array_equal(
        test_date_dow.to_numpy(dtype=int),
        test_date_reference.dt.dayofweek.to_numpy(dtype=int)
    )
    and np.array_equal(
        test_rossmann_dow.to_numpy(dtype=int),
        (test_date_reference.dt.dayofweek + 1).to_numpy(dtype=int)
    )
)

print("\n" + "-" * 90)
print("DAY-OF-WEEK CONSISTENCY")
print("-" * 90)
print(f"Training Weekday Consistent: {train_weekday_consistent}")
print(f"Test Weekday Consistent:     {test_weekday_consistent}")

if not train_weekday_consistent or not test_weekday_consistent:
    raise ValueError(
        "DayOfWeek and Date_DayOfWeek do not match the Date column. "
        "Review Section 4 before proceeding."
    )


# ------------------------------------------------------------
# STEP 5: REMOVE VERIFIED REDUNDANT FEATURE
# ------------------------------------------------------------

redundant_features = ["Date_DayOfWeek"]

X_final = X_candidate.drop(
    columns=redundant_features
).copy()

X_test_final = X_test_candidate.drop(
    columns=redundant_features
).copy()

if not X_final.columns.equals(X_test_final.columns):
    raise ValueError(
        "Feature columns do not match after redundant-feature removal."
    )

print("\nRemoved redundant feature:", redundant_features)
print("Training candidate shape:", X_final.shape)
print("Test candidate shape:", X_test_final.shape)


# ------------------------------------------------------------
# STEP 6: DEFINE THE SIX-WEEK (42-DAY) VALIDATION WINDOW
# ------------------------------------------------------------

# The final 42 calendar days of the historical training period
# are held out to simulate a six-week forecasting horizon.

validation_days = 42
validation_end_date = max_date

validation_start_date = (
    validation_end_date - pd.Timedelta(days=validation_days - 1)
)

if date_reference.nunique() <= validation_days:
    raise ValueError(
        "Insufficient historical dates for a 42-day validation window."
    )

expected_validation_dates = pd.date_range(
    start=validation_start_date,
    end=validation_end_date,
    freq="D"
)

train_mask = date_reference < validation_start_date
valid_mask = date_reference >= validation_start_date

all_rows_assigned = (
    (train_mask | valid_mask).all()
    and not (train_mask & valid_mask).any()
)

if not all_rows_assigned:
    raise ValueError(
        "Some rows were not assigned to exactly one split."
    )

actual_validation_dates = pd.DatetimeIndex(
    date_reference.loc[valid_mask].drop_duplicates().sort_values()
)

validation_dates_complete = actual_validation_dates.equals(
    pd.DatetimeIndex(expected_validation_dates)
)

validation_unique_dates = len(actual_validation_dates)

if not validation_dates_complete:
    raise ValueError(
        "Validation data does not contain all 42 expected calendar dates."
    )


# ------------------------------------------------------------
# STEP 7: CREATE CHRONOLOGICAL TRAINING AND VALIDATION SETS
# ------------------------------------------------------------

X_train = X_final.loc[train_mask].copy()
X_valid = X_final.loc[valid_mask].copy()

y_train = y.loc[train_mask].copy()
y_valid = y.loc[valid_mask].copy()

if X_train.empty or X_valid.empty:
    raise ValueError("Training or validation dataset is empty.")


# ------------------------------------------------------------
# STEP 8: REMOVE CONSTANT FEATURES USING TRAINING DATA ONLY
# ------------------------------------------------------------

# Only X_train is used to identify constant features.
# The same columns are then removed from validation and test.

constant_features = [
    col for col in X_train.columns
    if X_train[col].nunique(dropna=False) <= 1
]

X_train = X_train.drop(columns=constant_features).copy()
X_valid = X_valid.drop(columns=constant_features).copy()
X_test_final = X_test_final.drop(columns=constant_features).copy()

if X_train.shape[1] == 0:
    raise ValueError("No features remain after constant-feature removal.")

print("\n" + "-" * 90)
print("TRAINING-ONLY CONSTANT FEATURE REMOVAL")
print("-" * 90)
print(f"Constant features removed ({len(constant_features)}):")
print(constant_features if constant_features else "None")


# ------------------------------------------------------------
# STEP 9: VALIDATE TARGETS, TEST IDS AND FEATURE ALIGNMENT
# ------------------------------------------------------------

features_aligned = (
    X_train.columns.equals(X_valid.columns)
    and X_train.columns.equals(X_test_final.columns)
)

train_target_aligned = X_train.index.equals(y_train.index)
valid_target_aligned = X_valid.index.equals(y_valid.index)

test_ids_valid = (
    test_ids.notna().all()
    and test_ids.is_unique
    and test_ids.index.equals(X_test_final.index)
    and len(test_ids) == len(X_test_final)
)

targets_valid = (
    y_train.notna().all()
    and y_valid.notna().all()
    and np.isfinite(y_train.to_numpy(dtype=float)).all()
    and np.isfinite(y_valid.to_numpy(dtype=float)).all()
    and y_train.ge(0).all()
    and y_valid.ge(0).all()
)

if not features_aligned:
    raise ValueError("Train, validation and test feature columns do not match.")

if not train_target_aligned or not valid_target_aligned:
    raise ValueError("Features and Sales target are not correctly aligned.")

if not test_ids_valid:
    raise ValueError("Test IDs are missing, duplicated or misaligned.")

if not targets_valid:
    raise ValueError("Sales target contains invalid values.")


# ------------------------------------------------------------
# STEP 10: VALIDATE TEMPORAL SEPARATION
# ------------------------------------------------------------

train_start = date_reference.loc[X_train.index].min()
train_end = date_reference.loc[X_train.index].max()

valid_start = date_reference.loc[X_valid.index].min()
valid_end = date_reference.loc[X_valid.index].max()

no_temporal_overlap = (
    train_end < valid_start
    and valid_end < test_start
)

if not no_temporal_overlap:
    raise ValueError(
        "Training, validation and test periods overlap."
    )


# ------------------------------------------------------------
# STEP 11: DISPLAY FINAL DATASET SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FINAL TEMPORAL DATASET SUMMARY")
print("=" * 90)

print(f"\nTraining Period:   {train_start.date()} to {train_end.date()}")
print(f"Validation Period: {valid_start.date()} to {valid_end.date()}")
print(f"Test Period:       {test_start.date()} to {test_end.date()}")

print("\nDataset Shapes:")
print(f"X_train:      {X_train.shape}")
print(f"X_valid:      {X_valid.shape}")
print(f"X_test_final: {X_test_final.shape}")
print(f"y_train:      {y_train.shape}")
print(f"y_valid:      {y_valid.shape}")
print(f"test_ids:     {test_ids.shape}")

print(f"\nValidation Days: {validation_unique_dates}")
print(f"Final Raw Features: {X_train.shape[1]}")


# ------------------------------------------------------------
# STEP 12: FINAL SECTION 11 VALIDATION
# ------------------------------------------------------------

temporal_split_checks = {
    "Section 10 completed": bool(section_10_successful),

    "Training and test dates valid": (
        date_reference.notna().all()
        and test_date_reference.notna().all()
    ),

    "Training and test indexes preserved": (
        X_candidate.index.equals(train_working.index)
        and X_test_candidate.index.equals(test_working.index)
    ),

    "Weekday features verified": (
        train_weekday_consistent and test_weekday_consistent
    ),

    "All 42 validation dates present": (
        validation_unique_dates == validation_days
        and validation_dates_complete
    ),

    "All rows assigned to one split": bool(all_rows_assigned),

    "Train and validation sets non-empty": (
        not X_train.empty and not X_valid.empty
    ),

    "Train/validation/test features aligned": bool(features_aligned),

    "Training target aligned": bool(train_target_aligned),

    "Validation target aligned": bool(valid_target_aligned),

    "Original test IDs valid": bool(test_ids_valid),

    "Sales target values valid": bool(targets_valid),

    "No temporal overlap": bool(no_temporal_overlap),

    "At least one feature remains": X_train.shape[1] > 0
}

print("\n" + "=" * 90)
print("SECTION 11 VALIDATION CHECKLIST")
print("=" * 90)

for check_name, check_result in temporal_split_checks.items():
    print(f"{check_name:<45}: {check_result}")

temporal_split_successful = all(
    temporal_split_checks.values()
)

print("\n" + "=" * 90)
print(
    "TEMPORAL SPLIT COMPLETED SUCCESSFULLY: "
    f"{temporal_split_successful}"
)
print("=" * 90)

if not temporal_split_successful:
    failed_checks = [
        name for name, result in temporal_split_checks.items()
        if not result
    ]
    raise ValueError(
        f"Section 11 validation failed: {failed_checks}"
    )

logger.info(
    "Section 11 chronological train-validation split "
    "completed successfully."
)

print("\nSECTION 11 COMPLETED SUCCESSFULLY.")

SECTION 11: TEMPORAL TRAIN-VALIDATION SPLIT

------------------------------------------------------------------------------------------
DATE RANGE VALIDATION
------------------------------------------------------------------------------------------
Training: 2013-01-01 to 2015-07-31
Test:     2015-08-01 to 2015-09-17
Unique Training Dates: 942

------------------------------------------------------------------------------------------
DAY-OF-WEEK CONSISTENCY
------------------------------------------------------------------------------------------
Training Weekday Consistent: True
Test Weekday Consistent:     True

Removed redundant feature: ['Date_DayOfWeek']
Training candidate shape: (1017209, 28)
Test candidate shape: (41088, 28)


2026-11-06 16:12:28,308 | INFO | Project6_Preprocessing | Section 11 chronological train-validation split completed successfully.



------------------------------------------------------------------------------------------
TRAINING-ONLY CONSTANT FEATURE REMOVAL
------------------------------------------------------------------------------------------
Constant features removed (3):
['Promo2InfoMissing', 'PromoIntervalMissing', 'OpenMissing']

FINAL TEMPORAL DATASET SUMMARY

Training Period:   2013-01-01 to 2015-06-19
Validation Period: 2015-06-20 to 2015-07-31
Test Period:       2015-08-01 to 2015-09-17

Dataset Shapes:
X_train:      (970379, 25)
X_valid:      (46830, 25)
X_test_final: (41088, 25)
y_train:      (970379,)
y_valid:      (46830,)
test_ids:     (41088,)

Validation Days: 42
Final Raw Features: 25

SECTION 11 VALIDATION CHECKLIST
Section 10 completed                         : True
Training and test dates valid                : True
Training and test indexes preserved          : True
Weekday features verified                    : True
All 42 validation dates present              : True
All rows assigned 

# SECTION 12: FINAL LEAKAGE-AWARE DATA PREPROCESSING

## Purpose

This section transforms the final chronological training,
validation and test feature datasets created in Section 11
into model-ready numerical matrices.

Numerical and categorical variables are processed separately
using a Scikit-learn ColumnTransformer pipeline.

## Numerical Feature Processing

Numerical features are processed using:

- Median imputation for missing numerical values.
- StandardScaler for feature scaling.

The median values and scaling parameters are learned only
from X_train.

## Categorical Feature Processing

Categorical features are processed using:

- Most-frequent-value imputation.
- One-Hot Encoding.
- Unknown categories are ignored during validation and
  test transformations.

The categorical columns are defined from the feature strategy
established in Sections 8-10:

- Store
- StoreType
- Assortment
- StateHoliday
- Month_Position

## Leakage Prevention

The preprocessor is fitted only on X_train.

The fitted preprocessor is then used to transform:

- X_train
- X_valid
- X_test_final

Validation and test data are never used to learn imputation,
scaling or encoding parameters.

## Important Methodology

Constant features have already been identified and removed
in Section 11 using the chronological training split only.

This section verifies that no constant features remain.
It does not remove them again.

The Section 11 feature DataFrames are preserved unchanged.
Preprocessing is performed on separate copies so that the
original split, targets and test identifiers remain aligned.

## Expected Outcome

- X_train_processed
- X_valid_processed
- X_test_processed
- preprocessor
- feature_names
- numerical_features
- categorical_features
- section_12_successful

All processed datasets must have matching feature dimensions,
no missing values and no infinite values.

In [380]:

# ============================================================
# SECTION 12: FINAL LEAKAGE-AWARE DATA PREPROCESSING
# ============================================================

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


print("=" * 90)
print("SECTION 12: FINAL LEAKAGE-AWARE DATA PREPROCESSING")
print("=" * 90)


# ============================================================
# STEP 1: VERIFY SECTION 10 AND SECTION 11 COMPLETION
# ============================================================

if not globals().get("section_10_successful", False):
    raise RuntimeError(
        "Section 10 has not completed successfully. "
        "Run Sections 1-10 in order before Section 12."
    )

if not globals().get("temporal_split_successful", False):
    raise RuntimeError(
        "Section 11 has not completed successfully. "
        "Run Sections 1-11 in order before Section 12."
    )

required_objects = [
    "X_train",
    "X_valid",
    "X_test_final",
    "y_train",
    "y_valid",
    "test_ids",
    "categorical_features_review",
    "logger"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Required objects from previous sections are missing: "
        f"{missing_objects}. Re-run Sections 1-11 in order."
    )


# ============================================================
# STEP 2: VALIDATE THE SECTION 11 INPUT DATASETS
# ============================================================

input_datasets = {
    "X_train": X_train,
    "X_valid": X_valid,
    "X_test_final": X_test_final
}

for dataset_name, dataset in input_datasets.items():

    if not isinstance(dataset, pd.DataFrame):
        raise TypeError(
            f"{dataset_name} must be a pandas DataFrame."
        )

    if dataset.empty:
        raise ValueError(
            f"{dataset_name} is empty."
        )

    if dataset.columns.has_duplicates:
        raise ValueError(
            f"{dataset_name} contains duplicate feature names."
        )


if not isinstance(y_train, pd.Series):
    raise TypeError("y_train must be a pandas Series.")

if not isinstance(y_valid, pd.Series):
    raise TypeError("y_valid must be a pandas Series.")

if not isinstance(test_ids, pd.Series):
    raise TypeError("test_ids must be a pandas Series.")


# Work on copies so Section 11's original split remains unchanged.

X_train_prepared = X_train.copy()

X_valid_prepared = X_valid.copy()

X_test_prepared = X_test_final.copy()


prepared_datasets = {
    "X_train": X_train_prepared,
    "X_valid": X_valid_prepared,
    "X_test_final": X_test_prepared
}


# ============================================================
# STEP 3: VERIFY FEATURE, TARGET AND ID ALIGNMENT
# ============================================================

raw_features_aligned = (
    X_train_prepared.columns.equals(X_valid_prepared.columns)
    and X_train_prepared.columns.equals(X_test_prepared.columns)
)

if not raw_features_aligned:
    raise ValueError(
        "Training, validation and test features do not have "
        "identical columns in identical order. Check Section 11."
    )


index_alignment = (
    X_train_prepared.index.equals(y_train.index)
    and X_valid_prepared.index.equals(y_valid.index)
    and X_test_prepared.index.equals(test_ids.index)
)

if not index_alignment:
    raise ValueError(
        "Feature, target or test-ID indexes are misaligned. "
        "Check Section 11."
    )


print("\nSection 11 Input Dataset Shapes:")

print(f"X_train:      {X_train_prepared.shape}")

print(f"X_valid:      {X_valid_prepared.shape}")

print(f"X_test_final: {X_test_prepared.shape}")

print(f"Raw feature alignment: {raw_features_aligned}")

print(f"Index alignment: {index_alignment}")


# ============================================================
# STEP 4: VERIFY CONSTANT FEATURES WERE REMOVED IN SECTION 11
# ============================================================

# Section 11 already removed constant features using X_train only.
# Do not remove features again in Section 12.

remaining_constant_features = [
    column
    for column in X_train_prepared.columns
    if X_train_prepared[column].nunique(dropna=False) <= 1
]

print("\n" + "-" * 90)

print("CONSTANT FEATURE VALIDATION")

print("-" * 90)

print(
    "Remaining constant features: "
    f"{remaining_constant_features or 'None'}"
)

if remaining_constant_features:
    raise ValueError(
        "Constant features remain after Section 11: "
        f"{remaining_constant_features}. Review Section 11."
    )


# ============================================================
# STEP 5: IDENTIFY CATEGORICAL FEATURES FROM SECTION 10
# ============================================================

# Section 10 defines these categorical features:
# Store, StoreType, Assortment, StateHoliday, Month_Position.

categorical_features = [
    feature
    for feature in categorical_features_review
    if feature in X_train_prepared.columns
]

missing_categorical_features = [
    feature
    for feature in categorical_features_review
    if feature not in X_train_prepared.columns
]

if missing_categorical_features:
    raise KeyError(
        "Categorical features defined in Section 10 are missing "
        f"from Section 11: {missing_categorical_features}"
    )

# Preserve the original column order.

categorical_features = [
    column
    for column in X_train_prepared.columns
    if column in categorical_features
]

if not categorical_features:
    raise ValueError(
        "No categorical features were identified."
    )


# ============================================================
# STEP 6: IDENTIFY NUMERICAL FEATURES
# ============================================================

numerical_features = [
    column
    for column in X_train_prepared.columns
    if column not in categorical_features
]

if not numerical_features:
    raise ValueError(
        "No numerical features were identified."
    )

print("\n" + "-" * 90)

print("FEATURE IDENTIFICATION")

print("-" * 90)

print(f"Categorical Features ({len(categorical_features)}):")

print(categorical_features)

print(f"\nNumerical Features ({len(numerical_features)}):")

print(numerical_features)

print(
    f"\nTotal Raw Features: "
    f"{len(categorical_features) + len(numerical_features)}"
)


# ============================================================
# STEP 7: VALIDATE AND CONVERT NUMERICAL FEATURES
# ============================================================

# Convert numerical columns on copies only.
# Missing values are preserved for median imputation.

for column in numerical_features:

    for dataset_name, frame in prepared_datasets.items():

        original_values = frame[column]

        converted_values = pd.to_numeric(
            original_values,
            errors="coerce"
        )

        invalid_text = (
            original_values.notna()
            & converted_values.isna()
        )

        if invalid_text.any():

            examples = (
                original_values.loc[invalid_text]
                .head(5)
                .tolist()
            )

            raise TypeError(
                f"Non-numeric values found in '{column}' "
                f"in {dataset_name}. Examples: {examples}"
            )

        finite_values = converted_values.dropna().to_numpy(
            dtype="float64"
        )

        if not np.isfinite(finite_values).all():
            raise ValueError(
                f"Infinite values found in '{column}' "
                f"in {dataset_name}."
            )

        frame[column] = converted_values


all_missing_numeric_features = [
    column
    for column in numerical_features
    if X_train_prepared[column].isna().all()
]

if all_missing_numeric_features:
    raise ValueError(
        "Numerical features completely missing in training; "
        "median imputation is not possible: "
        f"{all_missing_numeric_features}"
    )


# ============================================================
# STEP 8: PREPARE CATEGORICAL VALUES CONSISTENTLY
# ============================================================

# Convert non-missing category values to strings in all splits.
# Preserve actual missing values as np.nan.

for column in categorical_features:

    for dataset_name, frame in prepared_datasets.items():

        source_values = frame[column].to_numpy(
            dtype=object
        )

        normalized_values = [
            np.nan if pd.isna(value) else str(value)
            for value in source_values
        ]

        frame[column] = pd.Series(
            normalized_values,
            index=frame.index,
            dtype=object
        )


all_missing_categorical_features = [
    column
    for column in categorical_features
    if X_train_prepared[column].isna().all()
]

if all_missing_categorical_features:
    raise ValueError(
        "Categorical features completely missing in training: "
        f"{all_missing_categorical_features}"
    )


# ============================================================
# STEP 9: CREATE NUMERICAL PREPROCESSING PIPELINE
# ============================================================

numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ============================================================
# STEP 10: CREATE CATEGORICAL PREPROCESSING PIPELINE
# ============================================================

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)


# ============================================================
# STEP 11: COMBINE BOTH PIPELINES
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_transformer,
            numerical_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ],
    remainder="drop"
)


print("\n" + "-" * 90)

print("PREPROCESSING PIPELINE")

print("-" * 90)

print("Numerical: Median Imputation -> StandardScaler")

print("Categorical: Most Frequent Imputation -> OneHotEncoder")

print("Fit rule: X_train only")


# ============================================================
# STEP 12: FIT PREPROCESSOR USING X_TRAIN ONLY
# ============================================================

print("\n" + "-" * 90)

print("FITTING PREPROCESSOR ON TRAINING DATA ONLY")

print("-" * 90)

X_train_processed = preprocessor.fit_transform(
    X_train_prepared
)

print(
    "Preprocessor fitted successfully on X_train."
)


# ============================================================
# STEP 13: TRANSFORM VALIDATION AND TEST DATA
# ============================================================

# Do not call fit or fit_transform on validation or test data.

X_valid_processed = preprocessor.transform(
    X_valid_prepared
)

X_test_processed = preprocessor.transform(
    X_test_prepared
)


# ============================================================
# STEP 14: GENERATE PROCESSED FEATURE NAMES
# ============================================================

feature_names = np.asarray(
    preprocessor.get_feature_names_out(),
    dtype=str
)

feature_names_unique = pd.Index(
    feature_names
).is_unique

if not feature_names_unique:
    raise ValueError(
        "Processed feature names are not unique."
    )


# ============================================================
# STEP 15: VALIDATE PROCESSED DATASET DIMENSIONS
# ============================================================

# IMPORTANT:
# Sparse matrices do not support len().
# Use shape[0] for row counts in dense and sparse matrices.

processed_dimensions_valid = (
    X_train_processed.shape[0] == X_train_prepared.shape[0]
    and X_valid_processed.shape[0] == X_valid_prepared.shape[0]
    and X_test_processed.shape[0] == X_test_prepared.shape[0]
    and X_train_processed.shape[1]
        == X_valid_processed.shape[1]
        == X_test_processed.shape[1]
    and X_train_processed.shape[1] == len(feature_names)
)

if not processed_dimensions_valid:
    raise ValueError(
        "Processed matrix dimensions or feature names do not match. "
        f"Train={X_train_processed.shape}, "
        f"Valid={X_valid_processed.shape}, "
        f"Test={X_test_processed.shape}, "
        f"Feature names={len(feature_names)}"
    )


# ============================================================
# STEP 16: CHECK MISSING AND INFINITE VALUES
# ============================================================

def get_matrix_quality_counts(matrix):
    """
    Count missing and infinite stored values
    in dense or sparse matrices.
    """

    if sp.issparse(matrix):
        values = matrix.data
    else:
        values = np.asarray(matrix).ravel()

    missing_count = int(
        np.isnan(values).sum()
    )

    infinite_count = int(
        np.isinf(values).sum()
    )

    return missing_count, infinite_count


train_missing, train_infinite = get_matrix_quality_counts(
    X_train_processed
)

valid_missing, valid_infinite = get_matrix_quality_counts(
    X_valid_processed
)

test_missing, test_infinite = get_matrix_quality_counts(
    X_test_processed
)


no_missing_values = (
    train_missing == 0
    and valid_missing == 0
    and test_missing == 0
)

no_infinite_values = (
    train_infinite == 0
    and valid_infinite == 0
    and test_infinite == 0
)


if not no_missing_values:
    raise ValueError(
        "Missing values remain after preprocessing: "
        f"train={train_missing}, "
        f"valid={valid_missing}, "
        f"test={test_missing}"
    )

if not no_infinite_values:
    raise ValueError(
        "Infinite values remain after preprocessing: "
        f"train={train_infinite}, "
        f"valid={valid_infinite}, "
        f"test={test_infinite}"
    )


# ============================================================
# STEP 17: VERIFY PREPROCESSOR AND TARGET ALIGNMENT
# ============================================================

preprocessor_fitted = hasattr(
    preprocessor,
    "transformers_"
)

target_alignment_valid = (
    X_train.index.equals(y_train.index)
    and X_valid.index.equals(y_valid.index)
)


# FIXED SPARSE MATRIX ROW COUNT CHECK:
# Do not use len(X_test_processed).
# Use .shape[0] because X_test_processed may be a sparse matrix.

processed_test_rows = X_test_processed.shape[0]

original_test_id_rows = test_ids.shape[0]

test_id_alignment_valid = (
    X_test_final.index.equals(test_ids.index)
    and processed_test_rows == original_test_id_rows
)


# Additional target and identifier integrity checks.

target_values_valid = (
    pd.to_numeric(y_train, errors="coerce").notna().all()
    and pd.to_numeric(y_valid, errors="coerce").notna().all()
    and np.isfinite(y_train.to_numpy(dtype="float64")).all()
    and np.isfinite(y_valid.to_numpy(dtype="float64")).all()
    and y_train.ge(0).all()
    and y_valid.ge(0).all()
)

test_ids_valid = (
    test_ids.notna().all()
    and test_ids.is_unique
    and test_id_alignment_valid
)


# ============================================================
# STEP 18: FINAL SECTION 12 VALIDATION
# ============================================================

preprocessing_checks = {

    "Section 10 completed":
        bool(section_10_successful),

    "Section 11 completed":
        bool(temporal_split_successful),

    "Raw feature columns aligned":
        bool(raw_features_aligned),

    "Input indexes aligned":
        bool(index_alignment),

    "No constant features remaining":
        len(remaining_constant_features) == 0,

    "Numerical features identified":
        len(numerical_features) > 0,

    "Categorical features identified":
        len(categorical_features) > 0,

    "No all-missing numerical training features":
        len(all_missing_numeric_features) == 0,

    "No all-missing categorical training features":
        len(all_missing_categorical_features) == 0,

    "Preprocessor fitted":
        bool(preprocessor_fitted),

    "Target alignment valid":
        bool(target_alignment_valid),

    "Target values valid":
        bool(target_values_valid),

    "Test IDs valid":
        bool(test_ids_valid),

    "Processed dimensions valid":
        bool(processed_dimensions_valid),

    "Feature names unique":
        bool(feature_names_unique),

    "No missing processed values":
        bool(no_missing_values),

    "No infinite processed values":
        bool(no_infinite_values)

}


print("\n" + "=" * 90)

print("SECTION 12 FINAL VALIDATION")

print("=" * 90)


print("\nProcessed Dataset Shapes:")

print(
    f"X_train_processed: {X_train_processed.shape}"
)

print(
    f"X_valid_processed: {X_valid_processed.shape}"
)

print(
    f"X_test_processed:  {X_test_processed.shape}"
)


print(
    f"\nTotal Processed Features: {len(feature_names):,}"
)

print(
    f"Numerical Features: {len(numerical_features)}"
)

print(
    f"Categorical Features: {len(categorical_features)}"
)


print("\nMissing Values:")

print(f"Training:   {train_missing:,}")

print(f"Validation: {valid_missing:,}")

print(f"Test:       {test_missing:,}")


print("\nInfinite Values:")

print(f"Training:   {train_infinite:,}")

print(f"Validation: {valid_infinite:,}")

print(f"Test:       {test_infinite:,}")


for check_name, check_result in preprocessing_checks.items():

    print(
        f"{check_name:<48}: {check_result}"
    )


preprocessing_successful = all(
    preprocessing_checks.values()
)

section_12_successful = preprocessing_successful

# Compatibility alias for later notebook code.

section12_successful = section_12_successful


print("\n" + "=" * 90)

print(
    "PREPROCESSING COMPLETED SUCCESSFULLY: "
    f"{section_12_successful}"
)

print("=" * 90)


if not section_12_successful:

    failed_checks = [
        name
        for name, result in preprocessing_checks.items()
        if not result
    ]

    raise ValueError(
        "Section 12 validation failed: "
        f"{failed_checks}"
    )


logger.info(
    "Section 12 leakage-aware preprocessing completed successfully."
)

print("\nSECTION 12 COMPLETED SUCCESSFULLY.")


SECTION 12: FINAL LEAKAGE-AWARE DATA PREPROCESSING

Section 11 Input Dataset Shapes:
X_train:      (970379, 25)
X_valid:      (46830, 25)
X_test_final: (41088, 25)
Raw feature alignment: True
Index alignment: True

------------------------------------------------------------------------------------------
CONSTANT FEATURE VALIDATION
------------------------------------------------------------------------------------------
Remaining constant features: None

------------------------------------------------------------------------------------------
FEATURE IDENTIFICATION
------------------------------------------------------------------------------------------
Categorical Features (5):
['Store', 'StateHoliday', 'StoreType', 'Assortment', 'Month_Position']

Numerical Features (20):
['DayOfWeek', 'Open', 'Promo', 'SchoolHoliday', 'CompetitionDistance', 'Promo2', 'Year', 'Month', 'Day', 'WeekOfYear', 'Quarter', 'IsWeekend', 'DaysSinceHoliday', 'DaysUntilHoliday', 'CompetitionDistanceMissing',

2026-11-06 16:12:35,158 | INFO | Project6_Preprocessing | Section 12 leakage-aware preprocessing completed successfully.



SECTION 12 FINAL VALIDATION

Processed Dataset Shapes:
X_train_processed: (970379, 1149)
X_valid_processed: (46830, 1149)
X_test_processed:  (41088, 1149)

Total Processed Features: 1,149
Numerical Features: 20
Categorical Features: 5

Missing Values:
Training:   0
Validation: 0
Test:       0

Infinite Values:
Training:   0
Validation: 0
Test:       0
Section 10 completed                            : True
Section 11 completed                            : True
Raw feature columns aligned                     : True
Input indexes aligned                           : True
No constant features remaining                  : True
Numerical features identified                   : True
Categorical features identified                 : True
No all-missing numerical training features      : True
No all-missing categorical training features    : True
Preprocessor fitted                             : True
Target alignment valid                          : True
Target values valid                     

# SECTION 13: FINAL MODEL-READINESS VALIDATION

## Purpose

This section performs final structural, numerical and temporal
validation of all processed datasets created in Section 12.

## Validation Areas

1. Training, validation and test row counts.
2. Raw and processed feature alignment.
3. Target and identifier leakage checks.
4. Training and validation target alignment.
5. Original test identifier integrity.
6. Chronological separation of all datasets.
7. Validation period of 42 calendar days.
8. Processed feature dimensions and feature names.
9. Missing and infinite value checks.
10. Preprocessor fitting and transformation consistency.

## Important Methodology

The chronological split created in Section 11 is preserved.

The preprocessing pipeline from Section 12 must be fitted
only on X_train.

Validation and test datasets must be transformed using the
same fitted preprocessing pipeline.

No model is trained in this section.

## Expected Outcome

- validation_results
- datasets_ready_for_modeling
- section_13_successful

All checks must pass before saving model-ready artifacts
in Section 14.

In [381]:
# ============================================================
# SECTION 13: FINAL MODEL-READINESS VALIDATION
# ============================================================

print("=" * 90)
print("SECTION 13: FINAL MODEL-READINESS VALIDATION")
print("=" * 90)


# ============================================================
# STEP 1: VERIFY SECTION 12 COMPLETION
# ============================================================

if not globals().get("section_12_successful", False):
    raise RuntimeError(
        "Section 12 has not completed successfully. "
        "Run Sections 1-12 in order."
    )


# ============================================================
# STEP 2: VERIFY REQUIRED OBJECTS
# ============================================================

required_objects = [
    "X_train",
    "X_valid",
    "X_test_final",
    "X_train_prepared",
    "X_valid_prepared",
    "X_test_prepared",
    "X_train_processed",
    "X_valid_processed",
    "X_test_processed",
    "y_train",
    "y_valid",
    "test_ids",
    "preprocessor",
    "feature_names",
    "numerical_features",
    "categorical_features",
    "date_reference",
    "test_working",
    "preprocessing_checks"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Required objects are missing: "
        f"{missing_objects}. Re-run Sections 11-12."
    )


# ============================================================
# STEP 3: VALIDATE INPUT DATA TYPES
# ============================================================

dataframes_to_check = {
    "X_train": X_train,
    "X_valid": X_valid,
    "X_test_final": X_test_final,
    "X_train_prepared": X_train_prepared,
    "X_valid_prepared": X_valid_prepared,
    "X_test_prepared": X_test_prepared
}

for dataset_name, dataset in dataframes_to_check.items():

    if not isinstance(dataset, pd.DataFrame):
        raise TypeError(
            f"{dataset_name} must be a pandas DataFrame."
        )


if not isinstance(y_train, pd.Series):
    raise TypeError("y_train must be a pandas Series.")

if not isinstance(y_valid, pd.Series):
    raise TypeError("y_valid must be a pandas Series.")

if not isinstance(test_ids, pd.Series):
    raise TypeError("test_ids must be a pandas Series.")


# ============================================================
# STEP 4: VALIDATE ROW COUNTS AND INDEX ALIGNMENT
# ============================================================

train_rows_valid = (
    X_train_processed.shape[0]
    == X_train.shape[0]
    == y_train.shape[0]
)

valid_rows_valid = (
    X_valid_processed.shape[0]
    == X_valid.shape[0]
    == y_valid.shape[0]
)

test_rows_valid = (
    X_test_processed.shape[0]
    == X_test_final.shape[0]
    == test_ids.shape[0]
)


raw_feature_alignment = (
    X_train.columns.equals(X_valid.columns)
    and X_train.columns.equals(X_test_final.columns)
)

prepared_feature_alignment = (
    X_train_prepared.columns.equals(X_train.columns)
    and X_valid_prepared.columns.equals(X_train.columns)
    and X_test_prepared.columns.equals(X_train.columns)
)

target_index_alignment = (
    X_train.index.equals(y_train.index)
    and X_valid.index.equals(y_valid.index)
)

test_id_index_alignment = (
    X_test_final.index.equals(test_ids.index)
)

prepared_index_alignment = (
    X_train_prepared.index.equals(X_train.index)
    and X_valid_prepared.index.equals(X_valid.index)
    and X_test_prepared.index.equals(X_test_final.index)
)


# ============================================================
# STEP 5: CHECK TARGET AND IDENTIFIER LEAKAGE
# ============================================================

forbidden_features = [
    "Sales",
    "Customers",
    "Id",
    "Date"
]

leakage_features_found = [
    column
    for column in forbidden_features
    if column in X_train.columns
]

no_target_or_identifier_leakage = (
    len(leakage_features_found) == 0
)


# ============================================================
# STEP 6: VERIFY CATEGORICAL AND NUMERICAL FEATURE COVERAGE
# ============================================================

categorical_features_valid = (
    len(categorical_features) > 0
    and len(categorical_features) == len(set(categorical_features))
    and all(
        column in X_train.columns
        for column in categorical_features
    )
)

numerical_features_valid = (
    len(numerical_features) > 0
    and len(numerical_features) == len(set(numerical_features))
    and all(
        column in X_train.columns
        for column in numerical_features
    )
)

feature_groups_do_not_overlap = (
    set(categorical_features).isdisjoint(
        set(numerical_features)
    )
)

feature_groups_cover_all_inputs = (
    set(categorical_features).union(
        set(numerical_features)
    ) == set(X_train.columns)
)


# ============================================================
# STEP 7: VALIDATE CONSTANT FEATURE REMOVAL
# ============================================================

# Constant features must already have been removed in Section 11.

remaining_constant_features = [
    column
    for column in X_train.columns
    if X_train[column].nunique(dropna=False) <= 1
]

no_constant_features_remaining = (
    len(remaining_constant_features) == 0
)


# ============================================================
# STEP 8: VALIDATE TARGET VALUES AND TEST IDS
# ============================================================

train_target_values = pd.to_numeric(
    y_train,
    errors="coerce"
).to_numpy(dtype="float64")

valid_target_values = pd.to_numeric(
    y_valid,
    errors="coerce"
).to_numpy(dtype="float64")


targets_valid = (
    np.isfinite(train_target_values).all()
    and np.isfinite(valid_target_values).all()
    and (train_target_values >= 0).all()
    and (valid_target_values >= 0).all()
)


test_ids_valid = (
    test_ids.notna().all()
    and test_ids.is_unique
    and test_id_index_alignment
    and test_ids.shape[0] == X_test_processed.shape[0]
)


# ============================================================
# STEP 9: VALIDATE CHRONOLOGICAL SPLIT
# ============================================================

train_dates = pd.to_datetime(
    date_reference.loc[X_train.index],
    errors="coerce"
).dt.normalize()

valid_dates = pd.to_datetime(
    date_reference.loc[X_valid.index],
    errors="coerce"
).dt.normalize()

test_dates = pd.to_datetime(
    test_working.loc[X_test_final.index, "Date"],
    errors="coerce"
).dt.normalize()


dates_valid = (
    train_dates.notna().all()
    and valid_dates.notna().all()
    and test_dates.notna().all()
)

if not dates_valid:
    raise ValueError(
        "Invalid or missing dates found in final datasets."
    )


validation_unique_dates = valid_dates.nunique()

expected_validation_dates = pd.date_range(
    start=valid_dates.min(),
    end=valid_dates.max(),
    freq="D"
)

actual_validation_dates = pd.DatetimeIndex(
    valid_dates.drop_duplicates().sort_values()
)

validation_period_valid = (
    validation_unique_dates == 42
    and actual_validation_dates.equals(
        pd.DatetimeIndex(expected_validation_dates)
    )
)

chronological_order_valid = (
    train_dates.max() < valid_dates.min()
    and valid_dates.max() < test_dates.min()
)


# ============================================================
# STEP 10: VALIDATE PROCESSED MATRIX DIMENSIONS
# ============================================================

processed_feature_dimensions_valid = (
    X_train_processed.shape[1]
    == X_valid_processed.shape[1]
    == X_test_processed.shape[1]
)

feature_names_from_pipeline = np.asarray(
    preprocessor.get_feature_names_out(),
    dtype=str
)

feature_names = np.asarray(
    feature_names,
    dtype=str
)

feature_names_count_valid = (
    len(feature_names)
    == X_train_processed.shape[1]
)

feature_names_match_pipeline = np.array_equal(
    feature_names,
    feature_names_from_pipeline
)

feature_names_unique = (
    pd.Index(feature_names).is_unique
)

preprocessor_fitted = hasattr(
    preprocessor,
    "transformers_"
)

preprocessor_input_columns_valid = (
    hasattr(preprocessor, "feature_names_in_")
    and np.array_equal(
        np.asarray(preprocessor.feature_names_in_, dtype=str),
        np.asarray(X_train_prepared.columns, dtype=str)
    )
)


# ============================================================
# STEP 11: CHECK PROCESSED MATRIX QUALITY
# ============================================================

def validate_processed_matrix(matrix):
    """
    Return missing and infinite value counts.

    Supports dense NumPy arrays and SciPy sparse matrices.
    """

    if sp.issparse(matrix):
        values = matrix.data
    else:
        values = np.asarray(matrix).ravel()

    if not np.issubdtype(values.dtype, np.number):
        raise TypeError(
            "Processed matrix contains non-numeric values."
        )

    missing_count = int(np.isnan(values).sum())
    infinite_count = int(np.isinf(values).sum())

    return missing_count, infinite_count


train_missing, train_infinite = validate_processed_matrix(
    X_train_processed
)

valid_missing, valid_infinite = validate_processed_matrix(
    X_valid_processed
)

test_missing, test_infinite = validate_processed_matrix(
    X_test_processed
)


no_missing_values = (
    train_missing == 0
    and valid_missing == 0
    and test_missing == 0
)

no_infinite_values = (
    train_infinite == 0
    and valid_infinite == 0
    and test_infinite == 0
)


# ============================================================
# STEP 12: BUILD FINAL VALIDATION CHECKLIST
# ============================================================

validation_results = {

    "Section 12 completed":
        bool(section_12_successful),

    "Training row count valid":
        bool(train_rows_valid),

    "Validation row count valid":
        bool(valid_rows_valid),

    "Test row count valid":
        bool(test_rows_valid),

    "Raw feature columns aligned":
        bool(raw_feature_alignment),

    "Prepared feature columns aligned":
        bool(prepared_feature_alignment),

    "Prepared indexes preserved":
        bool(prepared_index_alignment),

    "Training and validation targets aligned":
        bool(target_index_alignment),

    "Test identifiers aligned":
        bool(test_id_index_alignment),

    "No target or identifier leakage":
        bool(no_target_or_identifier_leakage),

    "Categorical features valid":
        bool(categorical_features_valid),

    "Numerical features valid":
        bool(numerical_features_valid),

    "Feature groups do not overlap":
        bool(feature_groups_do_not_overlap),

    "Feature groups cover all inputs":
        bool(feature_groups_cover_all_inputs),

    "No constant features remain":
        bool(no_constant_features_remaining),

    "Sales target values valid":
        bool(targets_valid),

    "Test identifiers valid":
        bool(test_ids_valid),

    "All dates valid":
        bool(dates_valid),

    "Validation period contains 42 days":
        bool(validation_period_valid),

    "Chronological order valid":
        bool(chronological_order_valid),

    "Processed dimensions valid":
        bool(processed_feature_dimensions_valid),

    "Feature names count valid":
        bool(feature_names_count_valid),

    "Feature names match fitted pipeline":
        bool(feature_names_match_pipeline),

    "Feature names unique":
        bool(feature_names_unique),

    "Preprocessor fitted":
        bool(preprocessor_fitted),

    "Preprocessor input columns valid":
        bool(preprocessor_input_columns_valid),

    "No missing processed values":
        bool(no_missing_values),

    "No infinite processed values":
        bool(no_infinite_values)

}


# ============================================================
# STEP 13: DISPLAY FINAL VALIDATION SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("SECTION 13: FINAL MODEL-READINESS CHECKLIST")
print("=" * 90)


print("\nProcessed Dataset Shapes:")

print(f"Training:   {X_train_processed.shape}")
print(f"Validation: {X_valid_processed.shape}")
print(f"Test:       {X_test_processed.shape}")


print("\nChronological Date Ranges:")

print(
    f"Training:   {train_dates.min().date()} "
    f"to {train_dates.max().date()}"
)

print(
    f"Validation: {valid_dates.min().date()} "
    f"to {valid_dates.max().date()}"
)

print(
    f"Test:       {test_dates.min().date()} "
    f"to {test_dates.max().date()}"
)


print("\nProcessed Matrix Quality:")

print(
    f"Missing values (train/valid/test): "
    f"{train_missing}/{valid_missing}/{test_missing}"
)

print(
    f"Infinite values (train/valid/test): "
    f"{train_infinite}/{valid_infinite}/{test_infinite}"
)


print("\nValidation Checks:")

for check_name, check_result in validation_results.items():
    print(f"{check_name:<46}: {check_result}")


# ============================================================
# STEP 14: FINAL SECTION 13 STATUS
# ============================================================

datasets_ready_for_modeling = all(
    validation_results.values()
)

section_13_successful = datasets_ready_for_modeling


print("\n" + "=" * 90)

print(
    "DATASETS READY FOR MODELING: "
    f"{datasets_ready_for_modeling}"
)

print("=" * 90)


if not section_13_successful:

    failed_checks = [
        name
        for name, result in validation_results.items()
        if not result
    ]

    logger.error(
        "Section 13 validation failed: %s",
        failed_checks
    )

    raise ValueError(
        "Section 13 validation failed. "
        f"Review: {failed_checks}"
    )


logger.info(
    "Section 13 final model-readiness validation completed successfully."
)

print("\nSECTION 13 COMPLETED SUCCESSFULLY.")

SECTION 13: FINAL MODEL-READINESS VALIDATION


2026-11-06 16:12:36,066 | INFO | Project6_Preprocessing | Section 13 final model-readiness validation completed successfully.



SECTION 13: FINAL MODEL-READINESS CHECKLIST

Processed Dataset Shapes:
Training:   (970379, 1149)
Validation: (46830, 1149)
Test:       (41088, 1149)

Chronological Date Ranges:
Training:   2013-01-01 to 2015-06-19
Validation: 2015-06-20 to 2015-07-31
Test:       2015-08-01 to 2015-09-17

Processed Matrix Quality:
Missing values (train/valid/test): 0/0/0
Infinite values (train/valid/test): 0/0/0

Validation Checks:
Section 12 completed                          : True
Training row count valid                      : True
Validation row count valid                    : True
Test row count valid                          : True
Raw feature columns aligned                   : True
Prepared feature columns aligned              : True
Prepared indexes preserved                    : True
Training and validation targets aligned       : True
Test identifiers aligned                      : True
No target or identifier leakage               : True
Categorical features valid                    : Tr

# SECTION 14: SAVE AND VERIFY MODEL-READY ARTIFACTS

## Purpose

This section saves all processed datasets and preprocessing
artifacts generated in the previous sections.

The saved artifacts will be used in the next stage of model
development and evaluation.

## Artifacts to be Saved

The following nine artifacts will be created:

1. X_train_processed
2. X_valid_processed
3. X_test_processed
4. y_train
5. y_valid
6. test_ids
7. processed_feature_names
8. modeling_metadata
9. preprocessor

## Storage Locations

Processed datasets and metadata:

data/model_ready/

Fitted preprocessing pipeline:

models/

## Verification Strategy

Each saved artifact will be checked for:

- File existence and non-zero file size.
- Correct matrix dimensions.
- Correct saved matrix values.
- Correct target and test identifier values.
- Correct processed feature names.
- Successful metadata reload.
- Successful preprocessing pipeline reload.
- Reproduction of processed output using the reloaded pipeline.

## Important Methodology

The preprocessing pipeline was fitted only on X_train.

The same fitted preprocessor was used to transform validation
and test datasets.

The original test identifiers are preserved without generating
new identifiers.

Existing outdated artifacts will be overwritten by the current
model-ready outputs.

## Expected Outcome

All nine artifacts must be saved and successfully verified.

The final status variable section_14_successful will become
True only after all verification checks pass.

In [382]:
# ============================================================
# SECTION 14: SAVE AND VERIFY MODEL-READY ARTIFACTS
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp

from pathlib import Path


print("=" * 90)
print("SECTION 14: SAVE AND VERIFY MODEL-READY ARTIFACTS")
print("=" * 90)


# ============================================================
# STEP 1: VERIFY SECTION 13 COMPLETION
# ============================================================

if not globals().get("section_13_successful", False):

    raise RuntimeError(
        "Section 13 has not completed successfully. "
        "Run Sections 1-13 in order before Section 14."
    )


# ============================================================
# STEP 2: VALIDATE REQUIRED INPUT OBJECTS
# ============================================================

required_objects = [
    "PROJECT_ROOT",

    "X_train",
    "X_valid",
    "X_test_final",

    "X_train_prepared",
    "X_valid_prepared",
    "X_test_prepared",

    "X_train_processed",
    "X_valid_processed",
    "X_test_processed",

    "y_train",
    "y_valid",
    "test_ids",

    "preprocessor",
    "feature_names",

    "numerical_features",
    "categorical_features",

    "date_reference",
    "test_working",

    "datasets_ready_for_modeling",
    "logger"
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise NameError(
        "Required objects are missing. "
        f"Re-run Sections 1-13: {missing_objects}"
    )


if not datasets_ready_for_modeling:

    raise RuntimeError(
        "Datasets are not ready for modeling. "
        "Review Section 13."
    )


# ============================================================
# STEP 3: VALIDATE PROCESSED MATRIX DIMENSIONS
# ============================================================

train_rows = X_train_processed.shape[0]
valid_rows = X_valid_processed.shape[0]
test_rows = X_test_processed.shape[0]


matrix_dimensions_valid = (

    train_rows == X_train.shape[0] == y_train.shape[0]

    and valid_rows == X_valid.shape[0] == y_valid.shape[0]

    and test_rows == X_test_final.shape[0] == test_ids.shape[0]

    and X_train_processed.shape[1]
        == X_valid_processed.shape[1]
        == X_test_processed.shape[1]

    and X_train_prepared.index.equals(X_train.index)

    and X_valid_prepared.index.equals(X_valid.index)

    and X_test_prepared.index.equals(X_test_final.index)

)


if not matrix_dimensions_valid:

    raise ValueError(
        "Processed matrix dimensions or input row alignment "
        "are inconsistent."
    )


# ============================================================
# STEP 4: VERIFY FITTED PREPROCESSOR
# ============================================================

preprocessor_fitted = hasattr(
    preprocessor,
    "transformers_"
)


if not preprocessor_fitted:

    raise ValueError(
        "The preprocessing pipeline has not been fitted."
    )


pipeline_feature_names = np.asarray(
    preprocessor.get_feature_names_out(),
    dtype=str
)

current_feature_names = np.asarray(
    feature_names,
    dtype=str
)


feature_names_valid = (

    len(current_feature_names) == X_train_processed.shape[1]

    and np.array_equal(
        current_feature_names,
        pipeline_feature_names
    )

    and pd.Index(current_feature_names).is_unique

)


if not feature_names_valid:

    raise ValueError(
        "Processed feature names do not match the fitted preprocessor."
    )


# ============================================================
# STEP 5: DEFINE PROJECT OUTPUT DIRECTORIES
# ============================================================

project_root_path = Path(
    os.fspath(PROJECT_ROOT)
).resolve()


if not project_root_path.is_dir():

    raise FileNotFoundError(
        f"Project root does not exist: {project_root_path}"
    )


model_ready_path = (
    project_root_path
    / "data"
    / "model_ready"
)

models_path = (
    project_root_path
    / "models"
)


# Create directories if they do not exist.

model_ready_path.mkdir(
    parents=True,
    exist_ok=True
)

models_path.mkdir(
    parents=True,
    exist_ok=True
)


print("\n" + "-" * 90)

print("PROJECT DIRECTORIES")

print("-" * 90)

print(f"Project Root: {project_root_path}")

print(f"Model-ready Directory: {model_ready_path}")

print(f"Models Directory: {models_path}")


# ============================================================
# STEP 6: DEFINE SAFE MATRIX SAVING FUNCTION
# ============================================================

def save_processed_matrix(matrix, file_base_path):
    """
    Save a processed matrix.

    Sparse matrix -> compressed .npz
    Dense matrix  -> .npy

    Removes an outdated file with the alternate extension.
    """

    file_base_path = Path(file_base_path)

    if sp.issparse(matrix):

        file_path = file_base_path.with_suffix(".npz")

        alternate_path = file_base_path.with_suffix(".npy")

        if alternate_path.exists():
            alternate_path.unlink()

        sp.save_npz(
            file_path,
            matrix.tocsr(),
            compressed=True
        )

    else:

        file_path = file_base_path.with_suffix(".npy")

        alternate_path = file_base_path.with_suffix(".npz")

        if alternate_path.exists():
            alternate_path.unlink()

        np.save(
            file_path,
            np.asarray(matrix),
            allow_pickle=False
        )

    if not file_path.is_file() or file_path.stat().st_size == 0:

        raise IOError(
            f"Matrix could not be saved correctly: {file_path}"
        )

    return file_path


# ============================================================
# STEP 7: SAVE THREE PROCESSED FEATURE MATRICES
# ============================================================

print("\n" + "=" * 90)

print("SAVING PROCESSED FEATURE MATRICES")

print("=" * 90)


train_matrix_path = save_processed_matrix(
    X_train_processed,
    model_ready_path / "X_train_processed"
)


valid_matrix_path = save_processed_matrix(
    X_valid_processed,
    model_ready_path / "X_valid_processed"
)


test_matrix_path = save_processed_matrix(
    X_test_processed,
    model_ready_path / "X_test_processed"
)


print(f"\nTraining Matrix Saved: {train_matrix_path}")

print(f"Validation Matrix Saved: {valid_matrix_path}")

print(f"Test Matrix Saved: {test_matrix_path}")


# ============================================================
# STEP 8: SAVE TARGET VARIABLES AND ORIGINAL TEST IDS
# ============================================================

# Preserve the original row order.
# No new test identifiers are generated.

y_train_path = (
    model_ready_path / "y_train.csv"
)

y_valid_path = (
    model_ready_path / "y_valid.csv"
)

test_ids_path = (
    model_ready_path / "test_ids.csv"
)


pd.DataFrame({

    "Sales": y_train.to_numpy(dtype="float64")

}).to_csv(
    y_train_path,
    index=False,
    float_format="%.17g"
)


pd.DataFrame({

    "Sales": y_valid.to_numpy(dtype="float64")

}).to_csv(
    y_valid_path,
    index=False,
    float_format="%.17g"
)


pd.DataFrame({

    "Id": test_ids.to_numpy()

}).to_csv(
    test_ids_path,
    index=False
)


print("\n" + "-" * 90)

print("TARGET AND TEST IDENTIFIER FILES SAVED")

print("-" * 90)

print(f"Training Target: {y_train_path}")

print(f"Validation Target: {y_valid_path}")

print(f"Original Test IDs: {test_ids_path}")


# ============================================================
# STEP 9: SAVE PROCESSED FEATURE NAMES
# ============================================================

feature_names_path = (
    model_ready_path / "processed_feature_names.csv"
)


feature_names_df = pd.DataFrame({

    "Feature_Name": current_feature_names

})


feature_names_df.to_csv(
    feature_names_path,
    index=False
)


print("\nProcessed Feature Names Saved:")

print(feature_names_path)

print(
    f"Total Processed Feature Names: "
    f"{len(current_feature_names):,}"
)


# ============================================================
# STEP 10: PREPARE TRAINING AND VALIDATION DATE METADATA
# ============================================================

train_dates_metadata = pd.to_datetime(
    date_reference.loc[X_train.index],
    errors="coerce"
).dt.normalize()


valid_dates_metadata = pd.to_datetime(
    date_reference.loc[X_valid.index],
    errors="coerce"
).dt.normalize()


test_dates_metadata = pd.to_datetime(
    test_working.loc[X_test_final.index, "Date"],
    errors="coerce"
).dt.normalize()


if (

    train_dates_metadata.isna().any()

    or valid_dates_metadata.isna().any()

    or test_dates_metadata.isna().any()

):

    raise ValueError(
        "Invalid dates found while creating modeling metadata."
    )


# ============================================================
# STEP 11: CREATE MODELING METADATA
# ============================================================

metadata = {

    "project":
        "Sales Forecasting Across Multiple Retail Stores",

    "notebook":
        "03_feature_engineering_preprocessing",

    "target_variable":
        "Sales",

    "training_rows":
        int(X_train_processed.shape[0]),

    "validation_rows":
        int(X_valid_processed.shape[0]),

    "test_rows":
        int(X_test_processed.shape[0]),

    "raw_feature_count":
        int(X_train.shape[1]),

    "processed_feature_count":
        int(X_train_processed.shape[1]),

    "numerical_feature_count":
        int(len(numerical_features)),

    "categorical_feature_count":
        int(len(categorical_features)),

    "numerical_features":
        [str(name) for name in numerical_features],

    "categorical_features":
        [str(name) for name in categorical_features],

    "processed_feature_names":
        [str(name) for name in current_feature_names],

    "preprocessing_strategy":
        "ColumnTransformer",

    "numerical_imputation":
        "Median",

    "numerical_scaling":
        "StandardScaler",

    "categorical_imputation":
        "Most Frequent",

    "categorical_encoding":
        "OneHotEncoder",

    "unknown_category_handling":
        "ignore",

    "preprocessor_fitted_on":
        "X_train only",

    "validation_strategy":
        "Chronological 42-day holdout",

    "training_date_start":
        str(train_dates_metadata.min().date()),

    "training_date_end":
        str(train_dates_metadata.max().date()),

    "validation_date_start":
        str(valid_dates_metadata.min().date()),

    "validation_date_end":
        str(valid_dates_metadata.max().date()),

    "test_date_start":
        str(test_dates_metadata.min().date()),

    "test_date_end":
        str(test_dates_metadata.max().date())

}


# ============================================================
# STEP 12: SAVE MODELING METADATA
# ============================================================

metadata_path = (
    model_ready_path / "modeling_metadata.json"
)


with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# STEP 13: SAVE THE FITTED PREPROCESSOR
# ============================================================

preprocessor_path = (
    models_path / "preprocessor.joblib"
)


joblib.dump(
    preprocessor,
    preprocessor_path
)


print("\n" + "-" * 90)

print("PREPROCESSOR AND METADATA SAVED")

print("-" * 90)

print(f"Preprocessor: {preprocessor_path}")

print(f"Metadata: {metadata_path}")


# ============================================================
# STEP 14: VERIFY EXISTENCE OF ALL NINE ARTIFACTS
# ============================================================

expected_artifacts = {

    "X_train_processed":
        train_matrix_path,

    "X_valid_processed":
        valid_matrix_path,

    "X_test_processed":
        test_matrix_path,

    "y_train":
        y_train_path,

    "y_valid":
        y_valid_path,

    "test_ids":
        test_ids_path,

    "processed_feature_names":
        feature_names_path,

    "modeling_metadata":
        metadata_path,

    "preprocessor":
        preprocessor_path

}


artifact_status = {}


print("\n" + "=" * 90)

print("ARTIFACT EXISTENCE VERIFICATION")

print("=" * 90)


for artifact_name, artifact_path in expected_artifacts.items():

    artifact_path = Path(artifact_path)

    exists = artifact_path.is_file()

    non_empty = (
        exists
        and artifact_path.stat().st_size > 0
    )

    artifact_status[artifact_name] = non_empty

    print(
        f"{artifact_name:<30}: "
        f"{non_empty}"
    )


all_artifacts_exist = (
    len(expected_artifacts) == 9
    and all(artifact_status.values())
)


if not all_artifacts_exist:

    raise FileNotFoundError(
        "One or more of the nine expected artifacts "
        "are missing or empty."
    )


# ============================================================
# STEP 15: DEFINE MATRIX RELOAD FUNCTION
# ============================================================

def load_processed_matrix(file_path):
    """
    Load a saved dense or sparse matrix.
    """

    file_path = Path(file_path)

    suffix = file_path.suffix.lower()

    if suffix == ".npz":

        return sp.load_npz(file_path)

    if suffix == ".npy":

        return np.load(
            file_path,
            allow_pickle=False
        )

    raise ValueError(
        f"Unsupported matrix file format: {file_path}"
    )


# ============================================================
# STEP 16: DEFINE MATRIX VALUE COMPARISON FUNCTION
# ============================================================

def matrices_match(saved_matrix, current_matrix):
    """
    Compare dense, sparse or mixed matrices.

    Checks dimensions and matrix values.
    """

    if saved_matrix.shape != current_matrix.shape:
        return False

    if (
        sp.issparse(saved_matrix)
        or sp.issparse(current_matrix)
    ):

        saved_sparse = sp.csr_matrix(saved_matrix)

        current_sparse = sp.csr_matrix(current_matrix)

        difference = saved_sparse - current_sparse

        difference.eliminate_zeros()

        return difference.nnz == 0

    return np.array_equal(
        np.asarray(saved_matrix),
        np.asarray(current_matrix)
    )


# ============================================================
# STEP 17: RELOAD AND VERIFY THREE PROCESSED MATRICES
# ============================================================

X_train_check = load_processed_matrix(
    train_matrix_path
)

X_valid_check = load_processed_matrix(
    valid_matrix_path
)

X_test_check = load_processed_matrix(
    test_matrix_path
)


train_matrix_match = matrices_match(
    X_train_check,
    X_train_processed
)

valid_matrix_match = matrices_match(
    X_valid_check,
    X_valid_processed
)

test_matrix_match = matrices_match(
    X_test_check,
    X_test_processed
)


# ============================================================
# STEP 18: RELOAD AND VERIFY TARGET VARIABLES
# ============================================================

y_train_check = pd.read_csv(
    y_train_path
)

y_valid_check = pd.read_csv(
    y_valid_path
)


train_target_match = np.allclose(
    y_train_check["Sales"].to_numpy(dtype="float64"),
    y_train.to_numpy(dtype="float64"),
    rtol=0,
    atol=1e-12
)


valid_target_match = np.allclose(
    y_valid_check["Sales"].to_numpy(dtype="float64"),
    y_valid.to_numpy(dtype="float64"),
    rtol=0,
    atol=1e-12
)


# ============================================================
# STEP 19: RELOAD AND VERIFY ORIGINAL TEST IDENTIFIERS
# ============================================================

test_ids_check = pd.read_csv(
    test_ids_path,
    dtype={"Id": "string"}
)


current_test_ids = (
    test_ids
    .reset_index(drop=True)
    .astype("string")
)


saved_test_ids = (
    test_ids_check["Id"]
    .astype("string")
)


test_ids_match = saved_test_ids.equals(
    current_test_ids
)


# ============================================================
# STEP 20: RELOAD AND VERIFY FEATURE NAMES
# ============================================================

feature_names_check = pd.read_csv(
    feature_names_path,
    dtype={"Feature_Name": "string"}
)


saved_feature_names = (
    feature_names_check["Feature_Name"]
    .astype(str)
    .to_numpy()
)


feature_names_match = np.array_equal(
    saved_feature_names,
    current_feature_names
)


# ============================================================
# STEP 21: RELOAD AND VERIFY MODELING METADATA
# ============================================================

with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as file:

    metadata_check = json.load(file)


metadata_match = (
    metadata_check == metadata
)


# ============================================================
# STEP 22: RELOAD AND VERIFY PREPROCESSOR
# ============================================================

preprocessor_check = joblib.load(
    preprocessor_path
)


preprocessor_loaded = hasattr(
    preprocessor_check,
    "transformers_"
)


reloaded_feature_names = np.asarray(
    preprocessor_check.get_feature_names_out(),
    dtype=str
)


preprocessor_feature_names_match = np.array_equal(
    reloaded_feature_names,
    current_feature_names
)


# IMPORTANT:
# Section 12 fitted the preprocessor on X_train_prepared.
# Therefore, verification must use X_train_prepared,
# not the original X_train dataframe.

sample_size = min(
    100,
    X_train_prepared.shape[0]
)


reloaded_sample = preprocessor_check.transform(
    X_train_prepared.iloc[:sample_size]
)


saved_sample = X_train_processed[:sample_size]


preprocessor_output_match = matrices_match(
    reloaded_sample,
    saved_sample
)


# ============================================================
# STEP 23: BUILD FINAL VERIFICATION CHECKLIST
# ============================================================

verification_results = {

    "All nine artifacts exist":
        bool(all_artifacts_exist),

    "Training matrix values match":
        bool(train_matrix_match),

    "Validation matrix values match":
        bool(valid_matrix_match),

    "Test matrix values match":
        bool(test_matrix_match),

    "Training target values match":
        bool(train_target_match),

    "Validation target values match":
        bool(valid_target_match),

    "Original test IDs match":
        bool(test_ids_match),

    "Processed feature names match":
        bool(feature_names_match),

    "Modeling metadata matches":
        bool(metadata_match),

    "Preprocessor reloaded":
        bool(preprocessor_loaded),

    "Preprocessor feature names match":
        bool(preprocessor_feature_names_match),

    "Preprocessor output reproduced":
        bool(preprocessor_output_match)

}


print("\n" + "=" * 90)

print("FINAL ARTIFACT VERIFICATION CHECKLIST")

print("=" * 90)


for check_name, check_result in verification_results.items():

    print(
        f"{check_name:<42}: {check_result}"
    )


# ============================================================
# STEP 24: FINAL SECTION 14 STATUS
# ============================================================

all_artifacts_verified = (
    all(verification_results.values())
)

section_14_successful = all_artifacts_verified


print("\n" + "=" * 90)

print(
    f"TOTAL EXPECTED ARTIFACTS: "
    f"{len(expected_artifacts)}"
)

print(
    f"SUCCESSFULLY SAVED: "
    f"{sum(artifact_status.values())}"
)

print(
    f"ALL ARTIFACTS VERIFIED: "
    f"{all_artifacts_verified}"
)

print("=" * 90)


if not section_14_successful:

    failed_checks = [
        name
        for name, result in verification_results.items()
        if not result
    ]

    logger.error(
        "Section 14 artifact verification failed: %s",
        failed_checks
    )

    raise ValueError(
        "Section 14 verification failed. "
        f"Review: {failed_checks}"
    )


logger.info(
    "Section 14 completed successfully. "
    "All nine model-ready artifacts were saved and verified."
)


print("\nSECTION 14 COMPLETED SUCCESSFULLY.")

print(
    "All model-ready datasets, targets, test IDs, "
    "feature names, metadata and preprocessor "
    "have been saved and verified."
)

print("=" * 90)

SECTION 14: SAVE AND VERIFY MODEL-READY ARTIFACTS

------------------------------------------------------------------------------------------
PROJECT DIRECTORIES
------------------------------------------------------------------------------------------
Project Root: D:\Project_6_Rossmann_Forecasting
Model-ready Directory: D:\Project_6_Rossmann_Forecasting\data\model_ready
Models Directory: D:\Project_6_Rossmann_Forecasting\models

SAVING PROCESSED FEATURE MATRICES

Training Matrix Saved: D:\Project_6_Rossmann_Forecasting\data\model_ready\X_train_processed.npz
Validation Matrix Saved: D:\Project_6_Rossmann_Forecasting\data\model_ready\X_valid_processed.npz
Test Matrix Saved: D:\Project_6_Rossmann_Forecasting\data\model_ready\X_test_processed.npz

------------------------------------------------------------------------------------------
TARGET AND TEST IDENTIFIER FILES SAVED
------------------------------------------------------------------------------------------
Training Target: D:\Pro

2026-11-06 16:12:40,854 | INFO | Project6_Preprocessing | Section 14 completed successfully. All nine model-ready artifacts were saved and verified.



FINAL ARTIFACT VERIFICATION CHECKLIST
All nine artifacts exist                  : True
Training matrix values match              : True
Validation matrix values match            : True
Test matrix values match                  : True
Training target values match              : True
Validation target values match            : True
Original test IDs match                   : True
Processed feature names match             : True
Modeling metadata matches                 : True
Preprocessor reloaded                     : True
Preprocessor feature names match          : True
Preprocessor output reproduced            : True

TOTAL EXPECTED ARTIFACTS: 9
SUCCESSFULLY SAVED: 9
ALL ARTIFACTS VERIFIED: True

SECTION 14 COMPLETED SUCCESSFULLY.
All model-ready datasets, targets, test IDs, feature names, metadata and preprocessor have been saved and verified.


# SECTION 15: FINAL NOTEBOOK COMPLETION SUMMARY

## Purpose

This section performs the final completion audit of Notebook 03.

It verifies the successful execution of all fourteen previous
sections and confirms that the model-ready datasets and
preprocessing artifacts are available and verified.

## Final Workflow Review

The notebook includes:

- Data loading and initial validation.
- Initial data quality checks.
- Calendar and holiday feature engineering.
- Competition and promotion feature engineering.
- Store-level and categorical feature validation.
- Missingness and structural validation.
- Final feature selection.
- Chronological 42-day train-validation split.
- Leakage-aware numerical and categorical preprocessing.
- Final model-readiness validation.
- Saving and verification of model-ready artifacts.

## Final Output

The final summary reports:

- Section-wise completion status.
- Model-ready dataset dimensions.
- Numerical and categorical feature counts.
- Processed feature count.
- Saved artifact count.
- Final Notebook 03 completion status.

## Important Note

Successful completion of Notebook 03 means that the
feature engineering and preprocessing stage is complete.

The next stage is model development and evaluation.
It does not mean that the complete Sales Forecasting project
or model deployment has been completed.

In [383]:

# ============================================================
# SECTION 15: FINAL NOTEBOOK COMPLETION SUMMARY
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import logging


print("=" * 95)
print("NOTEBOOK 03: FEATURE ENGINEERING & PREPROCESSING")
print("FINAL COMPLETION SUMMARY")
print("=" * 95)


# ============================================================
# STEP 1: VERIFY REQUIRED OBJECTS FROM SECTIONS 1-14
# ============================================================

required_objects = [

    "RANDOM_STATE",
    "logger",

    "TRAIN_PATH",
    "TEST_PATH",

    "train_df",
    "test_df",

    "train_working",
    "test_working",

    "store_date_unique",
    "sales_target_valid",

    "section4_successful",
    "competition_feature_validation",
    "promotion_feature_successful",

    "store_features",
    "section_8_successful",

    "section9_successful",
    "section_10_successful",
    "temporal_split_successful",

    "section_12_successful",
    "section_13_successful",
    "datasets_ready_for_modeling",

    "section_14_successful",
    "all_artifacts_verified",

    "X_candidate",
    "X_test_candidate",

    "X_train",
    "X_valid",
    "X_test_final",

    "X_train_prepared",
    "X_valid_prepared",
    "X_test_prepared",

    "X_train_processed",
    "X_valid_processed",
    "X_test_processed",

    "y",
    "y_train",
    "y_valid",
    "test_ids",

    "preprocessor",
    "feature_names",

    "numerical_features",
    "categorical_features",

    "expected_artifacts",
    "verification_results",
    "validation_results",
    "metadata",

    "date_reference"

]


missing_objects = [

    name

    for name in required_objects

    if name not in globals()

]


if missing_objects:

    raise NameError(
        "Section 15 cannot run because required outputs "
        "from previous sections are missing: "
        f"{missing_objects}. "
        "Run Sections 1-14 in order."
    )


# ============================================================
# STEP 2: SECTION 1 - IMPORTS AND CONFIGURATION
# ============================================================

section_1_valid = (

    RANDOM_STATE == 42

    and isinstance(logger, logging.Logger)

    and isinstance(np, type(np))

    and isinstance(pd, type(pd))

)


# ============================================================
# STEP 3: SECTION 2 - DATA LOADING
# ============================================================

train_file_exists = Path(
    TRAIN_PATH
).is_file()


test_file_exists = Path(
    TEST_PATH
).is_file()


section_2_valid = (

    train_file_exists

    and test_file_exists

    and isinstance(train_df, pd.DataFrame)

    and isinstance(test_df, pd.DataFrame)

    and isinstance(train_working, pd.DataFrame)

    and isinstance(test_working, pd.DataFrame)

    and not train_df.empty

    and not test_df.empty

    and len(train_df) == len(train_working)

    and len(test_df) == len(test_working)

    and "Sales" in train_df.columns

    and "Sales" not in test_df.columns

    and "Id" in test_df.columns

)


# ============================================================
# STEP 4: SECTION 3 - INITIAL DATA QUALITY
# ============================================================

section_3_valid = (

    bool(store_date_unique)

    and bool(sales_target_valid)

    and train_df["Date"].notna().all()

    and test_df["Date"].notna().all()

)


# ============================================================
# STEP 5: SECTION 4 - CALENDAR AND HOLIDAY FEATURES
# ============================================================

section_4_valid = bool(
    section4_successful
)


# ============================================================
# NOTEBOOK 03 - SECTION 14
# REPAIR AND VERIFY MODELING METADATA
# ============================================================

import json
from pathlib import Path
from sklearn.utils.validation import check_is_fitted


# STEP 1: VERIFY REQUIRED NOTEBOOK 03 OBJECTS

required_objects = [
    "metadata",
    "model_ready_path",
    "X_train_processed",
    "X_valid_processed",
    "X_test_processed",
    "preprocessor"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the relevant Notebook 03 preprocessing and metadata cells first. "
        f"Missing objects: {missing_objects}"
    )


# STEP 2: CONFIRM THE PREPROCESSOR IS FITTED

try:
    check_is_fitted(preprocessor)
except Exception as exc:
    raise ValueError(
        "The Notebook 03 preprocessor is not fitted. "
        "Do not write metadata until preprocessing has been completed."
    ) from exc


# STEP 3: UPDATE ONLY VERIFIED METADATA FIELDS

metadata = dict(metadata)

metadata.update({
    "train_rows": int(X_train_processed.shape[0]),
    "validation_rows": int(X_valid_processed.shape[0]),
    "test_rows": int(X_test_processed.shape[0]),
    "processed_feature_count": int(X_train_processed.shape[1]),
    "preprocessing_fitted_on": "X_train only"
})


# STEP 4: SAVE THE CORRECTED METADATA JSON

metadata_path = Path(model_ready_path) / "modeling_metadata.json"
metadata_path.parent.mkdir(parents=True, exist_ok=True)

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# STEP 5: RELOAD AND VERIFY THE SAVED JSON

with open(metadata_path, "r", encoding="utf-8") as file:
    metadata_reloaded = json.load(file)

required_metadata_fields = {
    "train_rows": int(X_train_processed.shape[0]),
    "validation_rows": int(X_valid_processed.shape[0]),
    "test_rows": int(X_test_processed.shape[0]),
    "processed_feature_count": int(X_train_processed.shape[1]),
    "preprocessing_fitted_on": "X_train only"
}

metadata_field_checks = {
    key: metadata_reloaded.get(key) == expected_value
    for key, expected_value in required_metadata_fields.items()
}

metadata_saved = metadata_path.is_file() and metadata_path.stat().st_size > 0

metadata_save_verified = (
    metadata_reloaded == metadata
    and all(metadata_field_checks.values())
)

metadata_repair_successful = (
    metadata_saved
    and metadata_save_verified
)

print("=" * 90)
print("NOTEBOOK 03 - MODELING METADATA REPAIR")
print("=" * 90)
print("Metadata file:", metadata_path)
print("Saved:", metadata_saved)
print("Reload verification:", metadata_save_verified)

print("Field checks:")
for field_name, field_status in metadata_field_checks.items():
    print(f"{field_name:<32}: {field_status}")

print("METADATA REPAIR SUCCESSFUL:", metadata_repair_successful)
print("=" * 90)

if not metadata_repair_successful:
    raise RuntimeError(
        "Modeling metadata could not be saved and verified. "
        "Review the failed metadata field checks."
    )


# ============================================================
# STEP 6: SECTION 5 - COMPETITION FEATURES
# ============================================================

competition_columns = [

    "CompetitionDistance",

    "CompetitionDistanceMissing",

    "CompetitionOpenInfoMissing",

    "CompetitionOpenDate",

    "CompetitionOpenDurationMonths"

]


competition_columns_valid = all(

    column in train_working.columns

    and column in test_working.columns

    for column in competition_columns

)


section_5_valid = (

    bool(competition_feature_validation)

    and competition_columns_valid

)


# ============================================================
# STEP 7: SECTION 6 - PROMOTION FEATURES
# ============================================================

promotion_columns = [

    "Promo2StartDate",

    "Promo2InfoMissing",

    "PromoIntervalMissing",

    "Promo2DurationMonths",

    "Promo2NotStarted",

    "Promo2IntervalMonth"

]


promotion_columns_valid = all(

    column in train_working.columns

    and column in test_working.columns

    for column in promotion_columns

)


section_6_valid = (

    bool(promotion_feature_successful)

    and promotion_columns_valid

)


# ============================================================
# STEP 8: SECTION 7 - STORE-LEVEL FEATURES
# ============================================================

store_columns_valid = all(

    column in train_working.columns

    and column in test_working.columns

    for column in store_features

)


if store_columns_valid:

    store_values_complete = (

        train_working[store_features]
        .notna()
        .all()
        .all()

        and

        test_working[store_features]
        .notna()
        .all()
        .all()

    )


    train_store_ids = set(
        train_working["Store"].unique()
    )


    test_store_ids = set(
        test_working["Store"].unique()
    )


    store_coverage_valid = test_store_ids.issubset(
        train_store_ids
    )


else:

    store_values_complete = False

    store_coverage_valid = False


section_7_valid = (

    store_columns_valid

    and store_values_complete

    and store_coverage_valid

)


# ============================================================
# STEP 9: SECTION 8 - CATEGORICAL FEATURE VALIDATION
# ============================================================

section_8_valid = bool(
    section_8_successful
)


# ============================================================
# STEP 10: SECTION 9 - STRUCTURAL VALIDATION
# ============================================================

section_9_valid = (

    bool(section9_successful)

    and bool(
        globals().get("section_9_successful", False)
    )

)


# ============================================================
# STEP 11: SECTION 10 - FINAL FEATURE SELECTION
# ============================================================

section_10_valid = (

    bool(section_10_successful)

    and isinstance(X_candidate, pd.DataFrame)

    and isinstance(X_test_candidate, pd.DataFrame)

    and X_candidate.columns.equals(
        X_test_candidate.columns
    )

    and y.index.equals(X_candidate.index)

    and test_ids.index.equals(X_test_candidate.index)

)


# ============================================================
# STEP 12: SECTION 11 - CHRONOLOGICAL TRAIN-VALIDATION SPLIT
# ============================================================

section_11_valid = (

    bool(temporal_split_successful)

    and X_train.shape == (970379, 25)

    and X_valid.shape == (46830, 25)

    and X_test_final.shape == (41088, 25)

    and y_train.shape[0] == X_train.shape[0]

    and y_valid.shape[0] == X_valid.shape[0]

    and test_ids.shape[0] == X_test_final.shape[0]

    and X_train.columns.equals(X_valid.columns)

    and X_train.columns.equals(X_test_final.columns)

)


# ============================================================
# STEP 13: SECTION 12 - PREPROCESSING QUALITY
# ============================================================

def section15_matrix_quality(matrix):
    """
    Count missing and infinite values.

    Supports dense NumPy and SciPy sparse matrices.
    """

    if sp.issparse(matrix):

        values = matrix.data

    else:

        values = np.asarray(matrix).ravel()


    missing_count = int(
        np.isnan(values).sum()
    )

    infinite_count = int(
        np.isinf(values).sum()
    )


    return missing_count, infinite_count


train_missing_count, train_infinite_count = (
    section15_matrix_quality(X_train_processed)
)


valid_missing_count, valid_infinite_count = (
    section15_matrix_quality(X_valid_processed)
)


test_missing_count, test_infinite_count = (
    section15_matrix_quality(X_test_processed)
)


processed_shapes_valid = (

    X_train_processed.shape == (970379, 1149)

    and X_valid_processed.shape == (46830, 1149)

    and X_test_processed.shape == (41088, 1149)

    and len(feature_names) == X_train_processed.shape[1]

)


processed_values_valid = (

    train_missing_count == 0

    and valid_missing_count == 0

    and test_missing_count == 0

    and train_infinite_count == 0

    and valid_infinite_count == 0

    and test_infinite_count == 0

)


section_12_valid = (

    bool(section_12_successful)

    and processed_shapes_valid

    and processed_values_valid

    and hasattr(preprocessor, "transformers_")

    and len(numerical_features) == 20

    and len(categorical_features) == 5

)


# ============================================================
# STEP 14: SECTION 13 - FINAL MODEL READINESS
# ============================================================

section_13_valid = (

    bool(section_13_successful)

    and bool(datasets_ready_for_modeling)

    and isinstance(validation_results, dict)

    and all(validation_results.values())

)


# ============================================================
# STEP 15: SECTION 14 - ARTIFACT VERIFICATION
# ============================================================

expected_artifact_names = {

    "X_train_processed",

    "X_valid_processed",

    "X_test_processed",

    "y_train",

    "y_valid",

    "test_ids",

    "processed_feature_names",

    "modeling_metadata",

    "preprocessor"

}


artifact_names_valid = (

    set(expected_artifacts.keys())
    == expected_artifact_names

)


artifact_files_valid = all(

    Path(file_path).is_file()

    and Path(file_path).stat().st_size > 0

    for file_path in expected_artifacts.values()

)


artifact_checks_valid = (

    isinstance(verification_results, dict)

    and len(verification_results) == 12

    and all(verification_results.values())

)


metadata_valid = (

    isinstance(metadata, dict)

    and metadata.get("training_rows")
        == X_train_processed.shape[0]

    and metadata.get("validation_rows")
        == X_valid_processed.shape[0]

    and metadata.get("test_rows")
        == X_test_processed.shape[0]

    and metadata.get("processed_feature_count")
        == X_train_processed.shape[1]

)


section_14_valid = (

    bool(section_14_successful)

    and bool(all_artifacts_verified)

    and artifact_names_valid

    and artifact_files_valid

    and artifact_checks_valid

    and metadata_valid

)


# ============================================================
# STEP 16: BUILD SECTION-WISE COMPLETION STATUS
# ============================================================

section_status = {

    "Section 1 - Imports and configuration":
        bool(section_1_valid),

    "Section 2 - Data loading":
        bool(section_2_valid),

    "Section 3 - Initial data quality":
        bool(section_3_valid),

    "Section 4 - Calendar and holiday features":
        bool(section_4_valid),

    "Section 5 - Competition features":
        bool(section_5_valid),

    "Section 6 - Promotion features":
        bool(section_6_valid),

    "Section 7 - Store-level features":
        bool(section_7_valid),

    "Section 8 - Categorical validation":
        bool(section_8_valid),

    "Section 9 - Structural validation":
        bool(section_9_valid),

    "Section 10 - Feature selection":
        bool(section_10_valid),

    "Section 11 - Temporal split":
        bool(section_11_valid),

    "Section 12 - Preprocessing":
        bool(section_12_valid),

    "Section 13 - Model readiness":
        bool(section_13_valid),

    "Section 14 - Artifact verification":
        bool(section_14_valid)

}


print("\n" + "=" * 95)

print("SECTION-WISE COMPLETION STATUS")

print("=" * 95)


for section_name, status in section_status.items():

    print(
        f"{section_name:<55}: {status}"
    )


# ============================================================
# STEP 17: DISPLAY FINAL MODEL-READY DATASET SUMMARY
# ============================================================

print("\n" + "=" * 95)

print("FINAL MODEL-READY DATASET SUMMARY")

print("=" * 95)


print(
    f"Training Rows:          "
    f"{X_train_processed.shape[0]:,}"
)

print(
    f"Validation Rows:        "
    f"{X_valid_processed.shape[0]:,}"
)

print(
    f"Test Rows:              "
    f"{X_test_processed.shape[0]:,}"
)


print(
    f"\nRaw Input Features:     "
    f"{X_train.shape[1]:,}"
)

print(
    f"Numerical Features:     "
    f"{len(numerical_features):,}"
)

print(
    f"Categorical Features:   "
    f"{len(categorical_features):,}"
)

print(
    f"Processed Features:     "
    f"{X_train_processed.shape[1]:,}"
)


print("\nProcessed Dataset Shapes:")

print(
    f"X_train_processed: {X_train_processed.shape}"
)

print(
    f"X_valid_processed: {X_valid_processed.shape}"
)

print(
    f"X_test_processed:  {X_test_processed.shape}"
)


print("\nTarget Shapes:")

print(f"y_train: {y_train.shape}")

print(f"y_valid: {y_valid.shape}")

print(f"Original Test IDs: {test_ids.shape}")


# ============================================================
# STEP 18: DISPLAY SAVED ARTIFACT SUMMARY
# ============================================================

existing_artifact_count = sum(

    Path(file_path).is_file()

    and Path(file_path).stat().st_size > 0

    for file_path in expected_artifacts.values()

)


print("\n" + "=" * 95)

print("SAVED ARTIFACT SUMMARY")

print("=" * 95)


print(
    f"Expected Artifacts: {len(expected_artifacts)}"
)

print(
    f"Existing Non-empty Artifacts: "
    f"{existing_artifact_count}"
)


for artifact_name, artifact_path in expected_artifacts.items():

    print(
        f"{artifact_name:<30}: {artifact_path}"
    )


# ============================================================
# STEP 19: FINAL NOTEBOOK COMPLETION STATUS
# ============================================================

notebook_03_completed = all(
    section_status.values()
)


section_15_successful = notebook_03_completed


print("\n" + "=" * 95)

print("NOTEBOOK 03 FINAL COMPLETION STATUS")

print("=" * 95)


print(
    f"\nAll Sections 1-14 Passed: "
    f"{notebook_03_completed}"
)


print(
    f"All Artifacts Verified: "
    f"{all_artifacts_verified}"
)


print(
    f"NOTEBOOK 03 COMPLETED SUCCESSFULLY: "
    f"{notebook_03_completed}"
)


if notebook_03_completed:

    print("\nNotebook 03 has been completed successfully.")

    print(
        "Feature engineering, temporal splitting, "
        "leakage-aware preprocessing and artifact verification "
        "are complete."
    )

    print(
        "\nThe model-ready datasets are prepared for the next stage: "
        "model development and evaluation."
    )


else:

    failed_sections = [

        name

        for name, status in section_status.items()

        if not status

    ]


    print("\nNotebook 03 requires additional review.")

    print("\nSections requiring review:")


    for section_name in failed_sections:

        print(f"- {section_name}")


print("=" * 95)


# ============================================================
# STEP 20: LOG FINAL COMPLETION
# ============================================================

if notebook_03_completed:

    logger.info(
        "Notebook 03 completed successfully. "
        "All Sections 1-14 passed and all nine artifacts "
        "were verified."
    )


else:

    logger.warning(
        "Notebook 03 completion validation failed. "
        "Review the section-wise status above."
    )

    raise ValueError(
        "Notebook 03 final completion check failed."
    )


print("\nSECTION 15 COMPLETED SUCCESSFULLY.")


NOTEBOOK 03: FEATURE ENGINEERING & PREPROCESSING
FINAL COMPLETION SUMMARY
NOTEBOOK 03 - MODELING METADATA REPAIR
Metadata file: D:\Project_6_Rossmann_Forecasting\data\model_ready\modeling_metadata.json
Saved: True
Reload verification: True
Field checks:
train_rows                      : True
validation_rows                 : True
test_rows                       : True
processed_feature_count         : True
preprocessing_fitted_on         : True
METADATA REPAIR SUCCESSFUL: True

SECTION-WISE COMPLETION STATUS
Section 1 - Imports and configuration                  : True
Section 2 - Data loading                               : True
Section 3 - Initial data quality                       : True
Section 4 - Calendar and holiday features              : True
Section 5 - Competition features                       : True
Section 6 - Promotion features                         : True
Section 7 - Store-level features                       : True
Section 8 - Categorical validation                  

2026-11-06 16:12:41,056 | INFO | Project6_Preprocessing | Notebook 03 completed successfully. All Sections 1-14 passed and all nine artifacts were verified.



SECTION 15 COMPLETED SUCCESSFULLY.
